# Mini ETL Framework

A reproducible batch ETL framework. It reads CSV files, validates every row against rules kept in YAML config, quarantines invalid rows with a reason instead of dropping them silently, keeps the full history of every record, loads results into SQLite, and records each run with counts, timings and failure reasons. The same core code runs both datasets: `customers` (parent) and `orders` (child, referencing customers). A new dataset is added with a new config file, not new code. A reconciliation report proves the loaded data agrees with the source files, the run log and a separately stored answer key.

Stack: Python, pandas, SQLite (built into Python), PyYAML, pytest. Runs in Google Colab or locally.

Status: the framework, both datasets, the seeded data generator, 188 automated tests, the reconciliation report with sample queries, and the performance measurement are built. Still to come: a separate design and trade-off note (see "Not built yet").

## Quick start

**Google Colab (documented cloud environment)**

1. Open colab.research.google.com, then File > Upload notebook, and pick `etl_framework_submission.ipynb`.
2. Runtime > Run all. Section 1 installs the libraries, section 2 writes every project file, section 3 runs the tests, section 4 loads the sample data and prints the reconciliation report, section 5 reruns the same input, section 6 runs the SQL files, section 7 loads a 100,000-order dataset, section 8 runs the benchmark (a few minutes), and section 9 packages everything into a zip you can download.

**Locally** (Python 3.10 or newer; developed on 3.14)

```bash
python -m venv .venv
# Windows PowerShell: .venv\Scripts\Activate.ps1
# macOS/Linux:       source .venv/bin/activate
python -m pip install -r requirements.txt
python -m pytest -q
python run_pipeline.py
python run_report.py --input-dir data/sample --expected-dir data/expected
```

Expected results on the included sample (columns of the printed summary):

| dataset | rows_in | rows_inserted | rows_rejected | duplicates_dropped | late_records |
|---|---|---|---|---|---|
| customers | 41 | 27 | 12 | 2 | 0 |
| orders | 114 | 95 | 17 | 2 | 11 |

`python -m pytest -q` reports 188 passed. The report command prints `Result: **PASS** (36 of 36 checks passed)`. Running the pipeline a second time inserts nothing: `rows_inserted` is 0, `rows_already_loaded` is 27 (customers) and 95 (orders), and no table changes.

## Architecture

```text
config/pipeline.yaml + config/datasets/*.yaml
                 |
                 v
        CSV file --> extract.py   read text, keep line numbers, set aside malformed rows
                 |
                 v
        validate.py   schema check (missing required column = run fails; extra column = warning)
                 |    row rules: required, type, pattern, allowed values, range, not in future
                 |                         |
          valid rows                 rejected rows
                 |                         |
                 v                         |
        transform.py   cast types, clean values, derived fields, record_hash
                 |                         |
                 v                         |
        validate.py   duplicate keys, then parent reference check
                 |                         |
                 +-------> load.py <-------+
                           one SQLite transaction per dataset
                                  |
             +--------------------+-------------------+
             v                    v                   v
     <name>_history        <name>_current      rejected_records
                                  |
                       metrics.py -> pipeline_runs      logging_setup.py -> JSON log lines
                                  |
                       reconcile.py -> run_report.py -> reports/reconciliation_report.md
```

`pipeline.py` coordinates the steps for each dataset in the order listed in `config/pipeline.yaml`. `config_loader.py` reads and checks the config files. `parsing.py` holds the shared type-parsing helpers. `errors.py` defines the named failure types. `reconcile.py` generates its checks from the same dataset configs. No module contains customer or order logic: every rule, key, table name and derived field comes from the dataset config. `generator/generate_data.py` creates the synthetic input and the answer key; `run_benchmark.py` measures performance.

## Configuration and commands

| Command | Purpose |
|---|---|
| `python run_pipeline.py` | Load every dataset in `config/pipeline.yaml` |
| `python run_report.py` | Reconciliation report for a database (exit 0 = all checks passed) |
| `python -m generator.generate_data` | Generate the small sample and its expected answers |
| `python run_benchmark.py` | Measure performance and the optimization |
| `python -m pytest -q` | Run the automated tests |

`run_pipeline.py` options:

| Option | Default | Purpose |
|---|---|---|
| `--config FILE` | `config/pipeline.yaml` | Pipeline config: database, input folder, log file, dataset order |
| `--input-dir DIR` | `data/sample` | Folder with the source CSV files |
| `--db FILE` | `output/warehouse.db` | SQLite target database |
| `--log-file FILE` | `logs/pipeline.log` | JSON-lines log file (appended) |
| `--dataset NAME` | all | Run only this dataset (repeatable) |

Exit code 0 means every dataset succeeded, 1 means a dataset failed, 2 means a configuration or argument error. Each invocation gets one `run_id` (a UUID) shared by its datasets and stamped on every log line, metrics row, loaded row and reject row.

`run_report.py` takes `--db`, `--input-dir` (re-counts the source files), `--expected-dir` (compares with the answer key) and `--report-file`; it exits 0 when every check passes, 1 when one fails, 2 when no report can be made. `run_benchmark.py` takes `--customers`, `--orders`, `--seed`, `--repeats`, `--no-scaling`, `--no-profile` and `--report-file`.

**Adding a dataset with config only.** Create `config/datasets/<name>.yaml`, add its path to `datasets` in `config/pipeline.yaml`, and place the CSV in the input folder. A dataset config defines `name`, `source_file`, `natural_key`, `version_column`, the target tables, `columns` (type, required, pattern, allowed_values, min, max, not_in_future, transform), `references` and `derived` fields. Column types are `string`, `integer`, `decimal`, `date`, `timestamp`. Derived operations are `split_part`, `multiply_to_cents` and `date_part`; a new kind of calculation would need a small code addition. Unknown options in a config file fail the run before any data is read, so a typo cannot silently disable a rule. A test adds a third dataset (`products`) this way, with no change to `etl/`.

## Data dictionary

### Input files

| File | Columns | Meaning |
|---|---|---|
| `data/sample/customers.csv` | `customer_id`, `full_name`, `email`, `country_code`, `signup_date`, `updated_at` | One row per customer version. `customer_id` is `C` plus 6 digits; `country_code` is one of IN, US, GB, AE, SG; `email` needs one `@` and a dot after it and is stored in lowercase; `signup_date` is `YYYY-MM-DD`; `signup_date` and `updated_at` may not be in the future. |
| `data/sample/orders.csv` | `order_id`, `customer_id`, `order_ts`, `status`, `quantity`, `unit_price`, `currency`, `updated_at` | One row per order version. `order_id` is `O` plus 8 digits; `customer_id` must exist in `customers_current`; `status` is placed, shipped, delivered or cancelled; `quantity` is 1 to 1000; `unit_price` is 0 to 100000 (0 is allowed); `currency` is INR or USD. |

Extraction reads every value as text, trims whitespace, and treats empty text as missing. Rows with the wrong number of fields are rejected as `MALFORMED_ROW`. Fully blank lines are ignored but counted in the log. A file with only a header is a valid empty batch; a completely empty file fails the run. Columns are matched by name, so their order does not matter.

### Output tables (SQLite)

| Table | Key and contents |
|---|---|
| `customers_history`, `orders_history` | Append-only, one row per version, primary key (natural key, `updated_at`). Stores business columns, derived columns, `record_hash`, `run_id`, `source_file`, `source_row_number`, `loaded_at`. |
| `customers_current`, `orders_current` | One row per natural key, the latest version only. Same columns as history. Replaced only when the incoming `updated_at` is newer. |
| `rejected_records` | One row per distinct bad record: `reject_id`, `dataset`, `source_file`, `row_number` (header is line 1), `reject_reasons`, `raw_record` (the original row as JSON), `first_seen_run_id`, `first_seen_at`, `last_seen_run_id`, `last_seen_at`, `resolved_run_id`, `resolved_at`. |
| `pipeline_runs` | One row per dataset per run: `run_id`, `dataset`, `source_file`, `status` (RUNNING, SUCCESS, FAILED, SKIPPED), `started_at`, `ended_at`, `duration_seconds`, `rows_in`, `rows_inserted`, `rows_already_loaded`, `rows_rejected`, `duplicates_dropped`, `late_records`, four per-stage timings, `failure_reason`, `warnings`. |

Derived fields: `email_domain` (text after the `@`), `order_amount_cents` (quantity times unit price times 100, exact decimal arithmetic rounded half up to a whole number of cents, so source and target totals reconcile exactly), `order_date` (UTC date of `order_ts`), and `record_hash` (SHA-256 of the cleaned business columns).

Naming: snake_case tables and columns; timestamps are ISO-8601 UTC with second precision (`2024-02-01T10:00:00Z`; an offset such as `+05:30` is converted to UTC and a timestamp without an offset is read as UTC); `run_id` and `source_file` appear on loaded and rejected rows; IDs that identify data come from the source and are never random.

## Validation rules and reject codes

Each rejected input row produces one reject record that lists every failed rule, for example `INVALID_ALLOWED_VALUE:status; OUT_OF_RANGE:unit_price`.

| Code | When |
|---|---|
| `MALFORMED_ROW` | Wrong number of fields on the line |
| `MISSING_REQUIRED_FIELD` | A required value is empty |
| `INVALID_TYPE` | Not a valid integer, decimal, date or timestamp (for example `2024-02-30`) |
| `INVALID_FORMAT` | Fails the column pattern (id format, email) |
| `INVALID_ALLOWED_VALUE` | Not in the allowed list; matching is case-sensitive |
| `OUT_OF_RANGE` | Below the minimum or above the maximum |
| `FUTURE_TIMESTAMP` | Date or timestamp later than the run time |
| `CONFLICTING_DUPLICATE` | Same key and version but different content, inside the file or against an already loaded record |
| `INVALID_REFERENCE` | `customer_id` has no matching row in `customers_current` |

A missing required column, a duplicated column name or a file with no header is not a row reject: it fails the dataset with a `failure_reason` such as `SCHEMA_MISSING_COLUMNS: ['email']`, and datasets after it in the same run are marked `SKIPPED`. Extra columns are ignored and recorded as a warning.

Duplicate handling inside one file: the first copy of an exact duplicate is kept and the rest are counted in `duplicates_dropped`. Copies with the same key and version but different content cannot be trusted, so all copies are rejected as `CONFLICTING_DUPLICATE`.

## Idempotency, retry, restart and late data

Loading one dataset is a single transaction: history, current and rejects are written together or not at all. A `RUNNING` row is committed to `pipeline_runs` before any work starts, so a crashed run leaves evidence; failures are recorded as `FAILED` with their reason. Before committing, the load checks `rows_in = rows_inserted + rows_already_loaded + rows_rejected + duplicates_dropped` and rolls everything back if it does not hold.

Rerunning the same file is safe. A version already stored with the same `record_hash` is counted as `rows_already_loaded` and nothing is written. Bad records already seen keep their first reject record and only `last_seen_run_id` and `last_seen_at` change; every run still counts them in its own `rows_rejected`. Retry and restart are the same command again: a failed transaction was rolled back, and anything already committed is skipped through the primary keys.

Late and out-of-order versions: an older version arriving after a newer one is stored in history, counted in `late_records`, and does not replace the current row, so for example an order stays `delivered` when an old `placed` version arrives later. A backfill is a run on an older file; old versions go to history and the current table is protected the same way.

Orders whose customer does not exist yet are rejected as `INVALID_REFERENCE`. Rerunning the same orders file after the customer has loaded inserts the order and marks the earlier reject as resolved (`resolved_run_id`, matched by file name and row number).

## Reproducible sample and larger run

The generator uses a fixed seed (default 42): the same seed and options always give identical files. It writes the **input** files (`customers.csv`, `orders.csv`) to one folder and the **answer key** to another, so labels never sit next to the data they describe:

| File in `data/expected/` | Meaning |
|---|---|
| `expected_rejects.csv` | Every planted bad row: file, line number, exact expected reject reason, and a readable defect name |
| `expected_summary.json` | Expected counts per dataset (rows in, inserted, rejected, duplicates, late, history and current rows) and order totals in cents |
| `expected_current_customers.csv`, `expected_current_orders.csv` | The latest version of every valid record |

It plants valid records with several versions, boundary orders (quantity 1 and 1000, price 0.00 and 100000.00), timestamps in `Z`, `+00:00` and `+05:30` forms, padded names and upper-case e-mails, older versions placed after newer ones in the file, exact duplicates, and 11 customer and 16 order defect types (missing values, wrong types, out-of-range values, conflicting duplicates, unknown customers, malformed lines and a row with two defects). Every defect type appears at any size. Generate a larger dataset (about 24,600 customer rows and 240,000 order rows) with:

```bash
python -m generator.generate_data --customers 20000 --orders 100000 --out-dir data/large --expected-dir data/large/expected
python run_pipeline.py --input-dir data/large --db output/large.db
python run_report.py --db output/large.db --input-dir data/large --expected-dir data/large/expected
```

## Correctness evidence and tests

`python -m pytest -q` runs 188 tests, each using its own temporary folder and database. They cover every required edge case: happy path, empty and header-only input, malformed rows, every required column blank, 28 invalid-value cases with exact reasons, boundary values and future-date edges, exact and conflicting duplicates, schema changes (missing, extra, repeated columns), late and out-of-order data (including an old status not overwriting a newer one), invalid references and their replay, reruns, failure mid-load with rollback and retry, a crash leaving a `RUNNING` row, log and metrics content, configuration mistakes, a new dataset added with config only, the command-line exit codes, deterministic generation, the reconciliation report (including corrupting the database on purpose to prove each check fails), and the performance tooling. Separate tests compare the pipeline's output with the generator's answer key for three seeds, the committed sample, and a 10,000-order run.

`python run_report.py --input-dir data/sample --expected-dir data/expected` prints a reconciliation report with 36 checks, each counting violations (0 = PASS):

- `rows_in = inserted + already loaded + rejected + duplicates dropped` for every successful run;
- the source file re-counted independently of the pipeline equals `rows_in`;
- primary keys are unique, every current row equals the newest history row, every history key has a current row, history rows equal the sum of `rows_inserted`;
- every order's customer exists, and the order total in cents agrees between current and the newest history versions;
- every reject has a reason, source file, line number and original row, and uses a documented reason code;
- the run counts, table row counts, rejected rows and reasons, current rows and order totals equal the answer key.

`sql/reconciliation.sql` holds 14 of the same checks as plain SQL and `sql/sample_queries.sql` holds 14 analysis queries (current status counts, revenue by country, order timelines from history, reject summaries, lineage, run history and throughput). `reports/reconciliation_report.md` and `reports/reconciliation_report_large.md` are saved results: both pass all 36 checks.

## Performance and trade-offs

`python run_benchmark.py` generates the data, runs the pipeline repeatedly on an empty database, and writes `reports/performance_report.md`. Measured on Windows, Python 3.14, pandas 3.1.1, 12 CPUs, with 24,600 customer rows and 240,373 order rows:

- The whole run takes about 10.9 s, roughly 24,000 rows per second. Within the orders dataset, load is the largest stage (44%), then transform (26%) and validate (25%); extract is 5%. Throughput stays between about 21,000 and 24,000 rows per second from 5,000 to 100,000 orders, so cost grows in proportion to the data.
- A profile of the first version showed about 2.5 million slow per-value accesses into pandas string columns, coming from row loops that iterate a column directly. The optimization converts each column to a plain list once (`_plain` in `etl/transform.py`). It made the transform stage 13% faster (faster in 5 of 5 paired repeats) and the whole run about 5% faster (4 of 5 repeats), and cut the slow accesses from 2,553,363 to 523,247. A content fingerprint of every table is identical before and after, so the stored data did not change.
- The gain is moderate. SQLite inserts and pandas' own string handling dominate what remains. Installing `pyarrow` would likely help more, but it is an extra dependency and was not measured.
- Other ideas measured separately during development changed the total by only 2 to 4%, within run-to-run noise, and were not adopted. Memory use was not measured.

Design trade-offs: SQLite and pandas keep the setup to one `pip install` and fit a single-machine batch job, at the cost of single-writer loading and string handling that is slower than a columnar engine. History is kept as an append-only table plus a current table instead of SCD Type 2: late data becomes one more history row and cannot corrupt date ranges, and `valid_to` can be derived with a window function when needed (sample query 4).

## Known limitations

- Timestamps are kept to second precision; sub-second differences are dropped.
- `unit_price` is stored as a floating-point `REAL`; the exact money value is `order_amount_cents`.
- Marking a reject as resolved matches by file name and row number, so a file edited in a way that shifts rows can resolve a reject for a different record.
- Rejects are checked in stages: a row rejected by an earlier stage is not checked by later stages (a row with a rule failure is not also reference-checked).
- A parent dataset must come before its children in `config/pipeline.yaml`.
- Batch processing with a single writer; concurrent writers are not handled.
- Changing a dataset config after its tables exist does not migrate the existing tables.
- The reconciliation report compares the latest successful run of each dataset and assumes nothing else writes to the database.
- The answer key and the pipeline share the same reject-reason names, so a naming mistake made in both would not be caught by the comparison.
- Developed and tested on Windows with Python 3.14 and pandas 3.1.1. The Colab notebook has been executed cell by cell on that machine; other platforms and versions are untested until it is run on Colab.

## Not built yet

A separate design and trade-off note (the trade-offs above are a summary).

## Project contents

```text
config/            pipeline.yaml and one YAML file per dataset (customers, orders)
etl/               extract, validate, transform, load, metrics, logging_setup, pipeline,
                   config_loader, parsing, errors, reconcile
generator/         seeded synthetic data generator with a separate answer key
sql/               reconciliation.sql and sample_queries.sql
tests/             188 automated tests
data/sample/       generated customers.csv and orders.csv (seed 42)
data/expected/     the answer key for the sample
reports/           saved reconciliation and performance reports
run_pipeline.py    command-line entry point
run_report.py      reconciliation report
run_benchmark.py   performance measurement
requirements.txt   pinned dependencies
logs/ output/      created at run time (JSON log file, SQLite database)
```


## 1. Setup

`pip` may print dependency warnings about other packages that Colab pre-installs; they do not affect this project, whose commands all run as separate Python processes with the versions installed here.

In [1]:
!pip -q install pandas==3.1.1 PyYAML==6.0.3 pytest==9.2.0
!python -c "import sys, platform, sqlite3, pandas, yaml, pytest; print('Python :', sys.version.split()[0]); print('Pandas :', pandas.__version__); print('PyYAML :', yaml.__version__); print('Pytest :', pytest.__version__); print('SQLite :', sqlite3.sqlite_version); print('OS     :', platform.platform())"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
Python : 3.13.18
Pandas : 3.1.1
PyYAML : 6.0.3
Pytest : 9.2.0
SQLite : 3.45.1
OS     : Linux-6.6.159+-x86_64-with-glibc2.39


## 2. Project files

Each cell below writes one file, exactly as it was tested.

In [2]:
from pathlib import Path

for folder in ['config/datasets', 'data/expected', 'data/large', 'data/sample', 'etl', 'generator', 'logs', 'output', 'reports', 'sql', 'tests']:
    Path(folder).mkdir(parents=True, exist_ok=True)

In [3]:
%%writefile requirements.txt
# Pinned to the versions verified on the development machine (Python 3.14.0).
pandas==3.1.1
PyYAML==6.0.3
pytest==9.2.0


Writing requirements.txt


In [4]:
%%writefile pytest.ini
[pytest]
pythonpath = .
testpaths = tests


Writing pytest.ini


In [5]:
%%writefile config/pipeline.yaml
# Global pipeline settings. Paths are relative to the project root (etl_framework/).
database_path: output/warehouse.db
input_dir: data/sample
log_file: logs/pipeline.log
reports_dir: reports

# Validation implementation used by validate.py.
#   vectorized = column-at-a-time pandas checks (default, fast)
#   rowwise    = one Python call per row (kept only for the performance before/after test)
validation_mode: vectorized

# Datasets run in this order. A parent dataset must run before its children,
# because the reference check looks up the parent's *_current table.
datasets:
  - config/datasets/customers.yaml
  - config/datasets/orders.yaml


Writing config/pipeline.yaml


In [6]:
%%writefile config/datasets/customers.yaml
# Dataset: customers (parent)
# Grain: one source row = one version of a customer at a point in time (customer_id, updated_at).
name: customers
source_file: customers.csv

natural_key: [customer_id]      # identifies the business entity
version_column: updated_at      # orders versions; newer replaces current, older goes to history only

target:
  history_table: customers_history   # append-only, PK (customer_id, updated_at)
  current_table: customers_current   # latest version only, PK (customer_id)

# Supported types: string, integer, decimal, date, timestamp
# Supported rules: required, pattern, allowed_values, min, max, not_in_future
# Supported transforms (applied after validation): lowercase, uppercase
# Leading/trailing whitespace is always trimmed before validation.
columns:
  customer_id:
    type: string
    required: true
    pattern: '^C\d{6}$'
  full_name:
    type: string
    required: true
  email:
    type: string
    required: true
    pattern: '^[^@\s]+@[^@\s]+\.[^@\s]+$'
    transform: lowercase
  country_code:
    type: string
    required: true
    allowed_values: [IN, US, GB, AE, SG]
  signup_date:
    type: date
    required: true
    not_in_future: true
  updated_at:
    type: timestamp
    required: true
    not_in_future: true

references: []   # customers has no parent

# Supported operations: split_part, multiply_to_cents, date_part
derived:
  email_domain:
    operation: split_part
    sources: [email]
    delimiter: "@"
    index: 1


Writing config/datasets/customers.yaml


In [7]:
%%writefile config/datasets/orders.yaml
# Dataset: orders (child of customers)
# Grain: one source row = one version of an order (order_id, updated_at),
# e.g. the same order appears as placed -> shipped -> delivered.
name: orders
source_file: orders.csv

natural_key: [order_id]
version_column: updated_at

target:
  history_table: orders_history   # append-only, PK (order_id, updated_at)
  current_table: orders_current   # latest version only, PK (order_id)

columns:
  order_id:
    type: string
    required: true
    pattern: '^O\d{8}$'
  customer_id:
    type: string
    required: true
    pattern: '^C\d{6}$'
  order_ts:
    type: timestamp
    required: true
    not_in_future: true
  status:
    type: string
    required: true
    allowed_values: [placed, shipped, delivered, cancelled]
  quantity:
    type: integer
    required: true
    min: 1
    max: 1000
  unit_price:
    type: decimal
    required: true
    min: 0          # boundary: 0.00 is allowed (free item)
    max: 100000
  currency:
    type: string
    required: true
    allowed_values: [INR, USD]
  updated_at:
    type: timestamp
    required: true
    not_in_future: true

# Referential integrity: every order must point to an existing customer.
# Failing rows are rejected as INVALID_REFERENCE and load on a later rerun
# once the customer exists (idempotent replay).
references:
  - column: customer_id
    parent_table: customers_current
    parent_column: customer_id

derived:
  order_amount_cents:          # integer cents so sums reconcile exactly
    operation: multiply_to_cents
    sources: [quantity, unit_price]
  order_date:
    operation: date_part
    sources: [order_ts]


Writing config/datasets/orders.yaml


In [8]:
%%writefile etl/__init__.py
"""Reusable mini ETL framework core. Contains no dataset-specific logic."""


Writing etl/__init__.py


In [9]:
%%writefile etl/config_loader.py
"""Load pipeline.yaml and dataset YAML files and check they are well-formed.

Why check so much: a typo such as `requried: true` would otherwise be ignored
silently and a rule would quietly stop working. We fail early with a clear message.
"""
from __future__ import annotations

import re
from pathlib import Path
from typing import Any

import yaml

from etl.errors import ConfigError
from etl.transform import DERIVED_SQL_TYPES

PROJECT_ROOT = Path(__file__).resolve().parents[1]
IDENTIFIER = re.compile(r"^[a-z][a-z0-9_]*$")  # snake_case; also keeps names safe to use in SQL
COLUMN_TYPES = {"string", "integer", "decimal", "date", "timestamp"}
COLUMN_KEYS = {"type", "required", "pattern", "allowed_values", "min", "max", "not_in_future", "transform"}
VALIDATION_MODES = {"vectorized", "rowwise"}


def resolve_path(value: str | Path) -> Path:
    """Relative paths are relative to the project root; absolute paths are kept."""
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def _read_yaml(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise ConfigError(f"config file not found: {path}")
    with open(path, encoding="utf-8") as handle:
        content = yaml.safe_load(handle)
    if not isinstance(content, dict):
        raise ConfigError(f"config file is not a YAML mapping: {path}")
    return content


def _require(config: dict, key: str, where: str) -> Any:
    if key not in config or config[key] in (None, ""):
        raise ConfigError(f"{where}: missing required key '{key}'")
    return config[key]


def _check_identifier(value: Any, where: str) -> None:
    if not isinstance(value, str) or not IDENTIFIER.match(value):
        raise ConfigError(f"{where}: '{value}' must be snake_case (lowercase letters, digits, underscores)")


def load_pipeline_config(path: str | Path) -> dict[str, Any]:
    """Read pipeline.yaml and resolve its paths."""
    config = _read_yaml(resolve_path(path))
    for key in ("database_path", "input_dir", "log_file", "reports_dir", "datasets"):
        _require(config, key, "pipeline.yaml")
    mode = config.setdefault("validation_mode", "vectorized")
    if mode not in VALIDATION_MODES:
        raise ConfigError(f"pipeline.yaml: validation_mode must be one of {sorted(VALIDATION_MODES)}")
    if not isinstance(config["datasets"], list) or not config["datasets"]:
        raise ConfigError("pipeline.yaml: 'datasets' must be a non-empty list of dataset config paths")
    for key in ("database_path", "input_dir", "log_file", "reports_dir"):
        config[key] = resolve_path(config[key])
    config["datasets"] = [resolve_path(item) for item in config["datasets"]]
    return config


def load_dataset_config(path: str | Path) -> dict[str, Any]:
    """Read one dataset YAML file and validate its structure."""
    config = _read_yaml(resolve_path(path))
    validate_dataset_config(config)
    return config


def validate_dataset_config(config: dict[str, Any]) -> None:
    """Raise ConfigError if the dataset config is incomplete or uses unsupported options."""
    name = _require(config, "name", "dataset config")
    where = f"dataset '{name}'"
    _check_identifier(name, where)
    _require(config, "source_file", where)
    columns = _require(config, "columns", where)
    natural_key = _require(config, "natural_key", where)
    version_column = _require(config, "version_column", where)
    target = _require(config, "target", where)
    for key in ("history_table", "current_table"):
        _check_identifier(_require(target, key, f"{where} target"), f"{where} target.{key}")

    for col, spec in columns.items():
        _check_identifier(col, f"{where} column")
        _validate_column_spec(col, spec, where)
    for key_col in [*natural_key, version_column]:
        if key_col not in columns:
            raise ConfigError(f"{where}: key column '{key_col}' is not defined under columns")
        if not columns[key_col].get("required", False):
            raise ConfigError(f"{where}: key column '{key_col}' must be required: true")

    for ref in config.get("references") or []:
        for key in ("column", "parent_table", "parent_column"):
            _require(ref, key, f"{where} reference")
        if ref["column"] not in columns:
            raise ConfigError(f"{where}: reference column '{ref['column']}' is not defined under columns")
        _check_identifier(ref["parent_table"], f"{where} reference parent_table")
        _check_identifier(ref["parent_column"], f"{where} reference parent_column")

    for derived_name, spec in (config.get("derived") or {}).items():
        _validate_derived(derived_name, spec, columns, where)


def _validate_column_spec(col: str, spec: dict, where: str) -> None:
    label = f"{where} column '{col}'"
    if spec.get("type") not in COLUMN_TYPES:
        raise ConfigError(f"{label}: type must be one of {sorted(COLUMN_TYPES)}")
    unknown = set(spec) - COLUMN_KEYS
    if unknown:
        raise ConfigError(f"{label}: unknown option(s) {sorted(unknown)}; supported: {sorted(COLUMN_KEYS)}")
    kind = spec["type"]
    if ("min" in spec or "max" in spec) and kind not in ("integer", "decimal"):
        raise ConfigError(f"{label}: min/max only apply to integer or decimal columns")
    if spec.get("not_in_future") and kind not in ("date", "timestamp"):
        raise ConfigError(f"{label}: not_in_future only applies to date or timestamp columns")
    if "transform" in spec and (kind != "string" or spec["transform"] not in ("lowercase", "uppercase")):
        raise ConfigError(f"{label}: transform must be 'lowercase' or 'uppercase' on a string column")
    if "allowed_values" in spec and not isinstance(spec["allowed_values"], list):
        raise ConfigError(f"{label}: allowed_values must be a list")
    if "pattern" in spec:
        try:
            re.compile(spec["pattern"])
        except re.error as exc:
            raise ConfigError(f"{label}: invalid regex pattern ({exc})") from exc


def _validate_derived(name: str, spec: dict, columns: dict, where: str) -> None:
    label = f"{where} derived '{name}'"
    _check_identifier(name, label)
    if name in columns:
        raise ConfigError(f"{label}: clashes with a source column of the same name")
    operation = spec.get("operation")
    if operation not in DERIVED_SQL_TYPES:
        raise ConfigError(f"{label}: operation must be one of {sorted(DERIVED_SQL_TYPES)}")
    sources = spec.get("sources") or []
    for source in sources:
        if source not in columns:
            raise ConfigError(f"{label}: source column '{source}' is not defined under columns")
    if operation == "split_part":
        if len(sources) != 1 or "delimiter" not in spec or "index" not in spec:
            raise ConfigError(f"{label}: split_part needs one source, a delimiter and an index")
    elif operation == "multiply_to_cents":
        if not sources or any(columns[s]["type"] not in ("integer", "decimal") for s in sources):
            raise ConfigError(f"{label}: multiply_to_cents needs integer/decimal source columns")
    elif operation == "date_part":
        if len(sources) != 1 or columns[sources[0]]["type"] not in ("date", "timestamp"):
            raise ConfigError(f"{label}: date_part needs one date/timestamp source column")


Writing etl/config_loader.py


In [10]:
%%writefile etl/errors.py
"""Custom exceptions so failures have clear, searchable names in logs and metrics."""


class ConfigError(Exception):
    """A config file is missing a key, has a typo, or uses an unsupported option."""


class SchemaError(Exception):
    """The source file's columns do not match the dataset config (e.g. a required column is missing)."""


class ReconciliationError(Exception):
    """Row counts do not add up: some input row was neither loaded, rejected nor counted."""


Writing etl/errors.py


In [11]:
%%writefile etl/extract.py
"""Extract: read a source CSV into a DataFrame of text values.

Why the csv module and not pandas.read_csv: we need the exact line number of
every row, and we must catch rows with the wrong number of fields (malformed)
instead of letting pandas skip or crash on them.
"""
from __future__ import annotations

import csv
import json
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

from etl.errors import SchemaError


@dataclass
class ExtractResult:
    """What extract hands to the next step."""

    header: list[str]
    data: pd.DataFrame        # well-formed rows; every value is text; index = source line number
    malformed: pd.DataFrame   # rows with the wrong field count, already in reject format
    blank_lines: int          # fully blank lines (ignored, but counted and logged)

    @property
    def rows_in(self) -> int:
        """Number of data rows read (well-formed + malformed). The header is not a data row."""
        return len(self.data) + len(self.malformed)


def extract_csv(path: Path) -> ExtractResult:
    """Read a CSV file. Raises FileNotFoundError if missing, SchemaError if it has no header."""
    row_numbers: list[int] = []
    rows: list[list[str]] = []
    bad_numbers: list[int] = []
    bad_records: list[str] = []
    blank_lines = 0

    with open(path, newline="", encoding="utf-8-sig") as handle:  # utf-8-sig removes an Excel BOM
        reader = csv.reader(handle)
        header = next(reader, None)
        if header is None:
            raise SchemaError("SCHEMA_NO_HEADER: file is empty")
        header = [name.strip() for name in header]

        for fields in reader:
            line_number = reader.line_num  # header is line 1
            if not fields:
                blank_lines += 1
            elif len(fields) != len(header):
                bad_numbers.append(line_number)
                bad_records.append(json.dumps({"raw_fields": fields}, ensure_ascii=False))
            else:
                row_numbers.append(line_number)
                rows.append(fields)

    index = pd.Index(row_numbers, name="row_number", dtype="int64")
    data = pd.DataFrame(rows, columns=header, index=index, dtype="string")
    malformed = pd.DataFrame(
        {"reject_reasons": "MALFORMED_ROW", "raw_record": bad_records},
        index=pd.Index(bad_numbers, name="row_number", dtype="int64"),
    )
    return ExtractResult(header=header, data=data, malformed=malformed, blank_lines=blank_lines)


Writing etl/extract.py


In [12]:
%%writefile etl/load.py
"""Load: write history, current and reject tables to SQLite in ONE transaction.

Idempotency rules (running the same input twice leaves the database unchanged):
  * history table : append-only, primary key (natural key + version). A version that is
                    already stored with the same record_hash is counted as 'already loaded'.
                    A stored version with a DIFFERENT record_hash is rejected (conflict),
                    never silently ignored.
  * current table : one row per natural key; replaced only when the incoming version is NEWER.
  * rejected_records : one row per distinct bad record; seen again later -> only last_seen_* updates.
Everything for one dataset commits together or not at all.
"""
from __future__ import annotations

import hashlib
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

from etl.errors import ReconciliationError
from etl.transform import DERIVED_SQL_TYPES
from etl.validate import build_reject_rows, empty_rejects

COLUMN_SQL_TYPES = {"string": "TEXT", "integer": "INTEGER", "decimal": "REAL", "date": "TEXT", "timestamp": "TEXT"}
META_COLUMNS = ["run_id", "source_file", "source_row_number", "loaded_at"]

REJECTED_RECORDS_DDL = """
CREATE TABLE IF NOT EXISTS rejected_records (
    reject_id          TEXT PRIMARY KEY,   -- hash of dataset|source_file|row_number|raw_record (stable across runs)
    dataset            TEXT NOT NULL,
    source_file        TEXT NOT NULL,
    row_number         INTEGER NOT NULL,   -- line number in the source file (header = line 1)
    reject_reasons     TEXT NOT NULL,
    raw_record         TEXT NOT NULL,      -- the original row as JSON, so it can be fixed and replayed
    first_seen_run_id  TEXT NOT NULL,
    first_seen_at      TEXT NOT NULL,
    last_seen_run_id   TEXT NOT NULL,
    last_seen_at       TEXT NOT NULL,
    resolved_run_id    TEXT,               -- run in which this row position later loaded successfully
    resolved_at        TEXT
)
"""


@dataclass
class LoadResult:
    """Counts produced by the load step."""

    rows_inserted: int
    rows_already_loaded: int
    rows_rejected: int       # all rejects for the dataset, including conflicts found at load time
    late_records: int


def connect(db_path: Path) -> sqlite3.Connection:
    """Open SQLite in autocommit mode; we control transactions explicitly with BEGIN/COMMIT."""
    db_path.parent.mkdir(parents=True, exist_ok=True)
    return sqlite3.connect(db_path, isolation_level=None)


def ensure_reject_table(conn: sqlite3.Connection) -> None:
    conn.execute(REJECTED_RECORDS_DDL)


def dataset_columns(config: dict[str, Any]) -> list[str]:
    """Business + derived columns + record_hash, in table order (metadata columns come after)."""
    return [*config["columns"], *(config.get("derived") or {}), "record_hash"]


def _table_ddl(table: str, config: dict[str, Any], primary_key: list[str]) -> str:
    lines = []
    for col, spec in config["columns"].items():
        not_null = " NOT NULL" if spec.get("required", False) else ""
        lines.append(f"    {col} {COLUMN_SQL_TYPES[spec['type']]}{not_null}")
    for col, spec in (config.get("derived") or {}).items():
        lines.append(f"    {col} {DERIVED_SQL_TYPES[spec['operation']]}")
    lines.append("    record_hash TEXT NOT NULL")
    lines += ["    run_id TEXT NOT NULL", "    source_file TEXT NOT NULL",
              "    source_row_number INTEGER NOT NULL", "    loaded_at TEXT NOT NULL"]
    lines.append(f"    PRIMARY KEY ({', '.join(primary_key)})")
    return f"CREATE TABLE IF NOT EXISTS {table} (\n" + ",\n".join(lines) + "\n)"


def ensure_dataset_tables(conn: sqlite3.Connection, config: dict[str, Any]) -> None:
    """Create the history and current tables for a dataset (from config) if needed."""
    key = config["natural_key"]
    conn.execute(_table_ddl(config["target"]["history_table"], config, [*key, config["version_column"]]))
    conn.execute(_table_ddl(config["target"]["current_table"], config, key))


def fetch_parent_keys(conn: sqlite3.Connection, table: str, column: str) -> set:
    """All values of a parent key column. A parent table that does not exist yet means no keys."""
    exists = conn.execute("SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = ?", (table,)).fetchone()
    if not exists:
        return set()
    return {row[0] for row in conn.execute(f"SELECT DISTINCT {column} FROM {table}")}


def _py_rows(df: pd.DataFrame, columns: list[str]) -> list[tuple]:
    """DataFrame rows as plain Python tuples (missing values -> None) that sqlite3 can bind."""
    sub = df[columns].astype(object)
    sub = sub.where(sub.notna(), None)
    return [tuple(row) for row in sub.itertuples(index=False, name=None)]


def _read_existing(conn: sqlite3.Connection, config: dict[str, Any], df: pd.DataFrame
                   ) -> tuple[dict[int, str], dict[tuple, str]]:
    """Look up incoming rows in the target: {row_number: stored record_hash}, {key: current version}."""
    if df.empty:
        return {}, {}
    keys, version = config["natural_key"], config["version_column"]
    hist, cur = config["target"]["history_table"], config["target"]["current_table"]
    incoming = df[[*keys, version, "record_hash"]].copy()
    incoming["row_number"] = df.index
    columns = [*keys, version, "record_hash", "row_number"]
    conn.execute(f"CREATE TEMP TABLE _incoming ({', '.join(columns)})")
    try:
        marks = ", ".join("?" * len(columns))
        conn.executemany(f"INSERT INTO _incoming VALUES ({marks})", _py_rows(incoming, columns))
        join_hist = " AND ".join(f"h.{c} = i.{c}" for c in [*keys, version])
        hashes = {row[0]: row[1] for row in conn.execute(
            f"SELECT i.row_number, h.record_hash FROM _incoming i JOIN {hist} h ON {join_hist}")}
        join_cur = " AND ".join(f"c.{c} = i.{c}" for c in keys)
        key_list = ", ".join(f"i.{c}" for c in keys)
        versions = {tuple(row[:-1]): row[-1] for row in conn.execute(
            f"SELECT DISTINCT {key_list}, c.{version} FROM _incoming i JOIN {cur} c ON {join_cur}")}
    finally:
        conn.execute("DROP TABLE IF EXISTS _incoming")
    return hashes, versions


def _flag_late(keys: list[tuple], versions: list[str], known: dict[tuple, str]) -> list[bool]:
    """A row is late if a NEWER version of the same key is already known (in the target or earlier in the file)."""
    newest = dict(known)
    flags = []
    for key, version in zip(keys, versions):
        top = newest.get(key)
        flags.append(top is not None and version < top)
        if top is None or version > top:
            newest[key] = version
    return flags


def _reject_id(dataset: str, source_file: str, row_number: int, raw_record: str) -> str:
    return hashlib.sha256(f"{dataset}|{source_file}|{row_number}|{raw_record}".encode("utf-8")).hexdigest()


def load_dataset(conn: sqlite3.Connection, config: dict[str, Any], df: pd.DataFrame, rejects: pd.DataFrame,
                 raw: pd.DataFrame, *, run_id: str, source_file: str, rows_in: int,
                 duplicates_dropped: int, now_iso: str) -> LoadResult:
    """Write one dataset atomically. Raises (after rolling back) on any error, including count mismatch."""
    ensure_dataset_tables(conn, config)
    conn.execute("BEGIN")
    try:
        result = _load_in_transaction(conn, config, df, rejects, raw, run_id=run_id, source_file=source_file,
                                      rows_in=rows_in, duplicates_dropped=duplicates_dropped, now_iso=now_iso)
        conn.execute("COMMIT")
    except BaseException:
        if conn.in_transaction:
            conn.execute("ROLLBACK")
        raise
    return result


def _load_in_transaction(conn: sqlite3.Connection, config: dict[str, Any], df: pd.DataFrame,
                         rejects: pd.DataFrame, raw: pd.DataFrame, *, run_id: str, source_file: str,
                         rows_in: int, duplicates_dropped: int, now_iso: str) -> LoadResult:
    keys, version = config["natural_key"], config["version_column"]
    hist, cur = config["target"]["history_table"], config["target"]["current_table"]
    dataset = config["name"]
    df = df.sort_index()

    # 1. Compare incoming rows with what is already stored.
    stored_hash, current_version = _read_existing(conn, config, df)
    stored = pd.Series(df.index.map(stored_hash.get), index=df.index, dtype="object")
    is_new = stored.isna()
    is_same = stored.notna() & (stored == df["record_hash"])
    is_conflict = stored.notna() & ~is_same
    new_rows, already_rows = df[is_new], df[is_same]

    # 2. Conflicts with an already-stored version are rejected, not ignored.
    conflict_reasons = pd.Series(
        f"CONFLICTING_DUPLICATE:{','.join([*keys, version])} (differs from already loaded record)",
        index=pd.Index(df.index[is_conflict], name="row_number"), dtype="object")
    all_rejects = pd.concat([rejects, build_reject_rows(raw, conflict_reasons)]).sort_index() \
        if not conflict_reasons.empty else rejects

    # 3. Late records: new versions that are older than a version we already know.
    key_tuples = list(zip(*[new_rows[c] for c in keys])) if len(new_rows) else []
    late_flags = _flag_late(key_tuples, list(new_rows[version]), current_version)

    # 4. Write history (append) and current (only if newer).
    columns = dataset_columns(config)
    stamped = new_rows.copy()
    stamped["run_id"], stamped["source_file"] = run_id, source_file
    stamped["source_row_number"], stamped["loaded_at"] = stamped.index, now_iso
    write_columns = [*columns, *META_COLUMNS]
    marks = ", ".join("?" * len(write_columns))
    conn.executemany(f"INSERT INTO {hist} ({', '.join(write_columns)}) VALUES ({marks})",
                     _py_rows(stamped, write_columns))
    latest = stamped.sort_values(version, kind="stable").drop_duplicates(keys, keep="last")
    updates = ", ".join(f"{c} = excluded.{c}" for c in write_columns if c not in keys)
    conn.executemany(
        f"INSERT INTO {cur} ({', '.join(write_columns)}) VALUES ({marks}) "
        f"ON CONFLICT({', '.join(keys)}) DO UPDATE SET {updates} WHERE excluded.{version} > {cur}.{version}",
        _py_rows(latest, write_columns))

    # 5. Rejects: insert once, or just refresh last_seen_* if we saw the same bad record before.
    reject_rows = [
        (_reject_id(dataset, source_file, int(n), rec), dataset, source_file, int(n), reasons, rec,
         run_id, now_iso, run_id, now_iso)
        for n, reasons, rec in zip(all_rejects.index, all_rejects["reject_reasons"], all_rejects["raw_record"])]
    conn.executemany(
        "INSERT INTO rejected_records (reject_id, dataset, source_file, row_number, reject_reasons, raw_record, "
        "first_seen_run_id, first_seen_at, last_seen_run_id, last_seen_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?) "
        "ON CONFLICT(reject_id) DO UPDATE SET reject_reasons = excluded.reject_reasons, "
        "last_seen_run_id = excluded.last_seen_run_id, last_seen_at = excluded.last_seen_at, "
        "resolved_run_id = NULL, resolved_at = NULL", reject_rows)

    # 6. An earlier reject at a row position that now loads (or is already loaded) is marked resolved.
    ok_rows = [(int(n),) for n in [*new_rows.index, *already_rows.index]]
    if ok_rows:
        conn.execute("CREATE TEMP TABLE _ok_rows (row_number INTEGER)")
        try:
            conn.executemany("INSERT INTO _ok_rows VALUES (?)", ok_rows)
            conn.execute(
                "UPDATE rejected_records SET resolved_run_id = ?, resolved_at = ? "
                "WHERE dataset = ? AND source_file = ? AND resolved_run_id IS NULL "
                "AND row_number IN (SELECT row_number FROM _ok_rows)", (run_id, now_iso, dataset, source_file))
        finally:
            conn.execute("DROP TABLE IF EXISTS _ok_rows")

    # 7. Safety net: every input row must be accounted for, or the whole load is rolled back.
    result = LoadResult(rows_inserted=len(new_rows), rows_already_loaded=len(already_rows),
                        rows_rejected=len(all_rejects), late_records=sum(late_flags))
    accounted = result.rows_inserted + result.rows_already_loaded + result.rows_rejected + duplicates_dropped
    if accounted != rows_in:
        raise ReconciliationError(
            f"rows_in={rows_in} but inserted+already_loaded+rejected+duplicates_dropped={accounted}")
    return result


Writing etl/load.py


In [13]:
%%writefile etl/logging_setup.py
"""Logging: one JSON object per line, every line tagged with run_id.

Why JSON lines: they are easy to search by machine, e.g. "all ERROR lines for run X".
Logs go to stderr (console) and to a log file (appended).
"""
from __future__ import annotations

import json
import logging
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

LOGGER_NAME = "etl"
_STANDARD_ATTRIBUTES = set(logging.LogRecord("", 0, "", 0, "", (), None).__dict__) | {"message", "asctime"}
_LEADING_FIELDS = ("run_id", "dataset", "step")


class JsonFormatter(logging.Formatter):
    """Format a log record as a single JSON line."""

    def format(self, record: logging.LogRecord) -> str:
        extras = {k: v for k, v in record.__dict__.items() if k not in _STANDARD_ATTRIBUTES}
        moment = datetime.fromtimestamp(record.created, tz=timezone.utc)
        payload: dict[str, Any] = {"ts": moment.strftime("%Y-%m-%dT%H:%M:%SZ"), "level": record.levelname}
        for name in _LEADING_FIELDS:
            if name in extras:
                payload[name] = extras.pop(name)
        payload["message"] = record.getMessage()
        payload.update(extras)
        if record.exc_info:
            payload["exception"] = self.formatException(record.exc_info)
        return json.dumps(payload, default=str, ensure_ascii=False)


class RunLogger(logging.LoggerAdapter):
    """Adds run_id to every log call and merges any per-call fields (dataset, step, counts...)."""

    def process(self, msg: Any, kwargs: Any) -> tuple[Any, Any]:
        kwargs["extra"] = {**self.extra, **kwargs.get("extra", {})}
        return msg, kwargs


def setup_logging(log_file: Path | None, level: int = logging.INFO) -> None:
    """(Re)configure the framework logger. Safe to call many times (e.g. once per test)."""
    shutdown_logging()
    logger = logging.getLogger(LOGGER_NAME)
    logger.setLevel(level)
    logger.propagate = False
    formatter = JsonFormatter()
    console = logging.StreamHandler(sys.stderr)
    console.setFormatter(formatter)
    logger.addHandler(console)
    if log_file is not None:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        file_handler = logging.FileHandler(log_file, encoding="utf-8")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)


def shutdown_logging() -> None:
    """Close and remove handlers (releases the log file, which matters on Windows)."""
    logger = logging.getLogger(LOGGER_NAME)
    for handler in list(logger.handlers):
        handler.close()
        logger.removeHandler(handler)


def get_run_logger(run_id: str) -> RunLogger:
    """Logger that stamps run_id on every line."""
    return RunLogger(logging.getLogger(LOGGER_NAME), {"run_id": run_id})


Writing etl/logging_setup.py


In [14]:
%%writefile etl/metrics.py
"""Metrics: the pipeline_runs table (one row per dataset per run).

A row is written as RUNNING *before* any work starts and committed immediately,
so even a crashed run leaves evidence. It is later updated to SUCCESS or FAILED.
"""
from __future__ import annotations

import sqlite3
from typing import Any

PIPELINE_RUNS_DDL = """
CREATE TABLE IF NOT EXISTS pipeline_runs (
    run_id               TEXT NOT NULL,
    dataset              TEXT NOT NULL,
    source_file          TEXT NOT NULL,
    status               TEXT NOT NULL,          -- RUNNING / SUCCESS / FAILED / SKIPPED
    started_at           TEXT NOT NULL,
    ended_at             TEXT,
    duration_seconds     REAL,
    rows_in              INTEGER,
    rows_inserted        INTEGER,
    rows_already_loaded  INTEGER,
    rows_rejected        INTEGER,
    duplicates_dropped   INTEGER,
    late_records         INTEGER,
    extract_seconds      REAL,
    validate_seconds     REAL,
    transform_seconds    REAL,
    load_seconds         REAL,
    failure_reason       TEXT,
    warnings             TEXT,
    PRIMARY KEY (run_id, dataset)
)
"""

COUNT_FIELDS = ("rows_in", "rows_inserted", "rows_already_loaded", "rows_rejected",
                "duplicates_dropped", "late_records")
TIMING_FIELDS = ("extract_seconds", "validate_seconds", "transform_seconds", "load_seconds")


def init_metrics(conn: sqlite3.Connection) -> None:
    """Create the pipeline_runs table if it does not exist."""
    conn.execute(PIPELINE_RUNS_DDL)


def start_run(conn: sqlite3.Connection, run_id: str, dataset: str, source_file: str, started_at: str) -> None:
    """Record that a dataset run has begun (status RUNNING)."""
    conn.execute(
        "INSERT INTO pipeline_runs (run_id, dataset, source_file, status, started_at) VALUES (?, ?, ?, 'RUNNING', ?)",
        (run_id, dataset, source_file, started_at),
    )


def _finish(conn: sqlite3.Connection, run_id: str, dataset: str, status: str, ended_at: str,
            duration: float, counts: dict[str, int], timings: dict[str, float],
            failure_reason: str | None, warnings: list[str]) -> None:
    values: dict[str, Any] = {"status": status, "ended_at": ended_at, "duration_seconds": round(duration, 4),
                              "failure_reason": failure_reason,
                              "warnings": "; ".join(warnings) if warnings else None}
    values.update({name: counts.get(name) for name in COUNT_FIELDS})
    values.update({name: round(timings[name], 4) if name in timings else None for name in TIMING_FIELDS})
    assignments = ", ".join(f"{name} = ?" for name in values)
    conn.execute(f"UPDATE pipeline_runs SET {assignments} WHERE run_id = ? AND dataset = ?",
                 [*values.values(), run_id, dataset])


def finish_success(conn: sqlite3.Connection, run_id: str, dataset: str, ended_at: str, duration: float,
                   counts: dict[str, int], timings: dict[str, float], warnings: list[str]) -> None:
    """Mark a dataset run SUCCESS and store its counts and timings."""
    _finish(conn, run_id, dataset, "SUCCESS", ended_at, duration, counts, timings, None, warnings)


def finish_failed(conn: sqlite3.Connection, run_id: str, dataset: str, ended_at: str, duration: float,
                  reason: str, counts: dict[str, int], timings: dict[str, float], warnings: list[str]) -> None:
    """Mark a dataset run FAILED with the failure reason (counts are whatever was known)."""
    _finish(conn, run_id, dataset, "FAILED", ended_at, duration, counts, timings, reason, warnings)


def record_skipped(conn: sqlite3.Connection, run_id: str, dataset: str, source_file: str,
                   now: str, reason: str) -> None:
    """Record a dataset that was not attempted because an earlier dataset failed."""
    conn.execute(
        "INSERT INTO pipeline_runs (run_id, dataset, source_file, status, started_at, ended_at, failure_reason) "
        "VALUES (?, ?, ?, 'SKIPPED', ?, ?, ?)",
        (run_id, dataset, source_file, now, now, reason),
    )


Writing etl/metrics.py


In [15]:
%%writefile etl/parsing.py
"""Small parsing helpers shared by validate.py and transform.py.

All values arrive as text. These helpers decide whether a text value is a valid
integer / decimal / date / timestamp and convert it in a deterministic way.
"""
from __future__ import annotations

from datetime import datetime, timezone
from decimal import Decimal

import pandas as pd

INTEGER_RE = r"[+-]?\d+"
DECIMAL_RE = r"[+-]?\d+(\.\d+)?"
DATE_RE = r"\d{4}-\d{2}-\d{2}"
TIMESTAMP_RE = r"\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}(:\d{2}(\.\d+)?)?(Z|[+-]\d{2}:?\d{2})?"

ISO_TIMESTAMP_FORMAT = "%Y-%m-%dT%H:%M:%SZ"


def matches(series: pd.Series, pattern: str) -> pd.Series:
    """True where the whole text matches the regex; missing values give False."""
    return series.str.fullmatch(pattern).fillna(False).astype(bool)


def parse_date(series: pd.Series) -> pd.Series:
    """Parse 'YYYY-MM-DD' text; anything invalid (including 2024-02-30) becomes NaT."""
    ok = matches(series, DATE_RE)
    return pd.to_datetime(series.where(ok), format="%Y-%m-%d", errors="coerce")


def parse_timestamp(series: pd.Series) -> pd.Series:
    """Parse ISO-8601 timestamps into UTC. A timestamp without an offset is treated as UTC."""
    ok = matches(series, TIMESTAMP_RE)
    return pd.to_datetime(series.where(ok), format="ISO8601", utc=True, errors="coerce")


def decimal_to_text(value: Decimal) -> str:
    """Canonical text for a decimal: '12.50' -> '12.5', '100' -> '100', '-0.00' -> '0'."""
    return format((value + 0).normalize(), "f")


def utc_iso(moment: datetime) -> str:
    """Format a datetime as ISO-8601 UTC with second precision, e.g. 2026-11-14T10:00:01Z."""
    return moment.astimezone(timezone.utc).strftime(ISO_TIMESTAMP_FORMAT)


Writing etl/parsing.py


In [16]:
%%writefile etl/pipeline.py
"""Orchestrator: run each configured dataset through
extract -> validate -> transform -> duplicate/reference checks -> load,
recording logs and metrics. It contains no dataset-specific logic.
"""
from __future__ import annotations

import time
import uuid
from contextlib import contextmanager
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterator

import pandas as pd

from etl import extract, load, metrics, transform, validate
from etl.config_loader import load_dataset_config, load_pipeline_config, resolve_path
from etl.errors import ConfigError
from etl.logging_setup import get_run_logger, setup_logging, shutdown_logging
from etl.parsing import utc_iso


@dataclass
class DatasetResult:
    """Outcome of one dataset in one run (mirrors a pipeline_runs row)."""

    run_id: str
    dataset: str
    status: str
    rows_in: int = 0
    rows_inserted: int = 0
    rows_already_loaded: int = 0
    rows_rejected: int = 0
    duplicates_dropped: int = 0
    late_records: int = 0
    duration_seconds: float = 0.0
    failure_reason: str | None = None
    warnings: list[str] = field(default_factory=list)


@contextmanager
def _timed(timings: dict[str, float], name: str) -> Iterator[None]:
    """Add the elapsed seconds of a block to timings[name]."""
    start = time.perf_counter()
    try:
        yield
    finally:
        timings[name] = timings.get(name, 0.0) + time.perf_counter() - start


def _failure_reason(exc: Exception) -> str:
    """SchemaError messages already start with a code such as SCHEMA_MISSING_COLUMNS."""
    return str(exc) if type(exc).__name__ == "SchemaError" else f"{type(exc).__name__}: {exc}"


def run_pipeline(config_path: str | Path = "config/pipeline.yaml", *, input_dir: str | Path | None = None,
                 db_path: str | Path | None = None, log_file: str | Path | None = None,
                 dataset_names: list[str] | None = None, now: datetime | None = None) -> list[DatasetResult]:
    """Run all (or the named) datasets once. One run_id is shared by every dataset in this call."""
    pipeline_cfg = load_pipeline_config(config_path)
    datasets = [load_dataset_config(p) for p in pipeline_cfg["datasets"]]
    if dataset_names:
        known = [d["name"] for d in datasets]
        unknown = [n for n in dataset_names if n not in known]
        if unknown:
            raise ConfigError(f"unknown dataset(s) {unknown}; configured: {known}")
        datasets = [d for d in datasets if d["name"] in dataset_names]

    input_path = resolve_path(input_dir) if input_dir else pipeline_cfg["input_dir"]
    database = resolve_path(db_path) if db_path else pipeline_cfg["database_path"]
    log_path = resolve_path(log_file) if log_file else pipeline_cfg["log_file"]
    run_id = str(uuid.uuid4())
    now = now or datetime.now(timezone.utc)

    setup_logging(log_path)
    log = get_run_logger(run_id)
    log.info("run started", extra={"step": "run", "datasets": [d["name"] for d in datasets],
                                   "input_dir": str(input_path), "database": str(database)})
    conn = load.connect(database)
    results: list[DatasetResult] = []
    try:
        metrics.init_metrics(conn)
        load.ensure_reject_table(conn)
        failed: str | None = None
        for config in datasets:
            if failed is not None:
                reason = f"UPSTREAM_FAILED: dataset '{failed}' failed earlier in this run"
                metrics.record_skipped(conn, run_id, config["name"], config["source_file"], utc_iso(now), reason)
                log.warning("dataset skipped", extra={"dataset": config["name"], "step": "run", "reason": reason})
                results.append(DatasetResult(run_id, config["name"], "SKIPPED", failure_reason=reason))
                continue
            result = _run_dataset(conn, config, input_path, run_id, now, pipeline_cfg["validation_mode"], log)
            results.append(result)
            if result.status == "FAILED":
                failed = config["name"]
        log.info("run finished", extra={"step": "run", "statuses": {r.dataset: r.status for r in results}})
    finally:
        conn.close()
        shutdown_logging()
    return results


def _run_dataset(conn, config: dict[str, Any], input_dir: Path, run_id: str, now: datetime,
                 validation_mode: str, log) -> DatasetResult:
    """Run one dataset; any error is recorded as FAILED (with reason) instead of crashing the run."""
    name = config["name"]
    source_path = input_dir / config["source_file"]
    source_file = source_path.name
    started = time.perf_counter()
    timings: dict[str, float] = {}
    counts: dict[str, int] = {}
    warnings: list[str] = []
    metrics.start_run(conn, run_id, name, source_file, utc_iso(datetime.now(timezone.utc)))
    log.info("dataset started", extra={"dataset": name, "step": "start", "source_file": source_file})
    try:
        with _timed(timings, "extract_seconds"):
            extracted = extract.extract_csv(source_path)
        counts["rows_in"] = extracted.rows_in
        log.info("extract done", extra={"dataset": name, "step": "extract", "rows_in": extracted.rows_in,
                                        "malformed_rows": len(extracted.malformed),
                                        "blank_lines_ignored": extracted.blank_lines})

        with _timed(timings, "validate_seconds"):
            extra_columns = validate.check_schema(extracted.header, config["columns"])
            if extra_columns:
                warnings.append(f"EXTRA_COLUMNS_IGNORED: {extra_columns}")
                log.warning("extra columns ignored", extra={"dataset": name, "step": "validate",
                                                            "extra_columns": extra_columns})
            normalized = validate.normalize(extracted.data, config["columns"])
            reasons = validate.validate_rows(normalized, config["columns"], now, validation_mode)
            valid = normalized.drop(index=reasons.index)
            reject_parts = [extracted.malformed, validate.build_reject_rows(extracted.data, reasons)]
        log.info("row validation done", extra={"dataset": name, "step": "validate", "valid_rows": len(valid),
                                               "rule_failures": len(reasons)})

        with _timed(timings, "transform_seconds"):
            typed = transform.transform(valid, config)

        with _timed(timings, "validate_seconds"):
            typed, conflict_reasons, duplicates_dropped = validate.check_duplicates(typed, config)
            reject_parts.append(validate.build_reject_rows(extracted.data, conflict_reasons))
            references = config.get("references") or []
            parent_keys = {(r["parent_table"], r["parent_column"]):
                           load.fetch_parent_keys(conn, r["parent_table"], r["parent_column"]) for r in references}
            reference_reasons = validate.check_references(typed, references, parent_keys)
            typed = typed.drop(index=reference_reasons.index)
            reject_parts.append(validate.build_reject_rows(extracted.data, reference_reasons))
        counts["duplicates_dropped"] = duplicates_dropped
        rejects = pd.concat([part for part in reject_parts if not part.empty] or [validate.empty_rejects()])
        log.info("duplicate and reference checks done", extra={
            "dataset": name, "step": "validate", "duplicates_dropped": duplicates_dropped,
            "conflicting_duplicates": len(conflict_reasons), "invalid_references": len(reference_reasons)})

        with _timed(timings, "load_seconds"):
            loaded = load.load_dataset(conn, config, typed, rejects.sort_index(), extracted.data, run_id=run_id,
                                       source_file=source_file, rows_in=extracted.rows_in,
                                       duplicates_dropped=duplicates_dropped, now_iso=utc_iso(now))
        counts.update(rows_inserted=loaded.rows_inserted, rows_already_loaded=loaded.rows_already_loaded,
                      rows_rejected=loaded.rows_rejected, late_records=loaded.late_records)
        duration = time.perf_counter() - started
        metrics.finish_success(conn, run_id, name, utc_iso(datetime.now(timezone.utc)), duration,
                               counts, timings, warnings)
        log.info("dataset finished", extra={"dataset": name, "step": "finish", "status": "SUCCESS",
                                            "duration_seconds": round(duration, 3), **counts})
        return DatasetResult(run_id, name, "SUCCESS", duration_seconds=round(duration, 3),
                             warnings=warnings, **counts)
    except Exception as exc:  # noqa: BLE001 - every failure must be recorded, then reported
        duration = time.perf_counter() - started
        reason = _failure_reason(exc)
        metrics.finish_failed(conn, run_id, name, utc_iso(datetime.now(timezone.utc)), duration,
                              reason, counts, timings, warnings)
        log.error("dataset failed", exc_info=True, extra={"dataset": name, "step": "failed", "failure_reason": reason})
        return DatasetResult(run_id, name, "FAILED", duration_seconds=round(duration, 3),
                             failure_reason=reason, warnings=warnings, **counts)


def results_as_dicts(results: list[DatasetResult]) -> list[dict[str, Any]]:
    """Plain dicts, handy for printing and tests."""
    return [asdict(r) for r in results]


Writing etl/pipeline.py


In [17]:
%%writefile etl/reconcile.py
"""Reconciliation: prove the target tables agree with the source file, the run log and (optionally)
the expected answers.

Every check counts VIOLATIONS: 0 means PASS. All checks are generated from the dataset configs,
so a new dataset is reconciled without code changes. Nothing here writes to the database.

Sections of the report:
  1. run reconciliation   rows_in = inserted + already loaded + rejected + duplicates dropped
  2. where the rows went  rejects by reason, so every difference is explained
  3. integrity checks     keys, current = latest history, references, aggregates, lineage, rejects
  4. source-to-target     the source file is re-counted independently of the pipeline
  5. expected answers     (optional) comparison with the generator's separately stored answers
"""
from __future__ import annotations

import csv
import json
import sqlite3
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from etl.parsing import utc_iso

KNOWN_REJECT_CODES = {"MALFORMED_ROW", "MISSING_REQUIRED_FIELD", "INVALID_TYPE", "INVALID_FORMAT",
                      "INVALID_ALLOWED_VALUE", "OUT_OF_RANGE", "FUTURE_TIMESTAMP", "CONFLICTING_DUPLICATE",
                      "INVALID_REFERENCE"}
RUN_COLUMNS = ["run_id", "source_file", "rows_in", "rows_inserted", "rows_already_loaded", "rows_rejected",
               "duplicates_dropped", "late_records"]


class ReportError(Exception):
    """The report cannot be produced (for example the pipeline has never run on this database)."""


@dataclass
class CheckResult:
    """One check. violations == 0 means PASS."""

    section: str          # integrity | source | expected
    dataset: str
    name: str
    description: str
    violations: int
    detail: str = ""

    @property
    def passed(self) -> bool:
        return self.violations == 0


@dataclass
class Report:
    generated_at: str
    database: str
    runs: list[dict[str, Any]]                          # latest successful run per dataset
    breakdowns: dict[str, list[tuple[str, int]]]        # dataset -> [(reject reason, rows)]
    checks: list[CheckResult] = field(default_factory=list)
    compared_with_expected: bool = False

    @property
    def passed(self) -> bool:
        return all(check.passed for check in self.checks)


# ---------- small database helpers ----------

def _scalar(conn: sqlite3.Connection, sql: str, params: tuple = ()) -> Any:
    return conn.execute(sql, params).fetchone()[0]


def _table_exists(conn: sqlite3.Connection, name: str) -> bool:
    return bool(conn.execute("select 1 from sqlite_master where type = 'table' and name = ?", (name,)).fetchone())


def _row_dict(cursor: sqlite3.Cursor) -> dict[str, Any] | None:
    row = cursor.fetchone()
    return dict(zip([d[0] for d in cursor.description], row)) if row else None


def _latest_run(conn: sqlite3.Connection, dataset: str, success_only: bool) -> dict[str, Any] | None:
    status = "and status = 'SUCCESS' " if success_only else ""
    return _row_dict(conn.execute(
        f"select status, {', '.join(RUN_COLUMNS)} from pipeline_runs where dataset = ? {status}"
        "order by started_at desc, rowid desc limit 1", (dataset,)))


def _on(keys: list[str], left: str, right: str) -> str:
    return " and ".join(f"{left}.{k} = {right}.{k}" for k in keys)


# ---------- integrity checks (database only) ----------

def _dataset_checks(conn: sqlite3.Connection, cfg: dict[str, Any]) -> list[CheckResult]:
    name, keys, version = cfg["name"], cfg["natural_key"], cfg["version_column"]
    hist, cur = cfg["target"]["history_table"], cfg["target"]["current_table"]
    key_list = ", ".join(keys)
    results: list[CheckResult] = []

    def add(check: str, description: str, violations: int, detail: str = "") -> None:
        results.append(CheckResult("integrity", name, check, description, int(violations), detail))

    latest_any = _latest_run(conn, name, success_only=False)
    add("latest_run_succeeded", "the most recent run of this dataset finished with status SUCCESS",
        0 if latest_any and latest_any["status"] == "SUCCESS" else 1,
        f"latest status = {latest_any['status'] if latest_any else 'no run recorded'}")
    add("run_counts_add_up", "in every successful run rows_in = inserted + already loaded + rejected + duplicates",
        _scalar(conn, "select count(*) from pipeline_runs where dataset = ? and status = 'SUCCESS' and rows_in != "
                      "rows_inserted + rows_already_loaded + rows_rejected + duplicates_dropped", (name,)))
    if not (_table_exists(conn, hist) and _table_exists(conn, cur)):
        add("target_tables_exist", "the history and current tables exist", 1, "run the pipeline first")
        return results

    add("history_key_unique", "primary key (natural key + version) is unique in the history table",
        _scalar(conn, f"select count(*) from (select 1 from {hist} group by {key_list}, {version} having count(*) > 1)"))
    add("current_key_unique", "natural key is unique in the current table",
        _scalar(conn, f"select count(*) from (select 1 from {cur} group by {key_list} having count(*) > 1)"))
    add("current_has_history", "every current row has a history row",
        _scalar(conn, f"select count(*) from {cur} c where not exists "
                      f"(select 1 from {hist} h where {_on(keys, 'h', 'c')})"))
    add("history_has_current", "every history key has a current row",
        _scalar(conn, f"select count(*) from (select distinct {key_list} from {hist}) h where not exists "
                      f"(select 1 from {cur} c where {_on(keys, 'c', 'h')})"))
    add("current_is_latest_history_row", "each current row is identical to the newest version in history",
        _scalar(conn, f"select count(*) from {cur} c where not exists (select 1 from {hist} h where "
                      f"{_on(keys, 'h', 'c')} and h.{version} = (select max(x.{version}) from {hist} x where "
                      f"{_on(keys, 'x', 'c')}) and h.record_hash = c.record_hash)"))
    inserted = _scalar(conn, "select coalesce(sum(rows_inserted), 0) from pipeline_runs "
                             "where dataset = ? and status = 'SUCCESS'", (name,))
    in_history = _scalar(conn, f"select count(*) from {hist}")
    add("history_equals_inserted", "history rows = sum of rows_inserted over successful runs",
        abs(in_history - inserted), f"history rows = {in_history}, sum(rows_inserted) = {inserted}")
    add("lineage_complete", "every history row points to a successful run of this dataset",
        _scalar(conn, f"select count(*) from {hist} where run_id not in (select run_id from pipeline_runs "
                      f"where dataset = ? and status = 'SUCCESS')", (name,)))

    for ref in cfg.get("references") or []:
        parent, column, parent_column = ref["parent_table"], ref["column"], ref["parent_column"]
        if not _table_exists(conn, parent):
            add(f"reference_{column}", f"{column} exists in {parent}", _scalar(conn, f"select count(*) from {hist}"),
                f"parent table {parent} does not exist")
            continue
        add(f"reference_{column}", f"every {hist}.{column} exists in {parent}.{parent_column} (referential integrity)",
            _scalar(conn, f"select count(*) from {hist} h where not exists "
                          f"(select 1 from {parent} p where p.{parent_column} = h.{column})"))

    for column, spec in (cfg.get("derived") or {}).items():
        if spec["operation"] == "multiply_to_cents":          # key aggregate: current total = total of newest versions
            in_current = _scalar(conn, f"select coalesce(sum({column}), 0) from {cur}")
            newest = _scalar(conn, f"select coalesce(sum(h.{column}), 0) from {hist} h where h.{version} = "
                                   f"(select max(x.{version}) from {hist} x where {_on(keys, 'x', 'h')})")
            add(f"aggregate_{column}", f"sum({column}) in current = sum over the newest history versions",
                abs(in_current - newest), f"current = {in_current}, newest history = {newest}")
    return results


def _reject_checks(conn: sqlite3.Connection, datasets: list[dict[str, Any]]) -> list[CheckResult]:
    results: list[CheckResult] = []
    if not _table_exists(conn, "rejected_records"):
        return [CheckResult("integrity", "all", "rejects_table_exists", "rejected_records exists", 1)]
    basic = "reject_reasons = '' or source_file = '' or row_number < 2"
    try:
        untraceable = _scalar(conn, f"select count(*) from rejected_records where {basic} or not json_valid(raw_record)")
    except sqlite3.OperationalError:                        # this SQLite build has no JSON functions
        untraceable = _scalar(conn, f"select count(*) from rejected_records where {basic} or raw_record = ''")
    results.append(CheckResult("integrity", "all", "rejects_traceable",
                               "every reject has a reason, source file, line number (>= 2) and the original row as JSON",
                               int(untraceable)))
    unknown = 0
    for (reasons,) in conn.execute("select distinct reject_reasons from rejected_records").fetchall():
        unknown += sum(1 for part in reasons.split("; ") if part.split(":")[0] not in KNOWN_REJECT_CODES)
    results.append(CheckResult("integrity", "all", "reject_reason_codes_known",
                               "every reject reason uses a documented reason code", unknown))
    for cfg in datasets:
        run = _latest_run(conn, cfg["name"], success_only=True)
        if run is None:
            continue
        seen = _scalar(conn, "select count(*) from rejected_records where dataset = ? and last_seen_run_id = ?",
                       (cfg["name"], run["run_id"]))
        results.append(CheckResult("integrity", cfg["name"], "rejects_cover_latest_run",
                                   "rejects seen in the latest run = rows_rejected of that run",
                                   abs(seen - run["rows_rejected"]),
                                   f"reject records = {seen}, rows_rejected = {run['rows_rejected']}"))
    return results


# ---------- source-to-target ----------

def _source_checks(conn: sqlite3.Connection, datasets: list[dict[str, Any]], input_dir: Path) -> list[CheckResult]:
    results = []
    for cfg in datasets:
        run = _latest_run(conn, cfg["name"], success_only=True)
        path = input_dir / cfg["source_file"]
        if not path.exists():
            results.append(CheckResult("source", cfg["name"], "source_rows_equal_rows_in",
                                       "source file row count = rows_in of the latest run", 1, f"{path} not found"))
            continue
        with open(path, newline="", encoding="utf-8-sig") as handle:      # counted without any pipeline code
            reader = csv.reader(handle)
            next(reader, None)
            source_rows = sum(1 for row in reader if row)
        rows_in = run["rows_in"] if run else None
        results.append(CheckResult(
            "source", cfg["name"], "source_rows_equal_rows_in",
            "source file row count (counted independently) = rows_in of the latest run",
            abs(source_rows - rows_in) if run else 1, f"source rows = {source_rows}, rows_in = {rows_in}"))
    return results


# ---------- comparison with expected answers ----------

def _compare_with_expected(conn: sqlite3.Connection, datasets: list[dict[str, Any]],
                           expected_dir: Path) -> list[CheckResult]:
    summary = json.loads((expected_dir / "expected_summary.json").read_text(encoding="utf-8"))
    rejects_file = expected_dir / "expected_rejects.csv"
    expected_rejects = list(csv.DictReader(open(rejects_file, encoding="utf-8"))) if rejects_file.exists() else []
    results: list[CheckResult] = []
    for cfg in datasets:
        name, want = cfg["name"], summary.get(cfg["name"])
        if want is None:
            continue
        hist, cur = cfg["target"]["history_table"], cfg["target"]["current_table"]
        run = _latest_run(conn, name, success_only=True)

        def add(check: str, description: str, violations: int, detail: str = "") -> None:
            results.append(CheckResult("expected", name, check, description, int(violations), detail))

        if run is None:
            add("expected_run_counts", "a successful run exists to compare", 1, "no successful run")
            continue
        fields = ["rows_in", "rows_inserted", "rows_rejected", "duplicates_dropped", "late_records"]
        wrong = [f"{f}: got {run[f]}, expected {want[f]}" for f in fields if run[f] != want[f]]
        add("expected_run_counts", "run counts equal expected_summary.json (in, inserted, rejected, duplicates, late)",
            len(wrong), "; ".join(wrong))
        tables = {"history_rows": _scalar(conn, f"select count(*) from {hist}"),
                  "current_rows": _scalar(conn, f"select count(*) from {cur}")}
        wrong = [f"{k}: got {v}, expected {want[k]}" for k, v in tables.items() if v != want[k]]
        add("expected_table_rows", "history and current row counts equal expected_summary.json", len(wrong),
            "; ".join(wrong))

        wanted = {(int(r["row_number"]), r["expected_reject_reasons"]) for r in expected_rejects
                  if r["dataset"] == name}
        found = set(conn.execute("select row_number, reject_reasons from rejected_records where dataset = ? "
                                 "and last_seen_run_id = ?", (name, run["run_id"])))
        add("expected_rejects", "rejected rows and their reasons equal expected_rejects.csv (row number and reason)",
            len(wanted ^ found), f"missing {len(wanted - found)}, unexpected {len(found - wanted)}")

        current_file = expected_dir / f"expected_current_{name}.csv"
        if current_file.exists():
            with open(current_file, encoding="utf-8", newline="") as handle:
                reader = csv.DictReader(handle)
                columns = list(reader.fieldnames or [])
                wanted_rows = {tuple(row[c] for c in columns) for row in reader}
            found_rows = {tuple(str(v) for v in row) for row in conn.execute(f"select {', '.join(columns)} from {cur}")}
            add("expected_current_rows", f"current rows equal {current_file.name} (the latest version of every record)",
                len(wanted_rows ^ found_rows), f"missing {len(wanted_rows - found_rows)}, unexpected "
                                               f"{len(found_rows - wanted_rows)}")
        for key, expected_total in want.items():                   # e.g. current_order_amount_cents_sum
            if key.endswith("_sum") and key.startswith(("current_", "history_")):
                prefix = key.split("_")[0]
                column = key[len(prefix) + 1:-len("_sum")]
                table = cur if prefix == "current" else hist
                total = _scalar(conn, f"select coalesce(sum({column}), 0) from {table}")
                add(f"expected_{key}", f"sum({column}) in {table} equals the expected total",
                    abs(total - expected_total), f"got {total}, expected {expected_total}")
    return results


# ---------- the report ----------

def build_report(conn: sqlite3.Connection, datasets: list[dict[str, Any]], *, input_dir: Path | None = None,
                 expected_dir: Path | None = None, database: str = "") -> Report:
    """Run every check and collect the numbers for the report."""
    if not _table_exists(conn, "pipeline_runs"):
        raise ReportError("pipeline_runs does not exist: run the pipeline on this database first")
    runs, breakdowns = [], {}
    for cfg in datasets:
        run = _latest_run(conn, cfg["name"], success_only=True)
        if run is not None:
            runs.append({"dataset": cfg["name"], **run})
            if _table_exists(conn, "rejected_records"):
                breakdowns[cfg["name"]] = conn.execute(
                    "select reject_reasons, count(*) from rejected_records where dataset = ? and last_seen_run_id = ? "
                    "group by reject_reasons order by count(*) desc, reject_reasons", (cfg["name"], run["run_id"])).fetchall()
    report = Report(utc_iso(datetime.now(timezone.utc)), database, runs, breakdowns)
    for cfg in datasets:
        report.checks += _dataset_checks(conn, cfg)
    report.checks += _reject_checks(conn, datasets)
    if input_dir is not None:
        report.checks += _source_checks(conn, datasets, input_dir)
    if expected_dir is not None:
        report.checks += _compare_with_expected(conn, datasets, expected_dir)
        report.compared_with_expected = True
    return report


def _table(headers: list[str], rows: list[list[Any]]) -> list[str]:
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join("---" for _ in headers) + "|"]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    return lines


def render_markdown(report: Report) -> str:
    """The report as Markdown (printed to the console and saved to a file)."""
    passed = sum(1 for c in report.checks if c.passed)
    lines = ["# Reconciliation report", "",
             f"Generated: {report.generated_at}  ", f"Database: `{report.database}`  ",
             f"Result: **{'PASS' if report.passed else 'FAIL'}** ({passed} of {len(report.checks)} checks passed)", "",
             "## 1. Run reconciliation (latest successful run of each dataset)", "",
             "`rows_in = rows_inserted + rows_already_loaded + rows_rejected + duplicates_dropped`", ""]
    rows = []
    for r in report.runs:
        total = r["rows_inserted"] + r["rows_already_loaded"] + r["rows_rejected"] + r["duplicates_dropped"]
        rows.append([r["dataset"], r["run_id"][:8], r["rows_in"], r["rows_inserted"], r["rows_already_loaded"],
                     r["rows_rejected"], r["duplicates_dropped"], r["late_records"],
                     f"{total} = {r['rows_in']}: {'PASS' if total == r['rows_in'] else 'FAIL'}"])
    lines += _table(["dataset", "run", "rows_in", "inserted", "already_loaded", "rejected", "duplicates_dropped",
                     "late (kept in history)", "sum of parts = rows_in"], rows)
    lines += ["", "## 2. Where the rows went: rejects by reason (latest run)", ""]
    reason_rows = [[ds, reason, n] for ds, items in report.breakdowns.items() for reason, n in items]
    lines += _table(["dataset", "reject reason", "rows"], reason_rows) if reason_rows else ["No rejected rows."]
    for section, title in (("integrity", "3. Integrity checks"), ("source", "4. Source-to-target"),
                           ("expected", "5. Comparison with the expected answers")):
        checks = [c for c in report.checks if c.section == section]
        if not checks:
            continue
        lines += ["", f"## {title}", ""]
        lines += _table(["dataset", "check", "violations", "result", "what it proves"],
                        [[c.dataset, c.name, c.violations, "PASS" if c.passed else "FAIL",
                          c.description + (f" ({c.detail})" if c.detail and not c.passed else "")] for c in checks])
    if not report.compared_with_expected:
        lines += ["", "_No expected answers were given, so section 5 was skipped._"]
    return "\n".join(lines) + "\n"


Writing etl/reconcile.py


In [18]:
%%writefile etl/transform.py
"""Transform: cast types, clean values, compute derived fields and record_hash.

Transformations are deterministic: the same input always gives the same output.
`record_hash` is a SHA-256 fingerprint of the cleaned business columns; it lets us
tell "exactly the same record again" from "same key but different content".
"""
from __future__ import annotations

import hashlib
from decimal import ROUND_HALF_UP, Decimal
from typing import Any

import pandas as pd

from etl.parsing import decimal_to_text, parse_date, parse_timestamp, ISO_TIMESTAMP_FORMAT

# SQLite type of every derived-field operation (also tells config_loader which operations exist).
DERIVED_SQL_TYPES = {"split_part": "TEXT", "multiply_to_cents": "INTEGER", "date_part": "TEXT"}

_FIELD_SEPARATOR = "\x1f"  # a control character that never appears in normal text


def _plain(series: pd.Series) -> list:
    """The values of a column as a plain Python list.

    Looping over a pandas string column directly costs one slow method call per value (about 2.5 million of
    them in a 240,000-row run). Converting the column once, then looping over the list, gives the same values
    faster. run_benchmark.py swaps this function for an identity function to reproduce the "before" timing.
    """
    return series.to_numpy(dtype=object).tolist()


def _canonicalize(s: pd.Series, spec: dict[str, Any]) -> tuple[pd.Series, pd.Series]:
    """Return (value to store, canonical text used for hashing and derived fields)."""
    kind = spec["type"]
    if kind == "string":
        transform = spec.get("transform")
        out = s.str.lower() if transform == "lowercase" else s.str.upper() if transform == "uppercase" else s
        return out, out
    if kind == "integer":
        ints = pd.to_numeric(s).astype("Int64")
        return ints, ints.astype("string")
    if kind == "decimal":
        decimals = _plain(s.map(Decimal, na_action="ignore"))
        stored = pd.Series([float(d) if isinstance(d, Decimal) else None for d in decimals],
                           index=s.index, dtype="Float64")
        text = pd.Series([decimal_to_text(d) if isinstance(d, Decimal) else pd.NA for d in decimals],
                         index=s.index, dtype="string")
        return stored, text
    if kind == "date":
        text = parse_date(s).dt.strftime("%Y-%m-%d").astype("string")
        return text, text
    text = parse_timestamp(s).dt.strftime(ISO_TIMESTAMP_FORMAT).astype("string")  # timestamp, UTC
    return text, text


def _derive(spec: dict[str, Any], canon: pd.DataFrame) -> pd.Series:
    """Compute one derived column from canonical source columns."""
    operation = spec["operation"]
    source = canon[spec["sources"][0]]
    if operation == "split_part":
        return source.str.split(spec["delimiter"]).str[spec["index"]].astype("string")
    if operation == "date_part":
        return source.str[:10].astype("string")
    # multiply_to_cents: exact Decimal math, rounded half-up to a whole number of cents
    parts = [_plain(canon[name].map(Decimal, na_action="ignore")) for name in spec["sources"]]
    cents: list[Any] = []
    for values in zip(*parts):
        if all(isinstance(v, Decimal) for v in values):
            product = Decimal(1)
            for v in values:
                product *= v
            cents.append(int((product * 100).quantize(Decimal(1), rounding=ROUND_HALF_UP)))
        else:
            cents.append(None)
    return pd.Series(cents, index=canon.index, dtype="Int64")


def _hash_rows(canon: pd.DataFrame) -> pd.Series:
    """SHA-256 of each row's canonical business values, in config column order."""
    columns = [_plain(canon[name].fillna("")) for name in canon.columns]
    hashes = [hashlib.sha256(_FIELD_SEPARATOR.join(values).encode("utf-8")).hexdigest()
              for values in zip(*columns)]
    return pd.Series(hashes, index=canon.index, dtype="string")


def transform(valid: pd.DataFrame, config: dict[str, Any]) -> pd.DataFrame:
    """Turn validated text rows into typed target rows (+ derived columns + record_hash)."""
    stored: dict[str, pd.Series] = {}
    canon: dict[str, pd.Series] = {}
    for col, spec in config["columns"].items():
        stored[col], canon[col] = _canonicalize(valid[col], spec)
    out = pd.DataFrame(stored, index=valid.index)
    canon_df = pd.DataFrame(canon, index=valid.index)
    for name, spec in (config.get("derived") or {}).items():
        out[name] = _derive(spec, canon_df)
    out["record_hash"] = _hash_rows(canon_df)
    return out


Writing etl/transform.py


In [19]:
%%writefile etl/validate.py
"""Validate: schema check, row-level rules, duplicate keys and reference keys.

Every function here is driven by the dataset config; none knows what a
"customer" or an "order" is. Failing rows are never dropped silently: each one
gets a reject reason string such as 'MISSING_REQUIRED_FIELD:email; OUT_OF_RANGE:quantity'.
"""
from __future__ import annotations

import json
from datetime import datetime
from decimal import Decimal
from typing import Any

import numpy as np
import pandas as pd

from etl.errors import SchemaError
from etl.parsing import (
    DECIMAL_RE,
    INTEGER_RE,
    matches,
    parse_date,
    parse_timestamp,
)


# ---------- schema ----------

def check_schema(header: list[str], columns_cfg: dict[str, Any]) -> list[str]:
    """Fail on duplicate or missing required columns; return the extra columns (to warn about)."""
    duplicates = sorted({name for name in header if header.count(name) > 1})
    if duplicates:
        raise SchemaError(f"SCHEMA_DUPLICATE_COLUMNS: {duplicates}")
    missing = [c for c, spec in columns_cfg.items() if spec.get("required", False) and c not in header]
    if missing:
        raise SchemaError(f"SCHEMA_MISSING_COLUMNS: {missing}")
    return [name for name in header if name not in columns_cfg]


def normalize(raw: pd.DataFrame, columns_cfg: dict[str, Any]) -> pd.DataFrame:
    """Keep only configured columns, trim whitespace, and turn empty text into a real missing value."""
    out = pd.DataFrame(index=raw.index)
    for col in columns_cfg:
        if col in raw.columns:
            text = raw[col].astype("string").str.strip()
            out[col] = text.mask((text == "").fillna(False), pd.NA)
        else:  # optional column absent from the file
            out[col] = pd.Series(pd.NA, index=raw.index, dtype="string")
    return out


# ---------- reject helpers ----------

def empty_rejects() -> pd.DataFrame:
    """An empty reject frame with the standard shape (index = source row number)."""
    return pd.DataFrame(
        {"reject_reasons": pd.Series(dtype="object"), "raw_record": pd.Series(dtype="object")},
        index=pd.Index([], name="row_number", dtype="int64"),
    )


def build_reject_rows(raw: pd.DataFrame, reasons: pd.Series) -> pd.DataFrame:
    """Turn {row_number: reason text} into reject rows that also keep the original record as JSON."""
    if reasons.empty:
        return empty_rejects()
    originals = raw.loc[reasons.index].astype(object).to_dict("records")
    return pd.DataFrame(
        {
            "reject_reasons": reasons.to_numpy(),
            "raw_record": [json.dumps(rec, ensure_ascii=False) for rec in originals],
        },
        index=pd.Index(reasons.index, name="row_number"),
    )


def reasons_from_flags(flags: dict[str, pd.Series], index: pd.Index) -> pd.Series:
    """Combine named True/False checks into one reason string per failing row."""
    empty = pd.Series(dtype="object", index=pd.Index([], name="row_number", dtype="int64"))
    if not flags or len(index) == 0:
        return empty
    flag_df = pd.DataFrame(flags, index=index).astype(bool)
    failing = flag_df.any(axis=1)
    if not failing.any():
        return empty
    names = np.array(flag_df.columns)
    texts = ["; ".join(names[row]) for row in flag_df[failing].to_numpy()]
    return pd.Series(texts, index=pd.Index(flag_df.index[failing], name="row_number"), dtype="object")


# ---------- row rules ----------

def _column_flags(col: str, spec: dict[str, Any], s: pd.Series, now: datetime) -> dict[str, pd.Series]:
    """Return {'REASON_CODE:column': True-where-the-rule-fails} for one column."""
    flags: dict[str, pd.Series] = {}
    present = s.notna()
    kind = spec["type"]

    if spec.get("required", False):
        flags[f"MISSING_REQUIRED_FIELD:{col}"] = ~present

    parsed = None
    if kind == "integer":
        type_ok = matches(s, INTEGER_RE)
    elif kind == "decimal":
        type_ok = matches(s, DECIMAL_RE)
    elif kind == "date":
        parsed = parse_date(s)
        type_ok = parsed.notna()
    elif kind == "timestamp":
        parsed = parse_timestamp(s)
        type_ok = parsed.notna()
    else:
        type_ok = present
    if kind != "string":
        flags[f"INVALID_TYPE:{col}"] = present & ~type_ok
    typed_ok = present & type_ok

    if "pattern" in spec:
        flags[f"INVALID_FORMAT:{col}"] = present & ~matches(s, spec["pattern"])
    if "allowed_values" in spec:
        flags[f"INVALID_ALLOWED_VALUE:{col}"] = present & ~s.isin([str(v) for v in spec["allowed_values"]])

    if "min" in spec or "max" in spec:
        out_of_range = pd.Series(False, index=s.index)
        values = s[typed_ok].map(Decimal)  # Decimal compares exactly (no float rounding surprises)
        if len(values):
            bad = pd.Series(False, index=values.index)
            if "min" in spec:
                bad |= values < Decimal(str(spec["min"]))
            if "max" in spec:
                bad |= values > Decimal(str(spec["max"]))
            out_of_range.loc[bad.index[bad]] = True
        flags[f"OUT_OF_RANGE:{col}"] = out_of_range

    if spec.get("not_in_future"):
        limit = pd.Timestamp(now) if kind == "timestamp" else pd.Timestamp(now.date())
        flags[f"FUTURE_TIMESTAMP:{col}"] = (parsed > limit).fillna(False).astype(bool)

    return flags


def validate_rows(norm: pd.DataFrame, columns_cfg: dict[str, Any], now: datetime,
                  mode: str = "vectorized") -> pd.Series:
    """Check every row against the column rules. Returns reason text for the rows that fail."""
    if mode != "vectorized":
        raise NotImplementedError("validation_mode 'rowwise' is added in Phase 3 step 7 (performance comparison)")
    flags: dict[str, pd.Series] = {}
    for col, spec in columns_cfg.items():
        flags.update(_column_flags(col, spec, norm[col], now))
    return reasons_from_flags(flags, norm.index)


# ---------- duplicates and references (run on transformed rows) ----------

def check_duplicates(df: pd.DataFrame, config: dict[str, Any]) -> tuple[pd.DataFrame, pd.Series, int]:
    """Handle repeated (natural key + version) rows inside one file.

    Same key+version and identical content: keep the first copy, count the rest as duplicates.
    Same key+version but different content: we cannot know which is right, so reject every copy.
    Returns (rows to keep, reject reasons for conflicting rows, number of exact duplicates dropped).
    """
    keys = [*config["natural_key"], config["version_column"]]
    if df.empty:
        return df, pd.Series(dtype="object"), 0
    group = df.groupby(keys, sort=False)["record_hash"]
    conflict = (group.transform("size") > 1) & (group.transform("nunique") > 1)
    exact_duplicate = ~conflict & df.duplicated(keys, keep="first")
    reasons = pd.Series(
        f"CONFLICTING_DUPLICATE:{','.join(keys)}",
        index=pd.Index(df.index[conflict], name="row_number"),
        dtype="object",
    )
    return df[~conflict & ~exact_duplicate], reasons, int(exact_duplicate.sum())


def check_references(df: pd.DataFrame, references: list[dict[str, str]],
                     parent_keys: dict[tuple[str, str], set]) -> pd.Series:
    """Reject rows whose reference column has no matching parent key (INVALID_REFERENCE)."""
    flags = {}
    for ref in references:
        known = parent_keys[(ref["parent_table"], ref["parent_column"])]
        flags[f"INVALID_REFERENCE:{ref['column']}"] = ~df[ref["column"]].isin(known)
    return reasons_from_flags(flags, df.index)


Writing etl/validate.py


In [20]:
%%writefile generator/__init__.py
"""Synthetic data generator package."""


Writing generator/__init__.py


In [21]:
%%writefile generator/generate_data.py
"""Generate synthetic customers and orders CSV files with a fixed random seed.

Two kinds of files are written, in two different folders so the answer key never
sits next to the input:
  * INPUT files     (--out-dir)      customers.csv, orders.csv
  * EXPECTED files  (--expected-dir) which rows must be rejected and why, expected
                                     counts and totals, and the expected "current" rows

The generator knows what it planted, so the expected answers are computed here,
independently of the pipeline. Same seed + same options = identical files.

Examples:
    python -m generator.generate_data                       # small sample -> data/sample, data/expected
    python -m generator.generate_data --customers 20000 --orders 100000 \\
        --out-dir data/large --expected-dir data/large/expected
"""
from __future__ import annotations

import argparse
import csv
import json
import random
from collections import Counter
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path(__file__).resolve().parents[1]

DEFAULT_SEED = 42
# All valid timestamps stay before 2025-08-01, so a run on any later date never sees them as 'future'.
LATE_RATE = 0.2             # share of multi-version records whose older versions arrive later in the file
DUPLICATE_RATE = 0.02       # exact duplicate rows, as a share of records (at least 2)
CUSTOMER_UPDATE_RATE = 0.15  # share of customers that get a second (changed) version
NOISE_RATE = 0.05           # share of valid rows with harmless noise (padded name, upper-case email)

COUNTRIES = ["IN", "US", "GB", "AE", "SG"]
CURRENCIES = ["INR", "USD"]
FIRST_NAMES = ["Asha", "John", "Meera", "Omar", "Li", "Priya", "Ravi", "Sara", "Tom", "Uma", "Vik", "Nina",
               "Carlos", "Fatima", "Hiro", "Ivan", "Julia", "Kofi", "Lena", "Mateo"]
LAST_NAMES = ["Rao", "Smith", "Nair", "Khan", "Wei", "Das", "Patel", "Lee", "Brown", "Iyer", "Shah", "Gupta",
              "Silva", "Ali", "Sato", "Petrov", "Meyer", "Mensah", "Berg", "Lopez"]
EMAIL_DOMAINS = ["example.com", "mail.example.org", "corp.example.net"]
ORDER_STATUS_PATHS = [["placed"], ["placed", "shipped"], ["placed", "shipped", "delivered"], ["placed", "cancelled"]]
ORDER_STATUS_WEIGHTS = [2, 2, 5, 1]
# (quantity, unit price in cents): exactly on the limits of the order rules, so they must be ACCEPTED
BOUNDARY_ORDERS = [(1, 0), (1000, 10_000_000), (1, 1), (1000, 0)]

CUSTOMER_HEADER = ["customer_id", "full_name", "email", "country_code", "signup_date", "updated_at"]
ORDER_HEADER = ["order_id", "customer_id", "order_ts", "status", "quantity", "unit_price", "currency", "updated_at"]

CUSTOMER_DEFECTS = ["missing_email", "missing_full_name", "invalid_email_format", "invalid_customer_id_format",
                    "invalid_country_code", "future_signup_date", "invalid_signup_date", "invalid_updated_at",
                    "multiple_defects", "malformed_row", "conflicting_duplicate"]
ORDER_DEFECTS = ["quantity_zero", "quantity_above_max", "negative_price", "price_above_max",
                 "quantity_not_a_number", "quantity_fractional", "invalid_status", "invalid_currency",
                 "missing_order_ts", "missing_customer_id", "future_order_ts", "invalid_order_id_format",
                 "invalid_reference", "multiple_defects", "malformed_row", "conflicting_duplicate"]


@dataclass
class Row:
    """One line of an input file plus what the generator knows about it."""

    fields: list[str]
    kind: str                      # valid | bad | duplicate_exact | conflict
    reasons: str = ""              # expected reject_reasons text (for bad and conflict rows)
    defect: str = ""               # human-readable name of the planted defect
    key: tuple = ()                # natural key (valid rows)
    version: datetime | None = None
    canon: dict[str, Any] = field(default_factory=dict)   # expected cleaned values (valid rows)
    cents: int | None = None       # order amount in cents (valid order rows)
    after: int | None = None       # a duplicate must be placed after base row number `after`


# ---------- small helpers ----------

def iso_z(moment: datetime) -> str:
    return moment.strftime("%Y-%m-%dT%H:%M:%SZ")


def format_timestamp(moment: datetime, rng: random.Random) -> str:
    """Mostly 'Z' timestamps, sometimes '+00:00' or '+05:30' (same instant, different text)."""
    roll = rng.random()
    if roll < 0.90:
        return iso_z(moment)
    if roll < 0.95:
        return moment.strftime("%Y-%m-%dT%H:%M:%S") + "+00:00"
    return (moment + timedelta(hours=5, minutes=30)).strftime("%Y-%m-%dT%H:%M:%S") + "+05:30"


def price_text(cents: int) -> str:
    return f"{cents // 100}.{cents % 100:02d}"


def random_moment(rng: random.Random, start: datetime, days: int) -> datetime:
    return start + timedelta(days=rng.randint(0, days), seconds=rng.randint(0, 86399))


def merge_rows(base: list[Row], extras: list[Row], rng: random.Random) -> list[Row]:
    """Place each extra row at a random position; a duplicate always lands after its original."""
    slots = []
    for seq, row in enumerate(extras):
        low = row.after + 1 if row.after is not None else 0
        slots.append((rng.randint(low, len(base)), seq, row))
    slots.sort(key=lambda slot: (slot[0], slot[1]))
    merged: list[Row] = []
    j = 0
    for index, row in enumerate(base):
        while j < len(slots) and slots[j][0] <= index:
            merged.append(slots[j][2])
            j += 1
        merged.append(row)
    merged.extend(slot[2] for slot in slots[j:])
    return merged


def duplicate_rows(base: list[Row], rng: random.Random) -> list[Row]:
    """Exact copies of some valid rows (same text), to be dropped as duplicates."""
    count = min(len(base), max(2, round(DUPLICATE_RATE * len(base))))
    return [Row(fields=list(base[i].fields), kind="duplicate_exact", defect="exact_duplicate", after=i)
            for i in sorted(rng.sample(range(len(base)), count))]


# ---------- customers ----------

def _person(rng: random.Random, number: int) -> tuple[str, str, str]:
    first, last = rng.choice(FIRST_NAMES), rng.choice(LAST_NAMES)
    return first, last, f"{first}.{last}{number}@{rng.choice(EMAIL_DOMAINS)}".lower()


def build_customers(count: int, bad_rate: float, rng: random.Random) -> tuple[list[Row], list[str]]:
    """Return (rows in file order, ids of customers that will NOT be loaded)."""
    keyed: list[tuple[int, int, Row]] = []
    epoch = datetime(2022, 1, 1, tzinfo=timezone.utc)
    for i in range(1, count + 1):
        cid = f"C{i:06d}"
        first, last, email = _person(rng, i)
        signup = random_moment(rng, epoch, 1094)
        versions = [(f"{first} {last}", rng.choice(COUNTRIES), signup)]
        if rng.random() < CUSTOMER_UPDATE_RATE:
            later = signup + timedelta(days=rng.randint(30, 200), seconds=rng.randint(1, 86399))
            versions.append((f"{first} {rng.choice(LAST_NAMES)}", rng.choice(COUNTRIES), later))
        late = len(versions) > 1 and rng.random() < LATE_RATE
        for v, (name, country, moment) in enumerate(versions):
            shown_name = f"  {name} " if rng.random() < NOISE_RATE else name
            shown_email = email.upper() if rng.random() < NOISE_RATE else email
            fields = [cid, shown_name, shown_email, country, signup.date().isoformat(), format_timestamp(moment, rng)]
            row = Row(fields=fields, kind="valid", key=(cid,), version=moment,
                      canon={"customer_id": cid, "full_name": name, "email": email, "country_code": country,
                             "updated_at": iso_z(moment)})
            shift = rng.randint(1, 20) if late and v < len(versions) - 1 else 0
            keyed.append((i + shift, len(keyed), row))
    base = [row for _, _, row in sorted(keyed, key=lambda item: (item[0], item[1]))]

    extras, rejected_ids = _customer_defect_rows(count, bad_rate, rng)
    extras += duplicate_rows(base, rng)
    return merge_rows(base, extras, rng), rejected_ids


def _customer_defect_rows(count: int, bad_rate: float, rng: random.Random) -> tuple[list[Row], list[str]]:
    items = max(len(CUSTOMER_DEFECTS), round(bad_rate * count))
    extras: list[Row] = []
    rejected_ids: list[str] = []
    for n in range(items):
        defect = CUSTOMER_DEFECTS[n % len(CUSTOMER_DEFECTS)]
        cid = f"C{count + 1 + n:06d}"
        first, last, email = _person(rng, count + 1 + n)
        moment = random_moment(rng, datetime(2022, 1, 1, tzinfo=timezone.utc), 1094)
        f = [cid, f"{first} {last}", email, rng.choice(COUNTRIES), moment.date().isoformat(), iso_z(moment)]
        reasons = {
            "missing_email": "MISSING_REQUIRED_FIELD:email",
            "missing_full_name": "MISSING_REQUIRED_FIELD:full_name",
            "invalid_email_format": "INVALID_FORMAT:email",
            "invalid_customer_id_format": "INVALID_FORMAT:customer_id",
            "invalid_country_code": "INVALID_ALLOWED_VALUE:country_code",
            "future_signup_date": "FUTURE_TIMESTAMP:signup_date",
            "invalid_signup_date": "INVALID_TYPE:signup_date",
            "invalid_updated_at": "INVALID_TYPE:updated_at",
            "multiple_defects": "MISSING_REQUIRED_FIELD:email; INVALID_ALLOWED_VALUE:country_code",
            "malformed_row": "MALFORMED_ROW",
            "conflicting_duplicate": "CONFLICTING_DUPLICATE:customer_id,updated_at",
        }[defect]
        if defect == "missing_email":
            f[2] = ""
        elif defect == "missing_full_name":
            f[1] = ""
        elif defect == "invalid_email_format":
            f[2] = "no-at-sign.example.com"
        elif defect == "invalid_customer_id_format":
            f[0] = f"CUST-{n}"
        elif defect == "invalid_country_code":
            f[3] = rng.choice(["XX", "ZZ", "in"])          # 'in' = wrong case, which must also be rejected
        elif defect == "future_signup_date":
            f[4] = "2999-01-01"
        elif defect == "invalid_signup_date":
            f[4] = rng.choice(["2024-02-30", "2024-13-01", "31/12/2024"])
        elif defect == "invalid_updated_at":
            f[5] = rng.choice(["not-a-timestamp", "2024-99-99T00:00:00Z", "yesterday"])
        elif defect == "multiple_defects":
            f[2], f[3] = "", "XX"
        elif defect == "malformed_row":
            f = f[:-1] if rng.random() < 0.5 else f + ["unexpected_extra_field"]
        if defect == "conflicting_duplicate":
            second = list(f)
            second[1] = f"{first} {last} (changed)"          # same key and version, different content
            extras += [Row(fields=f, kind="conflict", reasons=reasons, defect=defect),
                       Row(fields=second, kind="conflict", reasons=reasons, defect=defect)]
        else:
            extras.append(Row(fields=f, kind="bad", reasons=reasons, defect=defect))
        if defect != "invalid_customer_id_format":
            rejected_ids.append(cid)
    return extras, rejected_ids


# ---------- orders ----------

def build_orders(count: int, customers: int, bad_rate: float, orphan_ids: list[str],
                 rng: random.Random) -> list[Row]:
    keyed: list[tuple[int, int, Row]] = []
    epoch = datetime(2024, 1, 1, tzinfo=timezone.utc)
    for i in range(1, count + 1):
        oid, cust = f"O{i:08d}", f"C{rng.randint(1, customers):06d}"
        if i <= len(BOUNDARY_ORDERS):
            quantity, price_cents = BOUNDARY_ORDERS[i - 1]
        elif rng.random() < 0.01:
            quantity, price_cents = rng.choice(BOUNDARY_ORDERS)
        else:
            quantity, price_cents = rng.randint(1, 20), rng.randint(100, 200_000)
        currency = rng.choice(CURRENCIES)
        order_ts = random_moment(rng, epoch, 546)
        statuses = rng.choices(ORDER_STATUS_PATHS, ORDER_STATUS_WEIGHTS)[0]
        late = len(statuses) > 1 and rng.random() < LATE_RATE
        moment = order_ts
        for v, status in enumerate(statuses):
            if v:
                moment = moment + timedelta(hours=rng.randint(2, 72), seconds=rng.randint(0, 3599))
            fields = [oid, cust, format_timestamp(order_ts, rng), status, str(quantity), price_text(price_cents),
                      currency, format_timestamp(moment, rng)]
            row = Row(fields=fields, kind="valid", key=(oid,), version=moment, cents=quantity * price_cents,
                      canon={"order_id": oid, "customer_id": cust, "status": status,
                             "order_amount_cents": quantity * price_cents, "updated_at": iso_z(moment)})
            shift = rng.randint(1, 50) if late and v < len(statuses) - 1 else 0
            keyed.append((i + shift, len(keyed), row))
    base = [row for _, _, row in sorted(keyed, key=lambda item: (item[0], item[1]))]

    extras = _order_defect_rows(count, customers, bad_rate, orphan_ids, rng)
    extras += duplicate_rows(base, rng)
    return merge_rows(base, extras, rng)


def _order_defect_rows(count: int, customers: int, bad_rate: float, orphan_ids: list[str],
                       rng: random.Random) -> list[Row]:
    items = max(len(ORDER_DEFECTS), round(bad_rate * count))
    extras: list[Row] = []
    for n in range(items):
        defect = ORDER_DEFECTS[n % len(ORDER_DEFECTS)]
        oid = f"O{count + 1 + n:08d}"
        moment = random_moment(rng, datetime(2024, 1, 1, tzinfo=timezone.utc), 546)
        f = [oid, f"C{rng.randint(1, customers):06d}", iso_z(moment), "placed", str(rng.randint(1, 20)),
             price_text(rng.randint(100, 200_000)), rng.choice(CURRENCIES), iso_z(moment)]
        reasons = {
            "quantity_zero": "OUT_OF_RANGE:quantity",
            "quantity_above_max": "OUT_OF_RANGE:quantity",
            "negative_price": "OUT_OF_RANGE:unit_price",
            "price_above_max": "OUT_OF_RANGE:unit_price",
            "quantity_not_a_number": "INVALID_TYPE:quantity",
            "quantity_fractional": "INVALID_TYPE:quantity",
            "invalid_status": "INVALID_ALLOWED_VALUE:status",
            "invalid_currency": "INVALID_ALLOWED_VALUE:currency",
            "missing_order_ts": "MISSING_REQUIRED_FIELD:order_ts",
            "missing_customer_id": "MISSING_REQUIRED_FIELD:customer_id",
            "future_order_ts": "FUTURE_TIMESTAMP:order_ts",
            "invalid_order_id_format": "INVALID_FORMAT:order_id",
            "invalid_reference": "INVALID_REFERENCE:customer_id",
            "multiple_defects": "INVALID_ALLOWED_VALUE:status; OUT_OF_RANGE:unit_price",
            "malformed_row": "MALFORMED_ROW",
            "conflicting_duplicate": "CONFLICTING_DUPLICATE:order_id,updated_at",
        }[defect]
        if defect == "quantity_zero":
            f[4] = "0"
        elif defect == "quantity_above_max":
            f[4] = "1001"
        elif defect == "negative_price":
            f[5] = "-0.01"
        elif defect == "price_above_max":
            f[5] = "100000.01"
        elif defect == "quantity_not_a_number":
            f[4] = "abc"
        elif defect == "quantity_fractional":
            f[4] = "2.5"
        elif defect == "invalid_status":
            f[3] = "lost"
        elif defect == "invalid_currency":
            f[6] = "EUR"
        elif defect == "missing_order_ts":
            f[2] = ""
        elif defect == "missing_customer_id":
            f[1] = ""
        elif defect == "future_order_ts":
            f[2] = "2999-01-01T00:00:00Z"
        elif defect == "invalid_order_id_format":
            f[0] = f"ORD-{n}"
        elif defect == "invalid_reference":
            f[1] = rng.choice(orphan_ids) if orphan_ids else f"C{customers + 500_000:06d}"
        elif defect == "multiple_defects":
            f[3], f[5] = "lost", "-5.00"
        elif defect == "malformed_row":
            f = f[:-1] if rng.random() < 0.5 else f + ["unexpected_extra_field"]
        if defect == "conflicting_duplicate":
            second = list(f)
            second[5] = price_text(int(round(float(f[5]) * 100)) + 100)      # same key and version, other price
            extras += [Row(fields=f, kind="conflict", reasons=reasons, defect=defect),
                       Row(fields=second, kind="conflict", reasons=reasons, defect=defect)]
        else:
            extras.append(Row(fields=f, kind="bad", reasons=reasons, defect=defect))
    return extras


# ---------- expected answers ----------

def summarize(rows: list[Row]) -> tuple[dict[str, Any], dict[tuple, Row]]:
    """Expected counts for one dataset, worked out from what was planted (not from the pipeline)."""
    valid = [r for r in rows if r.kind == "valid"]
    rejected = [r for r in rows if r.kind in ("bad", "conflict")]
    dropped = [r for r in rows if r.kind == "duplicate_exact"]
    assert len({(r.key, r.version) for r in valid}) == len(valid), "generator made a duplicate key+version"

    newest: dict[tuple, datetime] = {}
    late = 0
    current: dict[tuple, Row] = {}
    for row in valid:                      # file order = the order the pipeline reads them
        top = newest.get(row.key)
        if top is not None and row.version < top:
            late += 1                      # an older version arriving after a newer one
        if top is None or row.version > top:
            newest[row.key] = row.version
            current[row.key] = row
    summary: dict[str, Any] = {
        "rows_in": len(rows), "rows_inserted": len(valid), "rows_rejected": len(rejected),
        "duplicates_dropped": len(dropped), "late_records": late,
        "history_rows": len(valid), "current_rows": len(current),
        "rejects_by_reason": dict(sorted(Counter(r.reasons for r in rejected).items())),
    }
    assert summary["rows_in"] == (summary["rows_inserted"] + summary["rows_rejected"]
                                  + summary["duplicates_dropped"]), "expected counts do not add up"
    if valid and valid[0].cents is not None:
        summary["history_order_amount_cents_sum"] = sum(r.cents for r in valid)
        summary["current_order_amount_cents_sum"] = sum(r.cents for r in current.values())
    return summary, current


def write_input(path: Path, header: list[str], rows: list[Row]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(header)
        for row in rows:
            assert not any("\n" in value for value in row.fields)
            writer.writerow(row.fields)


def write_expected(path: Path, columns: list[str], current: dict[tuple, Row]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(columns)
        for key in sorted(current):
            writer.writerow([current[key].canon[c] for c in columns])


def resolve(path: str) -> Path:
    p = Path(path)
    return p if p.is_absolute() else PROJECT_ROOT / p


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Generate synthetic customers/orders data with labelled bad rows.")
    parser.add_argument("--customers", type=int, default=25, help="distinct valid customers (default 25)")
    parser.add_argument("--orders", type=int, default=40, help="distinct valid orders (default 40)")
    parser.add_argument("--seed", type=int, default=DEFAULT_SEED, help="random seed (default 42)")
    parser.add_argument("--bad-rate", type=float, default=0.05,
                        help="bad rows per dataset as a share of its records; every defect type is always included")
    parser.add_argument("--out-dir", default="data/sample", help="folder for customers.csv and orders.csv")
    parser.add_argument("--expected-dir", default="data/expected", help="folder for the expected-answer files")
    args = parser.parse_args(argv)
    if args.customers < 1 or args.orders < 1 or args.customers > 400_000 or not 0 <= args.bad_rate <= 1:
        parser.error("need 1 <= customers <= 400000, orders >= 1 and 0 <= bad-rate <= 1")

    customer_rows, rejected_customer_ids = build_customers(
        args.customers, args.bad_rate, random.Random(f"{args.seed}-customers"))
    order_rows = build_orders(args.orders, args.customers, args.bad_rate, rejected_customer_ids,
                              random.Random(f"{args.seed}-orders"))

    out_dir, expected_dir = resolve(args.out_dir), resolve(args.expected_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    expected_dir.mkdir(parents=True, exist_ok=True)
    write_input(out_dir / "customers.csv", CUSTOMER_HEADER, customer_rows)
    write_input(out_dir / "orders.csv", ORDER_HEADER, order_rows)

    customers_summary, customers_current = summarize(customer_rows)
    orders_summary, orders_current = summarize(order_rows)
    with open(expected_dir / "expected_rejects.csv", "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(["dataset", "source_file", "row_number", "expected_reject_reasons", "defect"])
        for name, rows in (("customers", customer_rows), ("orders", order_rows)):
            for index, row in enumerate(rows):
                if row.kind in ("bad", "conflict"):
                    writer.writerow([name, f"{name}.csv", index + 2, row.reasons, row.defect])   # header is line 1
    write_expected(expected_dir / "expected_current_customers.csv",
                   ["customer_id", "full_name", "email", "country_code", "updated_at"], customers_current)
    write_expected(expected_dir / "expected_current_orders.csv",
                   ["order_id", "customer_id", "status", "order_amount_cents", "updated_at"], orders_current)
    (expected_dir / "expected_summary.json").write_text(json.dumps({
        "generator": {"seed": args.seed, "customers": args.customers, "orders": args.orders,
                      "bad_rate": args.bad_rate},
        "customers": customers_summary, "orders": orders_summary,
    }, indent=2) + "\n", encoding="utf-8")

    for name, summary in (("customers", customers_summary), ("orders", orders_summary)):
        print(f"{name:<10} rows={summary['rows_in']:>7}  valid={summary['rows_inserted']:>7}  "
              f"rejected={summary['rows_rejected']:>5}  exact_duplicates={summary['duplicates_dropped']:>4}  "
              f"late={summary['late_records']:>5}")
    print(f"inputs   -> {out_dir}\nexpected -> {expected_dir}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


Writing generator/generate_data.py


In [22]:
%%writefile run_pipeline.py
"""Command-line entry point.

Examples:
    python run_pipeline.py                         # run every dataset in config/pipeline.yaml
    python run_pipeline.py --dataset customers     # run only customers
    python run_pipeline.py --input-dir data/large --db output/large.db
Exit code: 0 = all datasets succeeded, 1 = a dataset failed, 2 = bad config/arguments.
"""
from __future__ import annotations

import argparse
import sys

from etl.errors import ConfigError
from etl.pipeline import run_pipeline

COLUMNS = ["dataset", "status", "rows_in", "rows_inserted", "rows_already_loaded", "rows_rejected",
           "duplicates_dropped", "late_records", "duration_seconds"]


def main() -> int:
    parser = argparse.ArgumentParser(description="Run the mini ETL pipeline.")
    parser.add_argument("--config", default="config/pipeline.yaml", help="pipeline config file")
    parser.add_argument("--input-dir", help="folder with the source CSV files (overrides config)")
    parser.add_argument("--db", help="SQLite database file (overrides config)")
    parser.add_argument("--log-file", help="log file (overrides config)")
    parser.add_argument("--dataset", action="append", help="run only this dataset (repeatable)")
    args = parser.parse_args()
    try:
        results = run_pipeline(args.config, input_dir=args.input_dir, db_path=args.db,
                               log_file=args.log_file, dataset_names=args.dataset)
    except ConfigError as exc:
        print(f"CONFIG ERROR: {exc}", file=sys.stderr)
        return 2

    print(f"\nrun_id: {results[0].run_id}")
    print("  ".join(f"{c:>19}" if c != "dataset" else f"{c:<12}" for c in COLUMNS))
    for r in results:
        row = vars(r)
        print("  ".join(f"{row[c]!s:>19}" if c != "dataset" else f"{row[c]!s:<12}" for c in COLUMNS))
        if r.failure_reason:
            print(f"  -> {r.failure_reason}")
        for warning in r.warnings:
            print(f"  -> warning: {warning}")
    return 1 if any(r.status == "FAILED" for r in results) else 0


if __name__ == "__main__":
    sys.exit(main())


Writing run_pipeline.py


In [23]:
%%writefile run_report.py
"""Command-line reconciliation report.

Examples:
    python run_report.py                                              # default database, no source/expected comparison
    python run_report.py --input-dir data/sample --expected-dir data/expected
Exit code: 0 = every check passed, 1 = at least one check failed, 2 = the report could not be produced.
"""
from __future__ import annotations

import argparse
import sqlite3
import sys

from etl.config_loader import PROJECT_ROOT, load_dataset_config, load_pipeline_config, resolve_path
from etl.errors import ConfigError
from etl.reconcile import ReportError, build_report, render_markdown


def _display_path(path) -> str:
    """Show the database path relative to the project (keeps personal folder names out of saved reports)."""
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.name


def main() -> int:
    parser = argparse.ArgumentParser(description="Reconcile the target database with the source and the run log.")
    parser.add_argument("--config", default="config/pipeline.yaml", help="pipeline config file")
    parser.add_argument("--db", help="SQLite database (overrides config)")
    parser.add_argument("--input-dir", help="folder with the source CSV files, to re-count source rows")
    parser.add_argument("--expected-dir", help="folder with expected_summary.json etc., to compare with expected answers")
    parser.add_argument("--report-file", help="where to save the Markdown report (default: reports/reconciliation_report.md)")
    args = parser.parse_args()
    try:
        pipeline = load_pipeline_config(args.config)
        datasets = [load_dataset_config(p) for p in pipeline["datasets"]]
        database = resolve_path(args.db) if args.db else pipeline["database_path"]
        if not database.exists():
            raise ReportError(f"database not found: {database} (run the pipeline first)")
        report_file = resolve_path(args.report_file) if args.report_file else pipeline["reports_dir"] / "reconciliation_report.md"
        connection = sqlite3.connect(f"{database.as_uri()}?mode=ro", uri=True)      # read-only
        try:
            report = build_report(
                connection, datasets, database=_display_path(database),
                input_dir=resolve_path(args.input_dir) if args.input_dir else None,
                expected_dir=resolve_path(args.expected_dir) if args.expected_dir else None)
        finally:
            connection.close()
    except (ConfigError, ReportError, FileNotFoundError, sqlite3.DatabaseError) as exc:
        print(f"REPORT ERROR: {exc}", file=sys.stderr)
        return 2
    text = render_markdown(report)
    report_file.parent.mkdir(parents=True, exist_ok=True)
    report_file.write_text(text, encoding="utf-8")
    print(text)
    print(f"report saved to {report_file}")
    return 0 if report.passed else 1


if __name__ == "__main__":
    sys.exit(main())


Writing run_report.py


In [24]:
%%writefile run_benchmark.py
"""Performance measurement: where the time goes, how it scales, and one measured optimization.

    python run_benchmark.py                                    # 20,000 customers + 100,000 orders, 5 repeats
    python run_benchmark.py --customers 4000 --orders 20000 --repeats 3

Writes reports/performance_report.md. Exit code 0 = outputs identical before and after, 1 = they differ.

Method
  * Input comes from the seeded generator, so every run measures exactly the same data.
  * Every run starts from an empty database in a temporary folder.
  * Stage times (extract, validate, transform, load) are the ones the pipeline itself records in pipeline_runs.
  * "before" and "after" runs alternate (before, after, after, before, ...) so a slow moment of the machine
    affects both equally. The report shows the median of the repeats, plus the fastest and slowest.
  * Before = pandas string columns are looped over directly. After = each column is converted to a plain list
    once (etl.transform._plain). The tables are fingerprinted afterwards: the optimization must not change
    a single stored value.
"""
from __future__ import annotations

import argparse
import contextlib
import cProfile
import hashlib
import io
import os
import platform
import pstats
import sqlite3
import statistics
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

from etl import transform
from etl.config_loader import resolve_path
from etl.parsing import utc_iso
from etl.pipeline import run_pipeline
from generator.generate_data import main as generate

STAGES = ["extract_seconds", "validate_seconds", "transform_seconds", "load_seconds"]
FINGERPRINT_TABLES = ["customers_history", "customers_current", "orders_history", "orders_current", "rejected_records"]
# Run ids and timestamps legitimately differ from run to run; every other stored value must be identical.
VOLATILE_COLUMNS = {"run_id", "loaded_at", "first_seen_run_id", "first_seen_at", "last_seen_run_id",
                    "last_seen_at", "resolved_run_id", "resolved_at"}


@contextlib.contextmanager
def baseline_mode():
    """Reproduce the 'before' behaviour: loop over pandas columns directly instead of over a plain list."""
    original = transform._plain
    transform._plain = lambda series: series
    try:
        yield
    finally:
        transform._plain = original


def fingerprints(db_path: Path) -> dict[str, str]:
    """A short hash of every data table's content (ignoring run ids and timestamps)."""
    connection = sqlite3.connect(db_path)
    try:
        result = {}
        for table in FINGERPRINT_TABLES:
            columns = [r[1] for r in connection.execute(f"pragma table_info({table})") if r[1] not in VOLATILE_COLUMNS]
            digest = hashlib.sha256()
            for row in connection.execute(f"select {', '.join(columns)} from {table} order by {', '.join(columns)}"):
                digest.update(repr(row).encode("utf-8"))
            result[table] = digest.hexdigest()[:16]
        return result
    finally:
        connection.close()


def run_once(input_dir: Path, db_path: Path, *, baseline: bool) -> dict[str, Any]:
    """Run the whole pipeline once on an empty database; return the timings the pipeline recorded."""
    db_path.unlink(missing_ok=True)
    mode = baseline_mode() if baseline else contextlib.nullcontext()
    start = time.perf_counter()
    with mode, contextlib.redirect_stderr(io.StringIO()):                # keep the JSON log lines off the console
        results = run_pipeline("config/pipeline.yaml", input_dir=input_dir, db_path=db_path,
                               log_file=db_path.with_suffix(".log"))
    wall = time.perf_counter() - start
    failed = [r for r in results if r.status != "SUCCESS"]
    if failed:
        raise RuntimeError(f"benchmark run failed: {failed[0].dataset}: {failed[0].failure_reason}")
    connection = sqlite3.connect(db_path)
    try:
        rows = connection.execute("select dataset, rows_in, extract_seconds, validate_seconds, transform_seconds, "
                                  "load_seconds, duration_seconds from pipeline_runs").fetchall()
    finally:
        connection.close()
    return {"wall": wall,
            "datasets": {name: {"rows_in": rows_in, "total": total, **dict(zip(STAGES, stages))}
                         for name, rows_in, *stages, total in rows}}


def _stats(values: list[float]) -> tuple[float, float, float]:
    return statistics.median(values), min(values), max(values)


def _pick(runs: list[dict], dataset: str, key: str) -> list[float]:
    return [run["datasets"][dataset][key] for run in runs]


def measure_before_after(input_dir: Path, work: Path, repeats: int) -> dict[str, Any]:
    runs: dict[str, list[dict]] = {"before": [], "after": []}
    prints: dict[str, list[dict]] = {"before": [], "after": []}
    for i in range(repeats):
        for variant in (("before", "after") if i % 2 == 0 else ("after", "before")):
            db = work / f"{variant}_{i}.db"
            runs[variant].append(run_once(input_dir, db, baseline=(variant == "before")))
            if i in (0, repeats - 1):                                       # fingerprint the first and the last run
                prints[variant].append(fingerprints(db))
            db.unlink(missing_ok=True)
            print(f"  run {i + 1}/{repeats} {variant:<6} {runs[variant][-1]['wall']:6.2f} s", flush=True)
    reference = prints["before"][0]
    identical = all(p == reference for variant in prints.values() for p in variant)
    return {"runs": runs, "fingerprints": reference, "identical": identical}


def measure_scaling(work: Path, seed: int, after_runs: list[dict[str, Any]], orders: int) -> list[dict[str, Any]]:
    """Run the optimized pipeline on smaller inputs; reuse the main measurement for the largest size."""
    rows = []
    for size in [s for s in (5_000, 25_000, 50_000) if s < orders]:
        folder = work / f"scale_{size}"
        with contextlib.redirect_stdout(io.StringIO()):
            generate(["--customers", str(max(1, size // 5)), "--orders", str(size), "--seed", str(seed),
                      "--out-dir", str(folder / "in"), "--expected-dir", str(folder / "expected")])
        result = run_once(folder / "in", folder / "scale.db", baseline=False)
        rows.append(_scaling_row(result, size))
        print(f"  scaling: {size} orders -> {result['wall']:.2f} s", flush=True)
    typical = sorted(after_runs, key=lambda run: run["wall"])[len(after_runs) // 2]      # the median run, not an outlier
    rows.append(_scaling_row(typical, orders))
    return rows


def _scaling_row(run: dict[str, Any], size: int) -> dict[str, Any]:
    total_rows = sum(d["rows_in"] for d in run["datasets"].values())
    return {"orders": size, "all_rows": total_rows, "seconds": run["wall"],
            "rows_per_second": total_rows / run["wall"]}


def profile_once(input_dir: Path, db_path: Path, *, baseline: bool, top: int = 10) -> dict[str, Any]:
    """Profile one run. Returns the functions using the most CPU time themselves and the number of slow
    per-value pandas accesses (calls of the pandas array __getitem__ method)."""
    profiler = cProfile.Profile()
    profiler.enable()
    run_once(input_dir, db_path, baseline=baseline)
    profiler.disable()
    rows = []
    for (filename, _line, name), (_cc, calls, own, cumulative, _callers) in pstats.Stats(profiler).stats.items():
        label = name if filename.startswith("~") else f"{Path(filename).parent.name}/{Path(filename).name}:{name}"
        rows.append((label, own, cumulative, calls))
    return {"top": sorted(rows, key=lambda r: -r[1])[:top],
            "getitem_calls": sum(r[3] for r in rows if r[0].endswith("_mixins.py:__getitem__"))}


def _table(headers: list[str], rows: list[list[Any]]) -> list[str]:
    return (["| " + " | ".join(headers) + " |", "|" + "|".join("---" for _ in headers) + "|"]
            + ["| " + " | ".join(str(c) for c in row) + " |" for row in rows])


def render_report(args: argparse.Namespace, comparison: dict[str, Any], scaling: list[dict] | None,
                  profiles: dict[str, dict[str, Any]] | None) -> str:
    before, after = comparison["runs"]["before"], comparison["runs"]["after"]
    rows_orders = after[0]["datasets"]["orders"]["rows_in"]
    rows_customers = after[0]["datasets"]["customers"]["rows_in"]
    wall_b, wall_a = _stats([r["wall"] for r in before]), _stats([r["wall"] for r in after])
    lines = ["# Performance report", "",
             f"Generated: {utc_iso(datetime.now(timezone.utc))}  ",
             f"Machine: {platform.platform()}, {os.cpu_count()} CPUs; Python {platform.python_version()}, "
             f"pandas {pd.__version__}, SQLite {sqlite3.sqlite_version}  ",
             f"Data: seed {args.seed}, {rows_customers:,} customer rows + {rows_orders:,} order rows "
             f"({args.customers:,} customers, {args.orders:,} orders, with bad rows, duplicates and late versions)  ",
             f"Method: {args.repeats} repeats per variant, alternating order, empty database each run, medians shown "
             "(see the notes at the end for what these numbers can and cannot tell you).", "",
             "## 1. Where the time goes (after the optimization)", ""]
    for dataset in ("orders", "customers"):
        total = _stats(_pick(after, dataset, "total"))[0]
        rows = []
        for stage in STAGES:
            med, low, high = _stats(_pick(after, dataset, stage))
            rows.append([stage.replace("_seconds", ""), f"{med:.2f}", f"{low:.2f}-{high:.2f}", f"{100 * med / total:.0f}%"])
        rows_in = after[0]["datasets"][dataset]["rows_in"]
        rows.append(["**total**", f"**{total:.2f}**", f"{_stats(_pick(after, dataset, 'total'))[1]:.2f}-"
                     f"{_stats(_pick(after, dataset, 'total'))[2]:.2f}", f"{rows_in / total:,.0f} rows/s"])
        lines += [f"**{dataset}** ({rows_in:,} rows in)", ""] + _table(
            ["stage", "median seconds", "fastest-slowest", "share of dataset time"], rows) + [""]
    lines += [f"Whole run (both datasets, wall clock): **{wall_a[0]:.2f} s**, "
              f"{(rows_orders + rows_customers) / wall_a[0]:,.0f} rows per second.", ""]
    lines += ["## 2. The optimization: loop over plain lists instead of pandas string columns", "",
              "**What the measurement showed.** Load is the largest stage, then transform and validate (section 1). "
              "A profile of the unoptimized code shows millions of calls into pandas' per-value string access, coming "
              "from loops that iterate a pandas string column directly (`transform._hash_rows` and the decimal "
              "handling). Converting each column to a list once (`etl.transform._plain`) returns exactly the same values "
              "with far fewer slow calls.", ""]
    if profiles:
        lines += [f"Profile of one **before** run (the functions that use the most CPU time themselves; profiling "
                  f"slows the code, so read the proportions, not the seconds):", ""]
        lines += _table(["function", "own seconds", "cumulative seconds", "calls"],
                        [[f"`{label}`", f"{own:.2f}", f"{cum:.2f}", f"{calls:,}"]
                         for label, own, cum, calls in profiles["before"]["top"]])
        lines += ["", f"Slow per-value pandas accesses (`__getitem__` calls): **{profiles['before']['getitem_calls']:,} before, "
                      f"{profiles['after']['getitem_calls']:,} after**.", ""]
    lines += [f"**Before vs after** ({args.repeats} repeats each, orders dataset, median seconds; change = after minus before, "
              "so a negative number is faster):", ""]
    rows = []
    for stage in STAGES:
        b, a = _stats(_pick(before, "orders", stage))[0], _stats(_pick(after, "orders", stage))[0]
        rows.append([stage.replace("_seconds", ""), f"{b:.2f}", f"{a:.2f}", f"{a - b:+.2f}",
                     f"{100 * (a - b) / b:+.0f}%", f"{b / a:.2f}x"])
    tb, ta = _stats(_pick(before, "orders", "total"))[0], _stats(_pick(after, "orders", "total"))[0]
    rows.append(["**orders total**", f"**{tb:.2f}**", f"**{ta:.2f}**", f"{ta - tb:+.2f}", f"{100 * (ta - tb) / tb:+.0f}%",
                 f"**{tb / ta:.2f}x**"])
    rows.append(["**whole run (wall clock)**", f"**{wall_b[0]:.2f}**", f"**{wall_a[0]:.2f}**", f"{wall_a[0] - wall_b[0]:+.2f}",
                 f"{100 * (wall_a[0] - wall_b[0]) / wall_b[0]:+.0f}%", f"**{wall_b[0] / wall_a[0]:.2f}x**"])
    lines += _table(["stage", "before", "after", "change (s)", "change", "speed-up"], rows)
    pairs = list(zip(before, after))               # repeat i ran "before" and "after" back to back
    def paired(key) -> str:
        diffs = [key(b) - key(a) for b, a in pairs]
        return f"faster in {sum(d > 0 for d in diffs)} of {len(diffs)} repeats (median saving {statistics.median(diffs):.2f} s)"
    lines += ["", f"Whole-run range: before {wall_b[1]:.2f}-{wall_b[2]:.2f} s, after {wall_a[1]:.2f}-{wall_a[2]:.2f} s. "
              "Ranges overlap when one repeat is slow, so each repeat's before and after (run back to back) are also "
              f"compared as pairs: after was {paired(lambda r: r['wall'])} for the whole run and "
              f"{paired(lambda r: r['datasets']['orders']['transform_seconds'])} for the transform stage.", "",
              "**Output unchanged:** " + ("**yes**. " if comparison["identical"] else "**NO - the outputs differ!** ")
              + "A content fingerprint (ignoring run ids and timestamps) of every table is identical for all measured "
                "runs of both variants: "
              + ", ".join(f"`{t}` {h}" for t, h in comparison["fingerprints"].items()) + ".", ""]
    if scaling:
        lines += ["## 3. How it scales (optimized pipeline)", ""]
        lines += _table(["orders (distinct)", "rows read", "seconds", "rows per second"],
                        [[f"{r['orders']:,}", f"{r['all_rows']:,}", f"{r['seconds']:.2f}", f"{r['rows_per_second']:,.0f}"]
                         for r in scaling])
        best, last = max(r["rows_per_second"] for r in scaling), scaling[-1]["rows_per_second"]
        lines += ["", f"Throughput at the largest size is {100 * last / best:.0f}% of the best measured ({best:,.0f} rows/s). "
                  "A big drop would suggest a step whose cost grows faster than the data; small inputs are slower per row "
                  "because fixed start-up costs are spread over few rows. The smaller sizes are single runs and the largest "
                  "is the median of the repeats above.", ""]
    lines += ["## Notes and limits", "",
              "- Timings vary from run to run (the repeats above show by how much); a difference smaller than that range "
              "is not evidence of an improvement.",
              "- The gain here is moderate: pandas' own string handling and SQLite inserts dominate the rest of the time.",
              "- Memory use was not measured.",
              "- Not tried: installing `pyarrow` (pandas then uses much faster string storage). It is an extra dependency, "
              "so it is a decision for the project owner; Google Colab already includes it, so results there may differ.",
              "- Other candidates were measured separately during development (this runner does not repeat them) and not "
              "adopted, because each changed the total by only 2-4%, close to the run-to-run noise: compiling regular "
              "expressions once and applying them to plain values, building SQLite rows with `to_numpy`, and raising "
              "SQLite's page cache."]
    return "\n".join(lines) + "\n"


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Measure pipeline performance and one optimization.")
    parser.add_argument("--customers", type=int, default=20_000)
    parser.add_argument("--orders", type=int, default=100_000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--repeats", type=int, default=5, help="runs per variant (default 5)")
    parser.add_argument("--report-file", default="reports/performance_report.md")
    parser.add_argument("--no-scaling", action="store_true", help="skip the smaller-input scaling runs")
    parser.add_argument("--no-profile", action="store_true", help="skip the profile")
    args = parser.parse_args(argv)
    if args.repeats < 1:
        parser.error("--repeats must be at least 1")

    with tempfile.TemporaryDirectory(ignore_cleanup_errors=True) as folder:
        work = Path(folder)
        print(f"generating {args.customers:,} customers + {args.orders:,} orders (seed {args.seed})", flush=True)
        with contextlib.redirect_stdout(io.StringIO()):
            generate(["--customers", str(args.customers), "--orders", str(args.orders), "--seed", str(args.seed),
                      "--out-dir", str(work / "in"), "--expected-dir", str(work / "expected")])
        print("measuring before/after:", flush=True)
        comparison = measure_before_after(work / "in", work, args.repeats)
        scaling = None if args.no_scaling else measure_scaling(work, args.seed, comparison["runs"]["after"], args.orders)
        profiles = None if args.no_profile else {
            variant: profile_once(work / "in", work / f"profile_{variant}.db", baseline=(variant == "before"))
            for variant in ("before", "after")}
        report = render_report(args, comparison, scaling, profiles)

    target = resolve_path(args.report_file)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(report, encoding="utf-8", newline="\n")
    print(report)
    print(f"report saved to {target}")
    return 0 if comparison["identical"] else 1


if __name__ == "__main__":
    sys.exit(main())


Writing run_benchmark.py


In [25]:
%%writefile sql/reconciliation.sql
-- Reconciliation queries for the customers + orders warehouse (SQLite).
-- Every query returns one row: check_name, violations. 0 violations = PASS.
-- Run them all with:  sqlite3 output/warehouse.db < sql/reconciliation.sql
-- (python run_report.py runs the same kinds of checks, generated from the config, and writes a report.)

-- 1. Every successful run accounts for every input row.
select 'run_counts_add_up' as check_name, count(*) as violations
from pipeline_runs
where status = 'SUCCESS'
  and rows_in != rows_inserted + rows_already_loaded + rows_rejected + duplicates_dropped;

-- 2. Primary keys are unique (history: key + version, current: key).
select 'customers_history_key_unique', count(*) from (
  select 1 from customers_history group by customer_id, updated_at having count(*) > 1);
select 'customers_current_key_unique', count(*) from (
  select 1 from customers_current group by customer_id having count(*) > 1);
select 'orders_history_key_unique', count(*) from (
  select 1 from orders_history group by order_id, updated_at having count(*) > 1);
select 'orders_current_key_unique', count(*) from (
  select 1 from orders_current group by order_id having count(*) > 1);

-- 3. "Current" is exactly the newest version found in history.
select 'customers_current_is_newest_history_row', count(*) from customers_current c
where not exists (
  select 1 from customers_history h
  where h.customer_id = c.customer_id and h.record_hash = c.record_hash
    and h.updated_at = (select max(x.updated_at) from customers_history x where x.customer_id = c.customer_id));
select 'orders_current_is_newest_history_row', count(*) from orders_current c
where not exists (
  select 1 from orders_history h
  where h.order_id = c.order_id and h.record_hash = c.record_hash
    and h.updated_at = (select max(x.updated_at) from orders_history x where x.order_id = c.order_id));

-- 4. Every history key has a current row.
select 'customers_history_has_current', count(*) from (select distinct customer_id from customers_history) h
where not exists (select 1 from customers_current c where c.customer_id = h.customer_id);
select 'orders_history_has_current', count(*) from (select distinct order_id from orders_history) h
where not exists (select 1 from orders_current c where c.order_id = h.order_id);

-- 5. Referential integrity: no order points to a customer that does not exist.
select 'orders_reference_existing_customer', count(*) from orders_history o
where not exists (select 1 from customers_current c where c.customer_id = o.customer_id);

-- 6. History holds exactly the rows the runs say they inserted (nothing lost, nothing added by hand).
select 'customers_history_equals_inserted',
       abs((select count(*) from customers_history)
           - (select coalesce(sum(rows_inserted), 0) from pipeline_runs where dataset = 'customers' and status = 'SUCCESS'));
select 'orders_history_equals_inserted',
       abs((select count(*) from orders_history)
           - (select coalesce(sum(rows_inserted), 0) from pipeline_runs where dataset = 'orders' and status = 'SUCCESS'));

-- 7. Key aggregate: total order amount (cents) in current = total over the newest history versions.
select 'orders_amount_cents_total',
       abs((select coalesce(sum(order_amount_cents), 0) from orders_current)
           - (select coalesce(sum(h.order_amount_cents), 0) from orders_history h
              where h.updated_at = (select max(x.updated_at) from orders_history x where x.order_id = h.order_id)));

-- 8. Every reject can be traced: reason, source file, line number and the original row as JSON.
select 'rejects_traceable', count(*) from rejected_records
where reject_reasons = '' or source_file = '' or row_number < 2 or not json_valid(raw_record);


Writing sql/reconciliation.sql


In [26]:
%%writefile sql/sample_queries.sql
-- Sample queries for the customers + orders warehouse (SQLite).
-- Run one at a time in any SQLite tool, or all of them:  sqlite3 -header -column output/warehouse.db < sql/sample_queries.sql
-- Money is stored as whole cents (order_amount_cents); divide by 100.0 to show it as a currency amount.

-- Q1. Orders by their CURRENT status, with total value.
select status, count(*) as orders, sum(order_amount_cents) / 100.0 as total_amount
from orders_current
group by status
order by orders desc;

-- Q2. Delivered revenue per customer country (joins the child to its parent).
select c.country_code, count(*) as delivered_orders, sum(o.order_amount_cents) / 100.0 as revenue
from orders_current o
join customers_current c on c.customer_id = o.customer_id
where o.status = 'delivered'
group by c.country_code
order by revenue desc;

-- Q3. Top 5 customers by order value (cancelled orders excluded).
select c.customer_id, c.full_name, count(*) as orders, sum(o.order_amount_cents) / 100.0 as total_amount
from orders_current o
join customers_current c on c.customer_id = o.customer_id
where o.status != 'cancelled'
group by c.customer_id, c.full_name
order by total_amount desc
limit 5;

-- Q4. Order lifecycle from history, with the time window each version was the latest one.
select order_id, status, updated_at as valid_from,
       lead(updated_at) over (partition by order_id order by updated_at) as valid_to
from orders_history
order by order_id, updated_at
limit 20;

-- Q5. Customers whose details changed (more than one stored version).
select customer_id, count(*) as versions, min(updated_at) as first_version, max(updated_at) as latest_version
from customers_history
group by customer_id
having count(*) > 1
order by versions desc, customer_id;

-- Q6. Superseded versions: history rows that are no longer the current row (includes late arrivals).
select h.order_id, h.status as old_status, h.updated_at, c.status as current_status, c.updated_at as current_version
from orders_history h
join orders_current c on c.order_id = h.order_id and c.updated_at != h.updated_at
order by h.order_id, h.updated_at
limit 20;

-- Q7. Rejected rows by reason: what is wrong with the source data, most common first.
select dataset, reject_reasons, count(*) as rows
from rejected_records
group by dataset, reject_reasons
order by rows desc, dataset, reject_reasons;

-- Q8. Rejects still waiting to be fixed, with the original row so it can be corrected and replayed.
select dataset, source_file, row_number, reject_reasons, raw_record
from rejected_records
where resolved_run_id is null
order by dataset, row_number
limit 10;

-- Q9. Rejects that a later rerun resolved (for example orders whose customer arrived later).
select dataset, source_file, row_number, reject_reasons, first_seen_run_id, resolved_run_id
from rejected_records
where resolved_run_id is not null;

-- Q10. Run history: counts and whether the run reconciles.
select substr(run_id, 1, 8) as run, dataset, status, rows_in, rows_inserted, rows_already_loaded, rows_rejected,
       duplicates_dropped, late_records,
       case when rows_in = rows_inserted + rows_already_loaded + rows_rejected + duplicates_dropped
            then 'OK' else 'MISMATCH' end as reconciles
from pipeline_runs
order by started_at, dataset;

-- Q11. Data-quality trend: share of input rows rejected in each run.
select substr(run_id, 1, 8) as run, dataset, rows_in, rows_rejected,
       round(100.0 * rows_rejected / nullif(rows_in, 0), 2) as reject_percent
from pipeline_runs
where status = 'SUCCESS'
order by started_at, dataset;

-- Q12. Where the time goes in each run (seconds per stage) and the throughput.
select substr(run_id, 1, 8) as run, dataset, extract_seconds, validate_seconds, transform_seconds, load_seconds,
       duration_seconds, round(rows_in / nullif(duration_seconds, 0)) as rows_per_second
from pipeline_runs
where status = 'SUCCESS'
order by started_at, dataset;

-- Q13. Lineage: trace loaded orders back to the exact source file and line.
select order_id, source_file, source_row_number, run_id
from orders_current
order by order_id
limit 5;

-- Q14. Failed or unfinished runs and why.
select run_id, dataset, status, failure_reason, started_at
from pipeline_runs
where status != 'SUCCESS'
order by started_at;


Writing sql/sample_queries.sql


In [27]:
%%writefile tests/__init__.py
"""Tests for the mini ETL framework (run with: python -m pytest -q)."""


Writing tests/__init__.py


In [28]:
%%writefile tests/conftest.py
"""pytest fixtures shared by every test file."""
from __future__ import annotations

import pytest

from tests.helpers import Project


@pytest.fixture
def project(tmp_path) -> Project:
    """A fresh, empty workspace (input folder + SQLite file + log file) for one test."""
    return Project(tmp_path)


Writing tests/conftest.py


In [29]:
%%writefile tests/helpers.py
"""Shared test helpers: build small CSV files, run the pipeline, query the database.

Every test gets its own temporary folder and SQLite file, so tests never touch real data
and never affect each other.
"""
from __future__ import annotations

import csv
import sqlite3
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from etl.pipeline import DatasetResult, run_pipeline

# A fixed "current time" for every test run, so future-timestamp checks are deterministic.
NOW = datetime(2025, 6, 1, 12, 0, 0, tzinfo=timezone.utc)

CUSTOMER_HEADER = ["customer_id", "full_name", "email", "country_code", "signup_date", "updated_at"]
ORDER_HEADER = ["order_id", "customer_id", "order_ts", "status", "quantity", "unit_price", "currency", "updated_at"]
HEADERS = {"customers": CUSTOMER_HEADER, "orders": ORDER_HEADER}


def customer(**overrides: str) -> list[str]:
    """One valid customer row (as a list of text fields); override any column by name."""
    row = {"customer_id": "C000001", "full_name": "Asha Rao", "email": "asha@example.com", "country_code": "IN",
           "signup_date": "2024-01-15", "updated_at": "2024-01-15T09:00:00Z"}
    row.update(overrides)
    return [row[c] for c in CUSTOMER_HEADER]


def order(**overrides: str) -> list[str]:
    """One valid order row (as a list of text fields); override any column by name."""
    row = {"order_id": "O00000001", "customer_id": "C000001", "order_ts": "2024-02-01T10:00:00Z",
           "status": "placed", "quantity": "2", "unit_price": "10.00", "currency": "INR",
           "updated_at": "2024-02-01T10:00:00Z"}
    row.update(overrides)
    return [row[c] for c in ORDER_HEADER]


def basic_customers(count: int = 3) -> list[list[str]]:
    """C000001..C00000N, each with its own id, name and email."""
    return [customer(customer_id=f"C{i:06d}", full_name=f"User {i}", email=f"user{i}@example.com")
            for i in range(1, count + 1)]


def write_csv(path: Path, header: list[str], rows: list[list[str]]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(header)
        writer.writerows(rows)


class Project:
    """A throwaway pipeline workspace: input folder, SQLite database and log file."""

    def __init__(self, root: Path) -> None:
        self.input_dir = root / "input"
        self.input_dir.mkdir()
        self.db_path = root / "warehouse.db"
        self.log_file = root / "pipeline.log"
        self.run_ids: list[str] = []

    def write(self, dataset: str, rows: list[list[str]], header: list[str] | None = None) -> Path:
        path = self.input_dir / f"{dataset}.csv"
        write_csv(path, header or HEADERS[dataset], rows)
        return path

    def write_raw(self, dataset: str, content: bytes | str) -> Path:
        path = self.input_dir / f"{dataset}.csv"
        path.write_bytes(content if isinstance(content, bytes) else content.encode("utf-8"))
        return path

    def run(self, datasets: list[str] | None = None, now: datetime = NOW) -> dict[str, DatasetResult]:
        """Run the pipeline once; returns {dataset name: result}."""
        results = run_pipeline("config/pipeline.yaml", input_dir=self.input_dir, db_path=self.db_path,
                               log_file=self.log_file, dataset_names=datasets, now=now)
        self.run_ids.append(results[0].run_id)
        return {r.dataset: r for r in results}

    def query(self, sql: str, params: tuple = ()) -> list[tuple]:
        con = sqlite3.connect(self.db_path)
        try:
            return con.execute(sql, params).fetchall()
        finally:
            con.close()

    def scalar(self, sql: str, params: tuple = ()) -> Any:
        return self.query(sql, params)[0][0]

    def table_exists(self, name: str) -> bool:
        return bool(self.query("select 1 from sqlite_master where type = 'table' and name = ?", (name,)))

    def rejects(self, dataset: str | None = None) -> list[tuple[int, str]]:
        """[(row_number, reject_reasons)] ordered by row number."""
        if dataset:
            return self.query("select row_number, reject_reasons from rejected_records where dataset = ? "
                              "order by row_number", (dataset,))
        return self.query("select row_number, reject_reasons from rejected_records order by dataset, row_number")


def counts(result: DatasetResult) -> tuple[int, int, int, int, int, int]:
    """(rows_in, inserted, already_loaded, rejected, duplicates_dropped, late_records)."""
    return (result.rows_in, result.rows_inserted, result.rows_already_loaded, result.rows_rejected,
            result.duplicates_dropped, result.late_records)


def assert_reconciled(result: DatasetResult) -> None:
    """The core promise: every input row is loaded, already loaded, rejected or a counted duplicate."""
    accounted = (result.rows_inserted + result.rows_already_loaded + result.rows_rejected
                 + result.duplicates_dropped)
    assert result.rows_in == accounted, f"{result.dataset}: rows_in={result.rows_in} but accounted={accounted}"


def write_mixed_input(project: Project) -> None:
    """A small realistic input with good rows, a late version, duplicates, orphans and several kinds of bad rows."""
    project.write("customers", basic_customers(3) + [
        customer(customer_id="C000004", email=""),                                       # missing email
        customer(customer_id="C000005", email="e@example.com")[:-1],                     # malformed
        customer(customer_id="C000001", full_name="Newest", updated_at="2024-06-01T09:00:00Z"),
        customer(customer_id="C000001", full_name="Oldest", updated_at="2023-12-01T09:00:00Z"),   # late
        basic_customers(1)[0]])                                                          # exact duplicate of row 2
    project.write("orders", [
        order(order_id="O00000001", customer_id="C000001"),
        order(order_id="O00000002", customer_id="C000002", status="delivered", updated_at="2024-02-05T12:00:00Z"),
        order(order_id="O00000002", customer_id="C000002", status="placed", updated_at="2024-02-01T10:00:00Z"),
        order(order_id="O00000003", customer_id="C777777"),                              # orphan
        order(order_id="O00000004", quantity="0"),                                       # out of range
        order(order_id="O00000005", customer_id="C000003", quantity="3", unit_price="19.99")])


Writing tests/helpers.py


In [30]:
%%writefile tests/test_cli.py
"""The command-line entry point: exit codes and printed summary."""
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

from tests.helpers import CUSTOMER_HEADER, basic_customers

PROJECT_ROOT = Path(__file__).resolve().parents[1]


def run_cli(project, *extra: str) -> subprocess.CompletedProcess:
    return subprocess.run(
        [sys.executable, "run_pipeline.py", "--input-dir", str(project.input_dir), "--db", str(project.db_path),
         "--log-file", str(project.log_file), *extra],
        cwd=PROJECT_ROOT, capture_output=True, text=True)


def test_exit_code_0_and_summary_on_success(project):
    project.write("customers", basic_customers(2))
    done = run_cli(project, "--dataset", "customers")
    assert done.returncode == 0
    assert "SUCCESS" in done.stdout and "run_id:" in done.stdout and "customers" in done.stdout


def test_exit_code_1_and_reason_when_a_dataset_fails(project):
    project.write("customers", [["C000001"]], header=["customer_id"])
    done = run_cli(project)
    assert done.returncode == 1
    assert "FAILED" in done.stdout and "SCHEMA_MISSING_COLUMNS" in done.stdout and "SKIPPED" in done.stdout


def test_exit_code_2_for_a_configuration_error(project):
    done = run_cli(project, "--dataset", "nope")
    assert done.returncode == 2
    assert "CONFIG ERROR" in done.stderr
    assert not project.db_path.exists()


def test_extra_column_warning_is_printed(project):
    project.write("customers", [row + ["x"] for row in basic_customers(1)], header=CUSTOMER_HEADER + ["note"])
    done = run_cli(project, "--dataset", "customers")
    assert done.returncode == 0
    assert "EXTRA_COLUMNS_IGNORED" in done.stdout


Writing tests/test_cli.py


In [31]:
%%writefile tests/test_config.py
"""Configuration: mistakes fail early with a clear message, and a new dataset plugs in with config only."""
from __future__ import annotations

import copy

import pytest
import yaml

from etl.config_loader import load_dataset_config, load_pipeline_config, validate_dataset_config
from etl.errors import ConfigError
from etl.pipeline import run_pipeline
from tests.helpers import NOW, assert_reconciled, basic_customers, counts, write_csv


def _orders_config() -> dict:
    return copy.deepcopy(load_dataset_config("config/datasets/orders.yaml"))


def test_project_config_files_are_valid():
    pipeline = load_pipeline_config("config/pipeline.yaml")
    assert [load_dataset_config(p)["name"] for p in pipeline["datasets"]] == ["customers", "orders"]


def _typo(cfg):
    cfg["columns"]["quantity"]["requried"] = True


def _not_snake_case(cfg):
    cfg["columns"]["OrderQty"] = {"type": "integer"}


def _key_not_required(cfg):
    cfg["columns"]["order_id"]["required"] = False


def _key_column_missing(cfg):
    cfg["natural_key"] = ["no_such_column"]


def _unknown_type(cfg):
    cfg["columns"]["quantity"]["type"] = "float"


def _range_on_string(cfg):
    cfg["columns"]["status"]["min"] = 1


def _bad_regex(cfg):
    cfg["columns"]["order_id"]["pattern"] = "([unclosed"


def _unknown_operation(cfg):
    cfg["derived"]["order_date"]["operation"] = "rot13"


def _derived_clashes(cfg):
    cfg["derived"]["status"] = {"operation": "date_part", "sources": ["order_ts"]}


def _reference_column_missing(cfg):
    cfg["references"][0]["column"] = "nope"


def _missing_target(cfg):
    del cfg["target"]


def _unsafe_table_name(cfg):
    cfg["target"]["history_table"] = "orders_history; drop table x"


@pytest.mark.parametrize("break_it, message", [
    (_typo, "requried"),
    (_not_snake_case, "snake_case"),
    (_key_not_required, "must be required"),
    (_key_column_missing, "no_such_column"),
    (_unknown_type, "type must be one of"),
    (_range_on_string, "min/max only apply"),
    (_bad_regex, "invalid regex"),
    (_unknown_operation, "operation must be one of"),
    (_derived_clashes, "clashes"),
    (_reference_column_missing, "nope"),
    (_missing_target, "target"),
    (_unsafe_table_name, "snake_case"),
])
def test_bad_dataset_config_fails_early_with_a_clear_message(break_it, message):
    config = _orders_config()
    break_it(config)
    with pytest.raises(ConfigError, match=message):
        validate_dataset_config(config)


def test_invalid_pipeline_config_is_rejected(tmp_path):
    path = tmp_path / "pipeline.yaml"
    base = {"database_path": "a.db", "input_dir": "in", "log_file": "l.log", "reports_dir": "r",
            "datasets": ["x.yaml"]}
    path.write_text(yaml.safe_dump({**base, "validation_mode": "turbo"}), encoding="utf-8")
    with pytest.raises(ConfigError, match="validation_mode"):
        load_pipeline_config(path)
    del base["datasets"]
    path.write_text(yaml.safe_dump(base), encoding="utf-8")
    with pytest.raises(ConfigError, match="datasets"):
        load_pipeline_config(path)


def test_unknown_dataset_name_is_a_config_error_and_creates_no_database(project):
    with pytest.raises(ConfigError, match="nope"):
        run_pipeline("config/pipeline.yaml", input_dir=project.input_dir, db_path=project.db_path,
                     log_file=project.log_file, dataset_names=["nope"], now=NOW)
    assert not project.db_path.exists()


def test_new_dataset_plugs_in_through_config_only(project, tmp_path):
    """The reusability proof: a third dataset ('products') is added with a YAML file and a CSV, no Python change."""
    products_config = {
        "name": "products", "source_file": "products.csv", "natural_key": ["sku"], "version_column": "updated_at",
        "target": {"history_table": "products_history", "current_table": "products_current"},
        "columns": {
            "sku": {"type": "string", "required": True, "pattern": "^P[0-9]{4}$"},
            "product_name": {"type": "string", "required": True},
            "category": {"type": "string", "required": True, "allowed_values": ["toys", "books"]},
            "price": {"type": "decimal", "required": True, "min": 0},
            "updated_at": {"type": "timestamp", "required": True, "not_in_future": True},
        },
        "references": [],
        "derived": {"price_cents": {"operation": "multiply_to_cents", "sources": ["price"]}},
    }
    products_path = tmp_path / "products.yaml"
    products_path.write_text(yaml.safe_dump(products_config), encoding="utf-8")
    base = load_pipeline_config("config/pipeline.yaml")
    pipeline_path = tmp_path / "pipeline_with_products.yaml"
    pipeline_path.write_text(yaml.safe_dump({
        "database_path": str(project.db_path), "input_dir": str(project.input_dir),
        "log_file": str(project.log_file), "reports_dir": str(tmp_path),
        "datasets": [str(p) for p in base["datasets"]] + [str(products_path)]}), encoding="utf-8")

    project.write("customers", basic_customers(1))
    project.write("orders", [])
    write_csv(project.input_dir / "products.csv", ["sku", "product_name", "category", "price", "updated_at"], [
        ["P0001", "Robot", "toys", "19.99", "2024-01-01T00:00:00Z"],
        ["P0002", "Novel", "books", "-1", "2024-01-01T00:00:00Z"],                 # out of range
        ["P0003", "Gadget", "tools", "5", "2024-01-01T00:00:00Z"],                 # category not allowed
        ["P0001", "Robot", "toys", "19.99", "2024-01-01T00:00:00Z"],               # exact duplicate
    ])
    results = {r.dataset: r for r in run_pipeline(pipeline_path, now=NOW)}

    assert results["products"].status == "SUCCESS"
    assert counts(results["products"]) == (4, 1, 0, 2, 1, 0)
    assert_reconciled(results["products"])
    assert project.query("select sku, price_cents from products_current") == [("P0001", 1999)]
    assert project.rejects("products") == [(3, "OUT_OF_RANGE:price"), (4, "INVALID_ALLOWED_VALUE:category")]
    again = {r.dataset: r for r in run_pipeline(pipeline_path, now=NOW)}          # and it is idempotent too
    assert counts(again["products"]) == (4, 0, 1, 2, 1, 0)


Writing tests/test_config.py


In [32]:
%%writefile tests/test_duplicates.py
"""Duplicate handling inside one file."""
from __future__ import annotations

from tests.helpers import assert_reconciled, counts, customer, order, basic_customers


def test_exact_duplicate_counted_not_loaded(project):
    project.write("customers", [customer(), customer()])
    result = project.run(["customers"])["customers"]

    assert counts(result) == (2, 1, 0, 0, 1, 0)                   # in, inserted, already, rejected, dups, late
    assert project.scalar("select count(*) from customers_history") == 1
    assert project.scalar("select source_row_number from customers_history") == 2   # the FIRST copy is kept
    assert project.rejects() == []
    assert_reconciled(result)


def test_duplicate_that_differs_only_in_formatting_is_still_a_duplicate(project):
    """Whitespace, e-mail case and timestamp text are cleaned first, so these two rows are the same record."""
    project.write("customers", [
        customer(),
        customer(full_name=" Asha Rao ", email="ASHA@EXAMPLE.COM", updated_at="2024-01-15T09:00:00+00:00"),
    ])
    result = project.run(["customers"])["customers"]
    assert counts(result) == (2, 1, 0, 0, 1, 0)


def test_conflicting_duplicate_rejects_all_copies(project):
    """Same key and version but different content: we cannot know which copy is right."""
    project.write("customers", [customer(), customer(full_name="Someone Else"),
                                customer(customer_id="C000002", email="b@example.com")])
    result = project.run(["customers"])["customers"]

    assert counts(result) == (3, 1, 0, 2, 0, 0)
    reason = "CONFLICTING_DUPLICATE:customer_id,updated_at"
    assert project.rejects() == [(2, reason), (3, reason)]
    assert project.scalar("select count(*) from customers_history where customer_id = 'C000001'") == 0
    assert_reconciled(result)


def test_same_key_with_different_versions_is_not_a_duplicate(project):
    project.write("customers", [customer(updated_at="2024-01-15T09:00:00Z"),
                                customer(full_name="Asha Rao-Menon", updated_at="2024-06-01T09:00:00Z")])
    result = project.run(["customers"])["customers"]
    assert counts(result) == (2, 2, 0, 0, 0, 0)
    assert project.scalar("select count(*) from customers_history") == 2
    assert project.query("select full_name from customers_current") == [("Asha Rao-Menon",)]


def test_conflicting_order_versions_are_rejected_too(project):
    project.write("customers", basic_customers(1))
    project.write("orders", [order(unit_price="10.00"), order(unit_price="12.00")])
    result = project.run()["orders"]
    assert counts(result) == (2, 0, 0, 2, 0, 0)
    assert project.rejects("orders") == [(2, "CONFLICTING_DUPLICATE:order_id,updated_at"),
                                         (3, "CONFLICTING_DUPLICATE:order_id,updated_at")]


def test_every_input_row_is_accounted_for(project):
    """A mixed file: nothing may disappear without being loaded, rejected or counted as a duplicate."""
    project.write("customers", [
        customer(customer_id="C000001"),                                      # valid
        customer(customer_id="C000001"),                                      # exact duplicate
        customer(customer_id="C000002", email=""),                            # missing email
        customer(customer_id="C000003")[:-1],                                 # malformed
        customer(customer_id="C000004", email="d@example.com"),               # conflicting pair ...
        customer(customer_id="C000004", email="d@example.com", full_name="X"),  # ... second copy
        customer(customer_id="C000005", email="e@example.com"),               # valid
    ])
    result = project.run(["customers"])["customers"]
    assert counts(result) == (7, 2, 0, 4, 1, 0)
    assert_reconciled(result)


Writing tests/test_duplicates.py


In [33]:
%%writefile tests/test_extract.py
"""Extract behaviour: malformed rows, empty input, blank lines, encoding, missing file."""
from __future__ import annotations

import json

from tests.helpers import CUSTOMER_HEADER, assert_reconciled, counts, customer


def test_malformed_row_rejected_with_line_number(project):
    rows = [customer(customer_id="C000001"),
            customer(customer_id="C000002")[:-1],                      # 5 fields instead of 6  -> line 3
            customer(customer_id="C000003") + ["unexpected_extra"],    # 7 fields              -> line 4
            customer(customer_id="C000004", email="four@example.com")]
    project.write("customers", rows)
    result = project.run(["customers"])["customers"]

    assert result.status == "SUCCESS"
    assert counts(result) == (4, 2, 0, 2, 0, 0)          # in, inserted, already, rejected, dups, late
    assert project.rejects() == [(3, "MALFORMED_ROW"), (4, "MALFORMED_ROW")]
    raw = project.scalar("select raw_record from rejected_records where row_number = 3")
    assert json.loads(raw)["raw_fields"][0] == "C000002"          # the original row is kept for replay
    assert_reconciled(result)


def test_empty_file_succeeds_with_zero_counts(project):
    """A file with only a header is a valid empty batch."""
    project.write("customers", [])
    project.write("orders", [])
    results = project.run()
    for name in ("customers", "orders"):
        assert results[name].status == "SUCCESS"
        assert counts(results[name]) == (0, 0, 0, 0, 0, 0)
        assert project.scalar(f"select count(*) from {name}_history") == 0
        assert project.scalar(f"select count(*) from {name}_current") == 0


def test_completely_empty_file_fails_with_no_header(project):
    project.write_raw("customers", b"")
    results = project.run()
    assert results["customers"].status == "FAILED"
    assert results["customers"].failure_reason.startswith("SCHEMA_NO_HEADER")
    assert results["orders"].status == "SKIPPED"


def test_blank_lines_are_ignored_and_not_counted_as_rows(project):
    header = ",".join(CUSTOMER_HEADER)
    line = ",".join(customer())
    project.write_raw("customers", f"{header}\n\n{line}\n\n\n")
    result = project.run(["customers"])["customers"]
    assert counts(result) == (1, 1, 0, 0, 0, 0)
    assert_reconciled(result)


def test_utf8_bom_in_header_is_handled(project):
    """Excel adds a hidden BOM character; the first column name must still be recognised."""
    header = ",".join(CUSTOMER_HEADER)
    project.write_raw("customers", b"\xef\xbb\xbf" + f"{header}\n{','.join(customer())}\n".encode("utf-8"))
    result = project.run(["customers"])["customers"]
    assert result.status == "SUCCESS"
    assert result.rows_inserted == 1


def test_quoted_field_containing_a_comma_is_one_value(project):
    project.write("customers", [customer(full_name="Rao, Asha")])
    result = project.run(["customers"])["customers"]
    assert result.rows_inserted == 1
    assert project.scalar("select full_name from customers_current") == "Rao, Asha"


def test_missing_source_file_fails_with_clear_reason(project):
    result = project.run(["customers"])["customers"]
    assert result.status == "FAILED"
    assert result.failure_reason.startswith("FileNotFoundError")


Writing tests/test_extract.py


In [34]:
%%writefile tests/test_generator.py
"""The data generator: reproducible, labels kept apart from input, and the pipeline agrees with the labels."""
from __future__ import annotations

import csv
import json
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pytest

from etl.pipeline import run_pipeline
from generator.generate_data import CUSTOMER_DEFECTS, ORDER_DEFECTS, main as generate
from tests.helpers import CUSTOMER_HEADER, ORDER_HEADER, assert_reconciled

PROJECT_ROOT = Path(__file__).resolve().parents[1]
# Generated data is dated before mid-2025; use a "now" well after that so nothing valid looks like the future.
GENERATOR_NOW = datetime(2030, 1, 1, tzinfo=timezone.utc)


def make(tmp_path: Path, name: str, *args: str) -> tuple[Path, Path]:
    inputs, expected = tmp_path / name / "input", tmp_path / name / "expected"
    generate([*args, "--out-dir", str(inputs), "--expected-dir", str(expected)])
    return inputs, expected


def assert_pipeline_matches_expected(inputs: Path, expected: Path, db_path: Path) -> None:
    """Run the pipeline on generated input and compare EVERYTHING with the separately stored answers."""
    kwargs = dict(input_dir=inputs, db_path=db_path, log_file=db_path.with_suffix(".log"), now=GENERATOR_NOW)
    results = {r.dataset: r for r in run_pipeline("config/pipeline.yaml", **kwargs)}
    summary = json.loads((expected / "expected_summary.json").read_text(encoding="utf-8"))
    con = sqlite3.connect(db_path)
    try:
        for name in ("customers", "orders"):
            want, got = summary[name], results[name]
            assert got.status == "SUCCESS", got.failure_reason
            assert (got.rows_in, got.rows_inserted, got.rows_rejected, got.duplicates_dropped, got.late_records) == \
                   (want["rows_in"], want["rows_inserted"], want["rows_rejected"], want["duplicates_dropped"],
                    want["late_records"]), name
            assert_reconciled(got)
            assert con.execute(f"select count(*) from {name}_history").fetchone()[0] == want["history_rows"]
            assert con.execute(f"select count(*) from {name}_current").fetchone()[0] == want["current_rows"]

        wanted = {(r["dataset"], int(r["row_number"]), r["expected_reject_reasons"])
                  for r in csv.DictReader(open(expected / "expected_rejects.csv", encoding="utf-8"))}
        found = set(con.execute("select dataset, row_number, reject_reasons from rejected_records"))
        assert found == wanted, f"missing={sorted(wanted - found)[:3]} unexpected={sorted(found - wanted)[:3]}"

        for name, columns in (("customers", ["customer_id", "full_name", "email", "country_code", "updated_at"]),
                              ("orders", ["order_id", "customer_id", "status", "order_amount_cents", "updated_at"])):
            wanted_rows = {tuple(r[c] for c in columns) for r in
                           csv.DictReader(open(expected / f"expected_current_{name}.csv", encoding="utf-8"))}
            found_rows = {tuple(str(v) for v in r) for r in
                          con.execute(f"select {', '.join(columns)} from {name}_current")}
            assert found_rows == wanted_rows, name

        totals = [con.execute(f"select sum(order_amount_cents) from orders_{t}").fetchone()[0]
                  for t in ("current", "history")]
        assert totals == [summary["orders"]["current_order_amount_cents_sum"], summary["orders"]["history_order_amount_cents_sum"]]
    finally:
        con.close()

    rerun = run_pipeline("config/pipeline.yaml", **kwargs)                    # a rerun must change nothing
    assert all(r.rows_inserted == 0 for r in rerun)


def test_same_seed_gives_identical_files_and_a_different_seed_does_not(tmp_path):
    first = make(tmp_path, "a", "--seed", "5")
    second = make(tmp_path, "b", "--seed", "5")
    other = make(tmp_path, "c", "--seed", "6")
    names = ["customers.csv", "orders.csv"]
    names_expected = ["expected_rejects.csv", "expected_summary.json", "expected_current_customers.csv",
                      "expected_current_orders.csv"]
    for name in names:
        assert (first[0] / name).read_bytes() == (second[0] / name).read_bytes()
    for name in names_expected:
        assert (first[1] / name).read_bytes() == (second[1] / name).read_bytes()
    assert (first[0] / "orders.csv").read_bytes() != (other[0] / "orders.csv").read_bytes()


def test_answer_key_is_kept_separate_from_input(tmp_path):
    inputs, expected = make(tmp_path, "run")
    assert sorted(p.name for p in inputs.iterdir()) == ["customers.csv", "orders.csv"]
    assert (inputs / "customers.csv").read_text(encoding="utf-8").splitlines()[0] == ",".join(CUSTOMER_HEADER)
    assert (inputs / "orders.csv").read_text(encoding="utf-8").splitlines()[0] == ",".join(ORDER_HEADER)
    assert sorted(p.name for p in expected.iterdir()) == [
        "expected_current_customers.csv", "expected_current_orders.csv", "expected_rejects.csv",
        "expected_summary.json"]


def test_every_defect_type_is_planted_at_any_size(tmp_path):
    for name, args in (("small", ["--customers", "1", "--orders", "1"]), ("default", [])):
        _, expected = make(tmp_path, name, *args)
        defects = {(r["dataset"], r["defect"])
                   for r in csv.DictReader(open(expected / "expected_rejects.csv", encoding="utf-8"))}
        assert {d for ds, d in defects if ds == "customers"} == set(CUSTOMER_DEFECTS)
        assert {d for ds, d in defects if ds == "orders"} == set(ORDER_DEFECTS)


def test_generated_data_contains_the_required_scenarios(tmp_path):
    inputs, expected = make(tmp_path, "run")
    orders = list(csv.reader(open(inputs / "orders.csv", encoding="utf-8")))[1:]
    boundary = {(r[4], r[5]) for r in orders if len(r) == 8}
    assert {("1", "0.00"), ("1000", "100000.00")} <= boundary                  # boundary values that must be accepted
    summary = json.loads((expected / "expected_summary.json").read_text(encoding="utf-8"))
    assert summary["orders"]["late_records"] > 0                                 # out-of-order data
    assert summary["orders"]["duplicates_dropped"] > 0                           # exact duplicates
    assert any(k.startswith("CONFLICTING_DUPLICATE") for k in summary["orders"]["rejects_by_reason"])
    assert "INVALID_REFERENCE:customer_id" in summary["orders"]["rejects_by_reason"]
    assert "MALFORMED_ROW" in summary["customers"]["rejects_by_reason"]


@pytest.mark.parametrize("seed", [42, 7, 2024])
def test_generator_bad_rows_match_expected_labels(tmp_path, seed):
    """The pipeline must reject exactly the rows (and reasons) the generator planted, and load the rest."""
    inputs, expected = make(tmp_path, f"seed{seed}", "--seed", str(seed), "--customers", "60", "--orders", "150")
    assert_pipeline_matches_expected(inputs, expected, tmp_path / f"seed{seed}.db")


def test_committed_sample_matches_its_expected_answers(tmp_path):
    assert_pipeline_matches_expected(PROJECT_ROOT / "data" / "sample", PROJECT_ROOT / "data" / "expected",
                                     tmp_path / "sample.db")


def test_larger_generated_run_still_matches_the_labels(tmp_path):
    inputs, expected = make(tmp_path, "big", "--customers", "2000", "--orders", "10000", "--bad-rate", "0.03")
    assert_pipeline_matches_expected(inputs, expected, tmp_path / "big.db")


Writing tests/test_generator.py


In [35]:
%%writefile tests/test_idempotency.py
"""Rerun safety: running the same input again must change nothing."""
from __future__ import annotations

from tests.helpers import assert_reconciled, basic_customers, counts, customer, order

TABLES = ["customers_history", "customers_current", "orders_history", "orders_current", "rejected_records"]


def _snapshot(project) -> dict[str, list[tuple]]:
    """Every row of every data table (rejects included), so 'unchanged' really means unchanged."""
    return {t: project.query(f"select * from {t} order by 1, 2") for t in TABLES}


def _mixed_input(project) -> None:
    project.write("customers", basic_customers(3) + [customer(customer_id="C000004", email=""),
                                                     customer(customer_id="C000001", email="a@example.com")[:-1]])
    project.write("orders", [order(order_id="O00000001", customer_id="C000001"),
                             order(order_id="O00000002", customer_id="C000002", status="delivered"),
                             order(order_id="O00000002", customer_id="C000002", status="placed",
                                   updated_at="2024-01-01T10:00:00Z"),
                             order(order_id="O00000003", customer_id="C777777"),       # orphan
                             order(order_id="O00000004", quantity="0")])               # out of range


def test_rerun_is_idempotent(project):
    _mixed_input(project)
    first = project.run()
    before = _snapshot(project)
    second = project.run()
    after = _snapshot(project)

    for name in ("customers", "orders"):
        assert second[name].rows_inserted == 0
        assert second[name].rows_already_loaded == first[name].rows_inserted
        assert second[name].rows_rejected == first[name].rows_rejected          # still reported by every run
        assert_reconciled(first[name])
        assert_reconciled(second[name])
    # ... and no row of any table was added, changed or re-stamped.
    for table in TABLES:
        if table != "rejected_records":
            assert after[table] == before[table], table
    assert len(after["rejected_records"]) == len(before["rejected_records"])


def test_three_reruns_still_change_nothing(project):
    _mixed_input(project)
    project.run()
    before = _snapshot(project)
    for _ in range(3):
        project.run()
    assert [r for t, r in _snapshot(project).items() if t != "rejected_records"] == \
           [r for t, r in before.items() if t != "rejected_records"]


def test_rejected_row_is_stored_once_and_only_last_seen_changes(project):
    project.write("customers", [customer(), customer(customer_id="C000002", email="")])
    project.run(["customers"])
    project.run(["customers"])

    assert project.scalar("select count(*) from rejected_records") == 1
    first_seen, last_seen = project.query("select first_seen_run_id, last_seen_run_id from rejected_records")[0]
    assert first_seen == project.run_ids[0]
    assert last_seen == project.run_ids[1]


def test_changed_record_with_same_key_and_version_is_rejected_not_ignored(project):
    """A stored version must never be silently kept while different data with the same key and version is dropped."""
    project.write("customers", [customer(full_name="Original")])
    project.run(["customers"])
    project.write("customers", [customer(full_name="Changed")])
    result = project.run(["customers"])["customers"]

    assert counts(result) == (1, 0, 0, 1, 0, 0)
    assert project.rejects() == [(2, "CONFLICTING_DUPLICATE:customer_id,updated_at (differs from already loaded record)")]
    assert project.query("select full_name from customers_current") == [("Original",)]
    assert_reconciled(result)


def test_rerun_after_adding_new_rows_loads_only_the_new_ones(project):
    project.write("customers", basic_customers(2))
    project.run(["customers"])
    project.write("customers", basic_customers(3))
    result = project.run(["customers"])["customers"]
    assert counts(result) == (3, 1, 2, 0, 0, 0)
    assert project.scalar("select count(*) from customers_current") == 3


Writing tests/test_idempotency.py


In [36]:
%%writefile tests/test_late_data.py
"""Late-arriving and out-of-order versions: kept in history, never allowed to replace newer current data."""
from __future__ import annotations

from tests.helpers import assert_reconciled, basic_customers, counts, customer, order


def test_late_version_kept_in_history_not_current(project):
    """The newer version is listed first in the file; the older one arrives after it."""
    project.write("customers", [
        customer(full_name="Asha Rao-Menon", updated_at="2024-06-01T09:00:00Z"),
        customer(full_name="Asha Rao", updated_at="2024-01-15T09:00:00Z"),
    ])
    result = project.run(["customers"])["customers"]

    assert counts(result) == (2, 2, 0, 0, 0, 1)                       # one late record
    assert project.scalar("select count(*) from customers_history") == 2
    assert project.query("select full_name, updated_at from customers_current") == \
        [("Asha Rao-Menon", "2024-06-01T09:00:00Z")]
    assert_reconciled(result)


def test_versions_in_normal_order_are_not_late(project):
    project.write("customers", [customer(updated_at="2024-01-15T09:00:00Z"),
                                customer(full_name="Newer", updated_at="2024-06-01T09:00:00Z")])
    result = project.run(["customers"])["customers"]
    assert result.late_records == 0
    assert project.scalar("select full_name from customers_current") == "Newer"


def test_older_status_does_not_overwrite_current(project):
    """Domain edge case: an old 'placed' version arriving after 'delivered' must not roll the order back."""
    project.write("customers", basic_customers(1))
    project.write("orders", [order(status="delivered", updated_at="2024-02-05T12:00:00Z"),
                             order(status="placed", updated_at="2024-02-01T10:00:00Z")])
    result = project.run()["orders"]

    assert counts(result) == (2, 2, 0, 0, 0, 1)
    assert project.scalar("select status from orders_current") == "delivered"
    assert sorted(r[0] for r in project.query("select status from orders_history")) == ["delivered", "placed"]


def test_late_batch_in_a_later_run_is_a_backfill(project):
    """Run 1 loads the newest version; run 2 (a backfill file) adds an older one without changing 'current'."""
    project.write("customers", [customer(full_name="Newest", updated_at="2024-06-01T09:00:00Z")])
    project.run(["customers"])

    project.write("customers", [customer(full_name="Oldest", updated_at="2024-01-15T09:00:00Z")])
    result = project.run(["customers"])["customers"]

    assert counts(result) == (1, 1, 0, 0, 0, 1)
    assert project.scalar("select count(*) from customers_history") == 2
    assert project.query("select full_name from customers_current") == [("Newest",)]


def test_newer_version_in_a_later_run_replaces_current(project):
    project.write("customers", [customer(full_name="Old", updated_at="2024-01-15T09:00:00Z")])
    project.run(["customers"])
    project.write("customers", [customer(full_name="New", updated_at="2024-06-01T09:00:00Z")])
    result = project.run(["customers"])["customers"]

    assert counts(result) == (1, 1, 0, 0, 0, 0)
    assert project.query("select full_name from customers_current") == [("New",)]
    assert project.scalar("select count(*) from customers_history") == 2


def test_versions_are_compared_as_instants_not_as_text(project):
    """'10:00+05:30' is 04:30 UTC, which is EARLIER than '06:00Z', although the text sorts later."""
    project.write("customers", [
        customer(full_name="Earlier instant", updated_at="2024-03-01T10:00:00+05:30"),
        customer(full_name="Later instant", updated_at="2024-03-01T06:00:00Z"),
    ])
    project.run(["customers"])
    assert project.scalar("select full_name from customers_current") == "Later instant"


def test_history_keeps_every_version_with_its_lineage(project):
    project.write("customers", [customer(updated_at="2024-01-15T09:00:00Z"),
                                customer(full_name="Newer", updated_at="2024-06-01T09:00:00Z")])
    project.run(["customers"])
    rows = project.query("select updated_at, source_file, source_row_number, run_id from customers_history "
                         "order by updated_at")
    assert rows == [("2024-01-15T09:00:00Z", "customers.csv", 2, project.run_ids[0]),
                    ("2024-06-01T09:00:00Z", "customers.csv", 3, project.run_ids[0])]


Writing tests/test_late_data.py


In [37]:
%%writefile tests/test_observability.py
"""run_id, the pipeline_runs metrics table, and the JSON log."""
from __future__ import annotations

import json
import re

from tests.helpers import basic_customers, customer, order

ISO_UTC = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$")


def _log_lines(project) -> list[dict]:
    return [json.loads(line) for line in project.log_file.read_text(encoding="utf-8").splitlines()]


def test_pipeline_runs_has_one_row_per_dataset_with_counts_and_timings(project):
    project.write("customers", basic_customers(2) + [customer(customer_id="C000003", email="")])
    project.write("orders", [order()])
    results = project.run()

    rows = project.query("select dataset, status, rows_in, rows_inserted, rows_already_loaded, rows_rejected, "
                         "duplicates_dropped, late_records, failure_reason from pipeline_runs order by dataset")
    assert rows == [("customers", "SUCCESS", 3, 2, 0, 1, 0, 0, None),
                    ("orders", "SUCCESS", 1, 1, 0, 0, 0, 0, None)]
    assert results["customers"].rows_in == 3
    for started, ended, duration, *timings in project.query(
            "select started_at, ended_at, duration_seconds, extract_seconds, validate_seconds, transform_seconds, "
            "load_seconds from pipeline_runs"):
        assert ISO_UTC.match(started) and ISO_UTC.match(ended) and started <= ended
        assert duration >= 0
        assert all(t is not None and t >= 0 for t in timings)


def test_one_run_id_is_shared_and_stamped_everywhere(project):
    project.write("customers", [customer(), customer(customer_id="C000002", email="")])
    project.write("orders", [order()])
    project.run()
    run_id = project.run_ids[0]

    assert {r[0] for r in project.query("select run_id from pipeline_runs")} == {run_id}
    assert {r[0] for r in project.query("select run_id from customers_history")} == {run_id}
    assert {r[0] for r in project.query("select run_id from orders_current")} == {run_id}
    assert {r[0] for r in project.query("select first_seen_run_id from rejected_records")} == {run_id}
    assert all(line["run_id"] == run_id for line in _log_lines(project))


def test_every_run_gets_a_new_run_id(project):
    project.write("customers", basic_customers(1))
    project.run(["customers"])
    project.run(["customers"])
    assert len(set(project.run_ids)) == 2
    assert project.scalar("select count(distinct run_id) from pipeline_runs") == 2


def test_log_is_json_lines_with_the_expected_steps(project):
    project.write("customers", basic_customers(1))
    project.run(["customers"])
    lines = _log_lines(project)

    assert all({"ts", "level", "run_id", "message"} <= set(line) for line in lines)
    assert all(ISO_UTC.match(line["ts"]) for line in lines)
    steps = [line.get("step") for line in lines if line.get("dataset") == "customers"]
    assert steps == ["start", "extract", "validate", "validate", "finish"]
    finish = next(line for line in lines if line.get("step") == "finish")
    assert finish["status"] == "SUCCESS" and finish["rows_in"] == 1 and finish["rows_inserted"] == 1


def test_failure_reason_is_logged_and_stored(project):
    project.write("customers", [["C000001"]], header=["customer_id"])
    project.run(["customers"])

    errors = [line for line in _log_lines(project) if line["level"] == "ERROR"]
    assert len(errors) == 1
    assert errors[0]["dataset"] == "customers"
    assert errors[0]["failure_reason"].startswith("SCHEMA_MISSING_COLUMNS")
    assert "exception" in errors[0]                                             # the traceback is kept for debugging
    row = project.query("select status, failure_reason from pipeline_runs")[0]
    assert row[0] == "FAILED" and row[1].startswith("SCHEMA_MISSING_COLUMNS")


def test_warning_is_logged_for_an_extra_column(project):
    project.write("customers", [customer() + ["x"]], header=["customer_id", "full_name", "email", "country_code",
                                                             "signup_date", "updated_at", "note"])
    project.run(["customers"])
    warnings = [line for line in _log_lines(project) if line["level"] == "WARNING"]
    assert [w["extra_columns"] for w in warnings] == [["note"]]


Writing tests/test_observability.py


In [38]:
%%writefile tests/test_performance.py
"""Performance tooling: the optimization must not change results, and the benchmark runner must be trustworthy."""
from __future__ import annotations

import contextlib
import io
import sqlite3

import pandas as pd
import pytest

import run_benchmark
from etl import transform
from generator.generate_data import main as generate
from tests.helpers import order
from tests.test_transform import transformed


def make_input(tmp_path, customers=150, orders=400, seed=42):
    folder = tmp_path / f"input_{seed}"
    with contextlib.redirect_stdout(io.StringIO()):
        generate(["--customers", str(customers), "--orders", str(orders), "--seed", str(seed),
                  "--out-dir", str(folder), "--expected-dir", str(tmp_path / f"expected_{seed}")])
    return folder


# ---------- the optimization must not change a single stored value ----------

def test_optimized_and_baseline_runs_store_identical_data(tmp_path):
    folder = make_input(tmp_path)
    run_benchmark.run_once(folder, tmp_path / "before.db", baseline=True)
    run_benchmark.run_once(folder, tmp_path / "after.db", baseline=False)

    before, after = run_benchmark.fingerprints(tmp_path / "before.db"), run_benchmark.fingerprints(tmp_path / "after.db")
    assert before == after
    assert len(set(before.values())) == len(before)                      # five different tables, five different hashes


def test_transform_gives_identical_frames_with_and_without_the_optimization(tmp_path):
    rows = [order(quantity="1000", unit_price="100000.00"), order(order_id="O00000002", unit_price="0.125"),
            order(order_id="O00000003", quantity="1", unit_price="0"), order(order_id="O00000004", unit_price="19.99")]
    for case in (rows, []):                                              # including an empty batch
        optimized = transformed(tmp_path, "orders", case)
        with run_benchmark.baseline_mode():
            baseline = transformed(tmp_path, "orders", case)
        pd.testing.assert_frame_equal(optimized, baseline)


def test_baseline_mode_swaps_the_helper_and_always_restores_it():
    original = transform._plain
    values = pd.Series(["a", "b"], dtype="string")
    assert transform._plain(values) == ["a", "b"] and isinstance(transform._plain(values), list)
    with run_benchmark.baseline_mode():
        assert transform._plain(values) is values                       # "before": the pandas column itself
    assert transform._plain is original
    with pytest.raises(RuntimeError), run_benchmark.baseline_mode():
        raise RuntimeError("something failed inside the block")
    assert transform._plain is original                                  # restored even after an error


# ---------- fingerprints ----------

def test_fingerprint_ignores_run_ids_and_timestamps(tmp_path):
    folder = make_input(tmp_path)
    run_benchmark.run_once(folder, tmp_path / "a.db", baseline=False)
    run_benchmark.run_once(folder, tmp_path / "b.db", baseline=False)    # a different run_id and loaded_at
    assert run_benchmark.fingerprints(tmp_path / "a.db") == run_benchmark.fingerprints(tmp_path / "b.db")


def test_fingerprint_detects_a_changed_value_and_a_different_input(tmp_path):
    folder = make_input(tmp_path)
    run_benchmark.run_once(folder, tmp_path / "a.db", baseline=False)
    reference = run_benchmark.fingerprints(tmp_path / "a.db")

    connection = sqlite3.connect(tmp_path / "a.db")
    connection.execute("update orders_current set quantity = quantity + 1 where rowid = 1")
    connection.commit()
    connection.close()
    changed = run_benchmark.fingerprints(tmp_path / "a.db")
    assert [t for t in reference if reference[t] != changed[t]] == ["orders_current"]

    other = make_input(tmp_path, seed=7)
    run_benchmark.run_once(other, tmp_path / "c.db", baseline=False)
    assert run_benchmark.fingerprints(tmp_path / "c.db") != reference


# ---------- measurements ----------

def test_run_once_reports_consistent_timings(tmp_path):
    result = run_benchmark.run_once(make_input(tmp_path), tmp_path / "t.db", baseline=False)
    assert set(result["datasets"]) == {"customers", "orders"}
    for name, data in result["datasets"].items():
        stages = [data[s] for s in run_benchmark.STAGES]
        assert all(s >= 0 for s in stages), name
        assert sum(stages) <= data["total"] + 0.05, name                 # stages cannot add up to more than the total
        assert data["rows_in"] > 0
    assert result["wall"] >= max(d["total"] for d in result["datasets"].values())


def test_benchmark_failure_is_reported_not_hidden(tmp_path):
    folder = tmp_path / "empty_input"
    folder.mkdir()                                                       # no CSV files at all
    with pytest.raises(RuntimeError, match="benchmark run failed"):
        run_benchmark.run_once(folder, tmp_path / "x.db", baseline=False)


# ---------- the runner and its report ----------

def run_main(tmp_path, *extra):
    report = tmp_path / "performance.md"
    code = run_benchmark.main(["--customers", "1200", "--orders", "6000", "--repeats", "2",
                               "--report-file", str(report), *extra])
    return code, report.read_text(encoding="utf-8")


def test_runner_writes_a_complete_report_and_exits_zero(tmp_path, capsys):
    code, text = run_main(tmp_path)
    assert code == 0
    for heading in ("# Performance report", "## 1. Where the time goes", "## 2. The optimization",
                    "## 3. How it scales", "## Notes and limits"):
        assert heading in text
    assert "**Output unchanged:** **yes**" in text
    for stage in ("extract", "validate", "transform", "load"):
        assert f"| {stage} |" in text
    assert "rows per second" in text and "Profile of one **before** run" in text and "Slow per-value pandas accesses" in text
    assert "| 5,000 |" in text                                           # a smaller size was measured for scaling too


def test_runner_can_skip_scaling_and_profile(tmp_path):
    code, text = run_main(tmp_path, "--no-scaling", "--no-profile")
    assert code == 0
    assert "## 3. How it scales" not in text and "Profile of one" not in text


def test_runner_exits_nonzero_when_the_two_variants_differ(tmp_path, monkeypatch):
    """If an 'optimization' changed the stored data, the runner must say so and fail."""
    @contextlib.contextmanager
    def broken_baseline():
        original = transform._plain
        transform._plain = lambda series: list(reversed(series.to_numpy(dtype=object).tolist()))
        try:
            yield
        finally:
            transform._plain = original

    monkeypatch.setattr(run_benchmark, "baseline_mode", broken_baseline)
    code, text = run_main(tmp_path, "--no-scaling", "--no-profile")
    assert code == 1
    assert "**NO - the outputs differ!**" in text


Writing tests/test_performance.py


In [39]:
%%writefile tests/test_reconcile.py
"""Reconciliation report: passes on good data, and FAILS when the data is corrupted on purpose."""
from __future__ import annotations

import json
import sqlite3
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pytest

from etl.config_loader import load_dataset_config, load_pipeline_config
from etl.reconcile import ReportError, build_report, render_markdown
from generator.generate_data import main as generate
from tests.helpers import write_mixed_input

PROJECT_ROOT = Path(__file__).resolve().parents[1]
GENERATOR_NOW = datetime(2030, 1, 1, tzinfo=timezone.utc)


def datasets():
    return [load_dataset_config(p) for p in load_pipeline_config("config/pipeline.yaml")["datasets"]]


def report_for(project, *, expected_dir=None, with_source=True):
    connection = sqlite3.connect(project.db_path)
    try:
        return build_report(connection, datasets(), input_dir=project.input_dir if with_source else None,
                            expected_dir=expected_dir, database=str(project.db_path))
    finally:
        connection.close()


def failed(report) -> set[tuple[str, str]]:
    return {(c.dataset, c.name) for c in report.checks if not c.passed}


def corrupt(project, *statements: str) -> None:
    connection = sqlite3.connect(project.db_path)
    try:
        for statement in statements:
            connection.execute(statement)
        connection.commit()
    finally:
        connection.close()


def sql_statements(path: Path):
    """Split a .sql file into complete statements (comment-only lines between them are skipped)."""
    buffer = ""
    for line in path.read_text(encoding="utf-8").splitlines(keepends=True):
        if line.strip().startswith("--") and not buffer:
            continue
        buffer += line
        if sqlite3.complete_statement(buffer):
            yield buffer.strip()
            buffer = ""


# ---------- good data passes ----------

def test_clean_run_passes_every_check(project):
    write_mixed_input(project)
    project.run()
    report = report_for(project)

    assert report.passed, [(c.dataset, c.name, c.violations, c.detail) for c in report.checks if not c.passed]
    names = {(c.dataset, c.name) for c in report.checks}
    assert {("orders", "reference_customer_id"), ("orders", "aggregate_order_amount_cents"),
            ("customers", "current_is_latest_history_row"), ("all", "rejects_traceable"),
            ("customers", "source_rows_equal_rows_in")} <= names
    assert len(report.checks) >= 25


def test_report_still_passes_after_reruns_and_an_orphan_replay(project):
    write_mixed_input(project)
    project.run()
    project.run()
    # the missing customer arrives, so the orphan order loads on the next run
    customers = project.input_dir / "customers.csv"
    customers.write_text(customers.read_text(encoding="utf-8") + "C777777,Late Arrival,late@example.com,IN,"
                         "2024-01-01,2024-01-01T00:00:00Z\n", encoding="utf-8")
    project.run()
    report = report_for(project)
    assert report.passed, [(c.dataset, c.name, c.detail) for c in report.checks if not c.passed]
    assert project.scalar("select count(*) from rejected_records where resolved_run_id is not null") == 1


def test_run_table_explains_every_input_row(project):
    write_mixed_input(project)
    project.run()
    runs = {r["dataset"]: r for r in report_for(project).runs}
    for run in runs.values():
        assert run["rows_in"] == (run["rows_inserted"] + run["rows_already_loaded"] + run["rows_rejected"]
                                  + run["duplicates_dropped"])
    assert runs["customers"]["duplicates_dropped"] == 1 and runs["customers"]["late_records"] == 1


# ---------- corrupted data fails: each corruption must trip the matching check ----------

ORDER_COPY = ("insert into orders_history (order_id, customer_id, order_ts, status, quantity, unit_price, currency, "
              "updated_at, order_amount_cents, order_date, record_hash, run_id, source_file, source_row_number, "
              "loaded_at) select 'O99999999', 'C777777', order_ts, status, quantity, unit_price, currency, "
              "updated_at, order_amount_cents, order_date, record_hash, run_id, source_file, source_row_number, "
              "loaded_at from orders_history limit 1")
BAD_REJECT = ("insert into rejected_records (reject_id, dataset, source_file, row_number, reject_reasons, raw_record, "
              "first_seen_run_id, first_seen_at, last_seen_run_id, last_seen_at) values "
              "('x', 'orders', 'orders.csv', 1, '', 'not json', 'r', 't', 'r', 't')")

CORRUPTIONS = [
    ("a history row is deleted", ["delete from orders_history where rowid = (select min(rowid) from orders_history)"],
     [("orders", "history_equals_inserted")]),
    ("a current row no longer matches the newest history row",
     ["update orders_current set record_hash = 'tampered'"], [("orders", "current_is_latest_history_row")]),
    ("a current row has no history", ["delete from customers_history where customer_id = 'C000002'"],
     [("customers", "current_has_history")]),
    ("an order points to a customer that does not exist", [ORDER_COPY], [("orders", "reference_customer_id")]),
    ("a run's counts do not add up", ["update pipeline_runs set rows_in = rows_in + 5 where dataset = 'customers'"],
     [("customers", "run_counts_add_up")]),
    ("an order amount is changed by hand", ["update orders_current set order_amount_cents = order_amount_cents + 100"],
     [("orders", "aggregate_order_amount_cents")]),
    ("a reject loses its reason and its original row", [BAD_REJECT], [("all", "rejects_traceable")]),
    ("a reject uses an undocumented reason code",
     ["update rejected_records set reject_reasons = 'MADE_UP_CODE:x' where rowid = (select min(rowid) from rejected_records)"],
     [("all", "reject_reason_codes_known")]),
    ("the latest run of a dataset failed", ["update pipeline_runs set status = 'FAILED' where dataset = 'orders'"],
     [("orders", "latest_run_succeeded")]),
    ("a reject record is missing for the latest run",
     ["delete from rejected_records where dataset = 'orders' and rowid = "
      "(select min(rowid) from rejected_records where dataset = 'orders')"],
     [("orders", "rejects_cover_latest_run")]),
    ("history rows lose their run", ["update customers_history set run_id = 'ghost'"],
     [("customers", "lineage_complete")]),
]


@pytest.mark.parametrize("label, statements, expected_failures", CORRUPTIONS, ids=[c[0] for c in CORRUPTIONS])
def test_corruption_is_detected(project, label, statements, expected_failures):
    write_mixed_input(project)
    project.run()
    assert report_for(project).passed
    corrupt(project, *statements)
    report = report_for(project)
    assert not report.passed
    assert set(expected_failures) <= failed(report), failed(report)


def test_source_file_changed_after_loading_is_detected(project):
    write_mixed_input(project)
    project.run()
    customers = project.input_dir / "customers.csv"
    customers.write_text(customers.read_text(encoding="utf-8") + "C000009,Extra,x@example.com,IN,2024-01-01,"
                         "2024-01-01T00:00:00Z\n", encoding="utf-8")
    assert ("customers", "source_rows_equal_rows_in") in failed(report_for(project))


def test_report_cannot_be_built_before_the_pipeline_has_run(project):
    sqlite3.connect(project.db_path).close()
    with pytest.raises(ReportError, match="pipeline_runs"):
        report_for(project)


# ---------- comparison with the expected answers ----------

@pytest.fixture
def generated(project):
    """Generator output written straight into the project's input folder, plus its expected answers."""
    expected = project.input_dir.parent / "expected"
    generate(["--customers", "40", "--orders", "80", "--out-dir", str(project.input_dir),
              "--expected-dir", str(expected)])
    project.run(now=GENERATOR_NOW)
    return expected


def test_pipeline_output_matches_the_expected_answers(project, generated):
    report = report_for(project, expected_dir=generated)
    expected_checks = [c for c in report.checks if c.section == "expected"]
    assert len(expected_checks) >= 10
    assert report.passed, [(c.dataset, c.name, c.detail) for c in report.checks if not c.passed]
    assert {c.name for c in expected_checks} >= {"expected_run_counts", "expected_rejects", "expected_current_rows",
                                                 "expected_current_order_amount_cents_sum"}


def test_wrong_expected_answers_are_reported_as_differences(project, generated):
    rejects = generated / "expected_rejects.csv"
    lines = rejects.read_text(encoding="utf-8").splitlines()
    rejects.write_text("\n".join(lines[:1] + lines[2:]) + "\n", encoding="utf-8")        # forget one planted reject
    summary = json.loads((generated / "expected_summary.json").read_text(encoding="utf-8"))
    summary["orders"]["current_order_amount_cents_sum"] += 1
    (generated / "expected_summary.json").write_text(json.dumps(summary), encoding="utf-8")

    failures = failed(report_for(project, expected_dir=generated))
    assert any(name == "expected_rejects" for _, name in failures)
    assert ("orders", "expected_current_order_amount_cents_sum") in failures


def test_data_corruption_is_caught_by_the_expected_comparison_too(project, generated):
    corrupt(project, "update orders_current set status = 'placed'")
    assert ("orders", "expected_current_rows") in failed(report_for(project, expected_dir=generated))


# ---------- the Markdown report ----------

def test_markdown_report_shows_sections_and_result(project, generated):
    text = render_markdown(report_for(project, expected_dir=generated))
    for heading in ("# Reconciliation report", "## 1. Run reconciliation", "## 2. Where the rows went",
                    "## 3. Integrity checks", "## 4. Source-to-target", "## 5. Comparison with the expected answers"):
        assert heading in text
    assert "Result: **PASS**" in text and "FAIL" not in text.replace("PASS or FAIL", "")


def test_markdown_report_marks_failures(project):
    write_mixed_input(project)
    project.run()
    corrupt(project, "update orders_current set record_hash = 'tampered'")
    text = render_markdown(report_for(project))
    assert "Result: **FAIL**" in text and "| FAIL |" in text
    assert "section 5 was skipped" in text


# ---------- the command line ----------

def run_report_cli(project, *extra: str) -> subprocess.CompletedProcess:
    return subprocess.run([sys.executable, "run_report.py", "--db", str(project.db_path),
                           "--report-file", str(project.input_dir.parent / "report.md"), *extra],
                          cwd=PROJECT_ROOT, capture_output=True, text=True)


def test_cli_exit_0_prints_and_saves_the_report(project, generated):
    done = run_report_cli(project, "--input-dir", str(project.input_dir), "--expected-dir", str(generated))
    assert done.returncode == 0, done.stdout + done.stderr
    assert "Result: **PASS**" in done.stdout
    assert "Result: **PASS**" in (project.input_dir.parent / "report.md").read_text(encoding="utf-8")


def test_cli_exit_1_when_a_check_fails(project, generated):
    corrupt(project, "update customers_current set record_hash = 'tampered'")
    done = run_report_cli(project, "--expected-dir", str(generated))
    assert done.returncode == 1
    assert "Result: **FAIL**" in done.stdout


def test_cli_exit_2_when_there_is_no_database(project):
    done = run_report_cli(project)
    assert done.returncode == 2
    assert "REPORT ERROR" in done.stderr


# ---------- the SQL files ----------

def test_reconciliation_sql_returns_zero_violations_on_good_data(project):
    write_mixed_input(project)
    project.run()
    connection = sqlite3.connect(project.db_path)
    results = {name: violations for name, violations in
               (connection.execute(s).fetchone() for s in sql_statements(PROJECT_ROOT / "sql" / "reconciliation.sql"))}
    connection.close()
    assert len(results) >= 14
    assert all(v == 0 for v in results.values()), results


def test_reconciliation_sql_detects_corruption(project):
    write_mixed_input(project)
    project.run()
    corrupt(project, "update orders_current set order_amount_cents = order_amount_cents + 1",
            "update pipeline_runs set rows_in = rows_in + 1 where dataset = 'orders'")
    connection = sqlite3.connect(project.db_path)
    results = dict(connection.execute(s).fetchone() for s in sql_statements(PROJECT_ROOT / "sql" / "reconciliation.sql"))
    connection.close()
    assert results["orders_amount_cents_total"] > 0
    assert results["run_counts_add_up"] > 0


def test_every_sample_query_runs_and_the_main_ones_return_rows(project):
    write_mixed_input(project)
    project.run()
    connection = sqlite3.connect(project.db_path)
    queries = list(sql_statements(PROJECT_ROOT / "sql" / "sample_queries.sql"))
    results = [connection.execute(q).fetchall() for q in queries]
    connection.close()
    assert len(queries) >= 12
    assert results[0], "Q1 (orders by current status) returned nothing"
    # Orders 1 and 5 are still "placed", order 2 is "delivered" (its older "placed" version is only in history).
    assert {status: n for status, n, _ in results[0]} == {"placed": 2, "delivered": 1}


Writing tests/test_reconcile.py


In [40]:
%%writefile tests/test_recovery.py
"""Failure, rollback and retry: a failed load must leave no half-written data, and a rerun must finish the job."""
from __future__ import annotations

import pytest

import etl.extract
import etl.load
import etl.validate
from tests.helpers import assert_reconciled, basic_customers, counts, order


def test_failure_mid_load_rolls_back_and_retry_succeeds(project, monkeypatch):
    project.write("customers", basic_customers(2))
    project.write("orders", [order(order_id="O00000001", customer_id="C000001"),
                             order(order_id="O00000002", customer_id="C000002"),
                             order(order_id="O00000003", quantity="0")])     # a reject, so the failure point is reached
    real_reject_id = etl.load._reject_id
    calls = []

    def explode_for_orders(dataset, source_file, row_number, raw_record):
        """Fails AFTER the history and current rows were already written inside the transaction."""
        if dataset == "orders":
            calls.append(row_number)
            raise RuntimeError("injected failure after history was written")
        return real_reject_id(dataset, source_file, row_number, raw_record)

    monkeypatch.setattr(etl.load, "_reject_id", explode_for_orders)
    first = project.run()

    assert calls, "the injected failure point was never reached"
    assert first["customers"].status == "SUCCESS"                             # committed before orders started
    assert first["orders"].status == "FAILED"
    assert "injected failure" in first["orders"].failure_reason
    for table in ("orders_history", "orders_current"):
        assert project.scalar(f"select count(*) from {table}") == 0, table    # rolled back, nothing half-written
    assert project.rejects("orders") == []
    assert project.scalar("select count(*) from customers_history") == 2
    assert project.query("select status, failure_reason is not null from pipeline_runs "
                         "where dataset = 'orders'") == [("FAILED", 1)]

    monkeypatch.undo()                                                         # the cause is fixed; just rerun
    second = project.run()

    assert counts(second["customers"]) == (2, 0, 2, 0, 0, 0)                 # already loaded, skipped
    assert counts(second["orders"]) == (3, 2, 0, 1, 0, 0)
    assert project.scalar("select count(*) from orders_current") == 2
    assert project.rejects("orders") == [(4, "OUT_OF_RANGE:quantity")]
    assert_reconciled(second["orders"])


def test_count_mismatch_rolls_everything_back(project, monkeypatch):
    """The reconciliation safety net: if rows do not add up, nothing is committed."""
    project.write("customers", basic_customers(2))
    real_check = etl.validate.check_duplicates

    def claims_a_duplicate_that_does_not_exist(df, config):
        kept, reasons, dropped = real_check(df, config)
        return kept, reasons, dropped + 1

    monkeypatch.setattr(etl.validate, "check_duplicates", claims_a_duplicate_that_does_not_exist)
    result = project.run(["customers"])["customers"]

    assert result.status == "FAILED"
    assert result.failure_reason.startswith("ReconciliationError")
    assert project.scalar("select count(*) from customers_history") == 0
    assert project.scalar("select count(*) from customers_current") == 0


def test_crash_leaves_a_running_row_as_evidence(project, monkeypatch):
    """Even a hard crash (not a normal error) leaves a RUNNING row, so an operator can see what was in flight."""
    project.write("customers", basic_customers(1))

    def crash(path):
        raise KeyboardInterrupt

    monkeypatch.setattr(etl.extract, "extract_csv", crash)
    with pytest.raises(KeyboardInterrupt):
        project.run(["customers"])
    assert project.query("select dataset, status, ended_at from pipeline_runs") == [("customers", "RUNNING", None)]


def test_failed_run_does_not_block_the_next_run(project):
    project.write("customers", [["C000001"]], header=["customer_id"])
    assert project.run(["customers"])["customers"].status == "FAILED"
    project.write("customers", basic_customers(1))
    assert project.run(["customers"])["customers"].status == "SUCCESS"
    assert project.query("select status from pipeline_runs order by started_at") == [("FAILED",), ("SUCCESS",)]


Writing tests/test_recovery.py


In [41]:
%%writefile tests/test_references.py
"""Referential integrity: every order must point to a customer that exists."""
from __future__ import annotations

from tests.helpers import assert_reconciled, basic_customers, counts, customer, order


def test_orphan_order_rejected_invalid_reference(project):
    project.write("customers", basic_customers(2))
    project.write("orders", [order(order_id="O00000001", customer_id="C000001"),
                             order(order_id="O00000002", customer_id="C999999")])
    result = project.run()["orders"]

    assert counts(result) == (2, 1, 0, 1, 0, 0)
    assert project.rejects("orders") == [(3, "INVALID_REFERENCE:customer_id")]
    assert project.query("select order_id from orders_current") == [("O00000001",)]
    assert_reconciled(result)


def test_order_for_a_customer_that_was_itself_rejected_is_an_orphan(project):
    project.write("customers", [customer(customer_id="C000001"),
                                customer(customer_id="C000002", email="")])          # C000002 is rejected
    project.write("orders", [order(customer_id="C000002")])
    results = project.run()
    assert results["customers"].rows_rejected == 1
    assert project.rejects("orders") == [(2, "INVALID_REFERENCE:customer_id")]


def test_orphan_loads_after_parent_arrives(project):
    """Idempotent replay: rerun the SAME orders file once the customer exists."""
    project.write("customers", basic_customers(1))
    project.write("orders", [order(customer_id="C000002")])
    first = project.run()
    assert first["orders"].rows_inserted == 0 and first["orders"].rows_rejected == 1

    project.write("customers", basic_customers(2))                  # the missing customer arrives
    second = project.run()

    assert counts(second["orders"]) == (1, 1, 0, 0, 0, 0)
    assert project.query("select order_id, customer_id from orders_current") == [("O00000001", "C000002")]
    # The reject record is kept for audit and marked as resolved by the later run.
    first_seen, last_seen, resolved = project.query(
        "select first_seen_run_id, last_seen_run_id, resolved_run_id from rejected_records")[0]
    assert first_seen == last_seen == project.run_ids[0]
    assert resolved == project.run_ids[1]
    assert project.scalar("select count(*) from rejected_records") == 1


def test_reference_check_uses_customers_loaded_in_earlier_runs(project):
    project.write("customers", basic_customers(2))
    project.run(["customers"])
    project.write("orders", [order(customer_id="C000002")])
    result = project.run(["orders"])["orders"]                      # customers file is not re-read
    assert counts(result) == (1, 1, 0, 0, 0, 0)


def test_orders_run_before_any_customer_exists_are_all_orphans(project):
    project.write("orders", [order(), order(order_id="O00000002")])
    result = project.run(["orders"])["orders"]
    assert counts(result) == (2, 0, 0, 2, 0, 0)
    assert project.rejects("orders") == [(2, "INVALID_REFERENCE:customer_id"), (3, "INVALID_REFERENCE:customer_id")]


Writing tests/test_references.py


In [42]:
%%writefile tests/test_schema.py
"""Schema changes in the source file: missing, extra and duplicated columns."""
from __future__ import annotations

from tests.helpers import CUSTOMER_HEADER, assert_reconciled, basic_customers, customer, order


def test_missing_required_column_fails_batch(project):
    header = [c for c in CUSTOMER_HEADER if c != "email"]
    rows = [[v for c, v in zip(CUSTOMER_HEADER, customer()) if c != "email"]]
    project.write("customers", rows, header=header)
    project.write("orders", [order()])

    results = project.run()
    assert results["customers"].status == "FAILED"
    assert results["customers"].failure_reason == "SCHEMA_MISSING_COLUMNS: ['email']"
    assert results["customers"].rows_inserted == 0
    assert not project.table_exists("customers_history")           # nothing was written
    assert results["orders"].status == "SKIPPED"                  # a child is not run after its parent failed
    assert results["orders"].failure_reason.startswith("UPSTREAM_FAILED")


def test_fixing_the_file_and_rerunning_succeeds(project):
    project.write("customers", [["C000001", "A B"]], header=["customer_id", "full_name"])
    project.write("orders", [order()])
    assert project.run()["customers"].status == "FAILED"

    project.write("customers", basic_customers(1))
    results = project.run()
    assert results["customers"].status == "SUCCESS"
    assert results["orders"].status == "SUCCESS"
    assert results["orders"].rows_inserted == 1


def test_extra_column_warns_and_loads(project):
    project.write("customers", [customer() + ["hello"]], header=CUSTOMER_HEADER + ["note"])
    result = project.run(["customers"])["customers"]

    assert result.status == "SUCCESS"
    assert result.rows_inserted == 1
    assert result.warnings == ["EXTRA_COLUMNS_IGNORED: ['note']"]
    assert "EXTRA_COLUMNS_IGNORED" in project.scalar("select warnings from pipeline_runs")
    columns = [row[1] for row in project.query("pragma table_info(customers_history)")]
    assert "note" not in columns                                   # the extra column is ignored
    assert_reconciled(result)


def test_duplicated_column_name_fails(project):
    project.write("customers", [customer() + ["x@example.com"]], header=CUSTOMER_HEADER + ["email"])
    result = project.run(["customers"])["customers"]
    assert result.status == "FAILED"
    assert result.failure_reason == "SCHEMA_DUPLICATE_COLUMNS: ['email']"


Writing tests/test_schema.py


In [43]:
%%writefile tests/test_transform.py
"""Transform: deterministic output, exact money, timestamp normalisation, record_hash."""
from __future__ import annotations

import hashlib

import pandas as pd
import pytest

from etl.config_loader import load_dataset_config
from etl.extract import extract_csv
from etl.transform import transform
from etl.validate import normalize
from tests.helpers import HEADERS, customer, order, write_csv


def transformed(tmp_path, dataset: str, rows: list[list[str]]) -> pd.DataFrame:
    """Run extract -> normalize -> transform on a few rows (no database involved)."""
    config = load_dataset_config(f"config/datasets/{dataset}.yaml")
    path = tmp_path / f"{dataset}.csv"
    write_csv(path, HEADERS[dataset], rows)
    return transform(normalize(extract_csv(path).data, config["columns"]), config)


def test_transform_is_deterministic(tmp_path):
    rows = [customer(), customer(customer_id="C000002", email="b@example.com")]
    first = transformed(tmp_path, "customers", rows)
    second = transformed(tmp_path, "customers", rows)
    pd.testing.assert_frame_equal(first, second)
    assert first["record_hash"].tolist() == second["record_hash"].tolist()


def test_record_hash_matches_an_independent_sha256(tmp_path):
    """Pins the hash recipe: changing it would make every already-loaded row look 'different'."""
    frame = transformed(tmp_path, "customers", [customer()])
    canonical = ["C000001", "Asha Rao", "asha@example.com", "IN", "2024-01-15", "2024-01-15T09:00:00Z"]
    expected = hashlib.sha256(chr(31).join(canonical).encode("utf-8")).hexdigest()
    assert frame["record_hash"].iloc[0] == expected


def test_record_hash_ignores_formatting_but_not_content(tmp_path):
    same = transformed(tmp_path, "customers", [
        customer(),
        customer(full_name="  Asha Rao ", email="ASHA@EXAMPLE.COM", updated_at="2024-01-15T09:00:00+00:00"),
        customer(updated_at="2024-01-15 09:00:00"),                       # no offset = UTC
    ])
    assert same["record_hash"].nunique() == 1
    different = transformed(tmp_path, "customers", [customer(), customer(full_name="Asha Rao-Menon")])
    assert different["record_hash"].nunique() == 2


@pytest.mark.parametrize("quantity, price, cents", [
    ("3", "19.99", 5997),                 # 3 x 19.99, exact (binary floating point would give 5996.999...)
    ("1", "0.10", 10),
    ("7", "0.10", 70),
    ("1", "0", 0),
    ("1000", "100000.00", 10_000_000_000),
    ("1", "0.125", 13),                   # 12.5 cents rounds half up
    ("1", "0.124", 12),
])
def test_order_amount_is_exact_whole_cents(tmp_path, quantity, price, cents):
    frame = transformed(tmp_path, "orders", [order(quantity=quantity, unit_price=price)])
    assert int(frame["order_amount_cents"].iloc[0]) == cents


def test_total_of_many_small_amounts_is_exact(tmp_path):
    rows = [order(order_id=f"O{i:08d}", quantity="1", unit_price="0.10") for i in range(1, 1001)]
    frame = transformed(tmp_path, "orders", rows)
    assert int(frame["order_amount_cents"].sum()) == 10_000        # 1000 orders x 10 cents, no drift


@pytest.mark.parametrize("raw, expected", [
    ("2024-02-01T10:00:00Z", "2024-02-01T10:00:00Z"),
    ("2024-02-01T10:00:00+00:00", "2024-02-01T10:00:00Z"),
    ("2024-02-01T10:00:00+05:30", "2024-02-01T04:30:00Z"),            # converted to UTC
    ("2024-02-01T10:00:00-08:00", "2024-02-01T18:00:00Z"),
    ("2024-02-01 10:00:00", "2024-02-01T10:00:00Z"),                  # no offset is read as UTC
    ("2024-02-01T10:00", "2024-02-01T10:00:00Z"),
])
def test_timestamps_are_normalised_to_utc_iso(tmp_path, raw, expected):
    frame = transformed(tmp_path, "orders", [order(order_ts=raw)])
    assert frame["order_ts"].iloc[0] == expected


def test_derived_fields(tmp_path):
    customers = transformed(tmp_path, "customers", [customer(email="Asha@Mail.Example.org")])
    assert customers["email_domain"].iloc[0] == "mail.example.org"
    orders = transformed(tmp_path, "orders", [order(order_ts="2024-02-01T23:30:00-02:00")])
    assert orders["order_date"].iloc[0] == "2024-02-02"               # the UTC date, not the local one


def test_transform_does_not_depend_on_row_order(tmp_path):
    rows = [customer(customer_id=f"C{i:06d}", email=f"u{i}@example.com") for i in range(1, 6)]
    forward = transformed(tmp_path, "customers", rows).set_index("customer_id")
    backward = transformed(tmp_path, "customers", rows[::-1]).set_index("customer_id")
    pd.testing.assert_frame_equal(forward.sort_index(), backward.sort_index())


Writing tests/test_transform.py


In [44]:
%%writefile tests/test_validate.py
"""Row-level validation: missing values, types, formats, allowed values, ranges, future dates."""
from __future__ import annotations

import pytest

from tests.helpers import HEADERS, NOW, assert_reconciled, basic_customers, counts, customer, order


def _run_one(project, dataset: str, row: list[str]):
    """Load three good customers, then one test row of the given dataset; return that dataset's result."""
    project.write("customers", basic_customers() if dataset == "orders" else [row])
    if dataset == "orders":
        project.write("orders", [row])
    return project.run([dataset] if dataset == "customers" else None)[dataset]


# ---------- missing required values (nulls) ----------

@pytest.mark.parametrize("column", HEADERS["customers"])
def test_missing_required_customer_field_rejected(project, column):
    result = _run_one(project, "customers", customer(**{column: ""}))
    assert project.rejects() == [(2, f"MISSING_REQUIRED_FIELD:{column}")]
    assert counts(result) == (1, 0, 0, 1, 0, 0)


@pytest.mark.parametrize("column", HEADERS["orders"])
def test_missing_required_order_field_rejected(project, column):
    result = _run_one(project, "orders", order(**{column: ""}))
    assert project.rejects("orders") == [(2, f"MISSING_REQUIRED_FIELD:{column}")]
    assert counts(result) == (1, 0, 0, 1, 0, 0)


def test_whitespace_only_value_counts_as_missing(project):
    _run_one(project, "customers", customer(email="    "))
    assert project.rejects() == [(2, "MISSING_REQUIRED_FIELD:email")]


# ---------- invalid values: one reason code per rule ----------

BAD_VALUES = [
    ("customers", "customer_id", "X12", "INVALID_FORMAT:customer_id"),
    ("customers", "customer_id", "c000001", "INVALID_FORMAT:customer_id"),      # wrong case
    ("customers", "customer_id", "C0000012", "INVALID_FORMAT:customer_id"),     # 7 digits
    ("customers", "email", "a@b", "INVALID_FORMAT:email"),
    ("customers", "email", "no-at-sign.example.com", "INVALID_FORMAT:email"),
    ("customers", "email", "a b@example.com", "INVALID_FORMAT:email"),
    ("customers", "country_code", "XX", "INVALID_ALLOWED_VALUE:country_code"),
    ("customers", "country_code", "in", "INVALID_ALLOWED_VALUE:country_code"),  # matching is case-sensitive
    ("customers", "signup_date", "2024-02-30", "INVALID_TYPE:signup_date"),     # not a real date
    ("customers", "signup_date", "31/12/2024", "INVALID_TYPE:signup_date"),
    ("customers", "signup_date", "2999-01-01", "FUTURE_TIMESTAMP:signup_date"),
    ("customers", "updated_at", "not-a-timestamp", "INVALID_TYPE:updated_at"),
    ("customers", "updated_at", "2999-01-01T00:00:00Z", "FUTURE_TIMESTAMP:updated_at"),
    ("orders", "order_id", "ORD-1", "INVALID_FORMAT:order_id"),
    ("orders", "quantity", "abc", "INVALID_TYPE:quantity"),
    ("orders", "quantity", "2.5", "INVALID_TYPE:quantity"),
    ("orders", "quantity", "1e3", "INVALID_TYPE:quantity"),
    ("orders", "quantity", "0", "OUT_OF_RANGE:quantity"),
    ("orders", "quantity", "-1", "OUT_OF_RANGE:quantity"),
    ("orders", "quantity", "1001", "OUT_OF_RANGE:quantity"),
    ("orders", "unit_price", "abc", "INVALID_TYPE:unit_price"),
    ("orders", "unit_price", "-0.01", "OUT_OF_RANGE:unit_price"),
    ("orders", "unit_price", "100000.01", "OUT_OF_RANGE:unit_price"),
    ("orders", "status", "lost", "INVALID_ALLOWED_VALUE:status"),
    ("orders", "status", "Placed", "INVALID_ALLOWED_VALUE:status"),              # case-sensitive
    ("orders", "currency", "EUR", "INVALID_ALLOWED_VALUE:currency"),
    ("orders", "order_ts", "not-a-time", "INVALID_TYPE:order_ts"),
    ("orders", "order_ts", "2999-01-01T00:00:00Z", "FUTURE_TIMESTAMP:order_ts"),
]


@pytest.mark.parametrize("dataset, column, value, reason", BAD_VALUES)
def test_invalid_value_rejected_with_reason(project, dataset, column, value, reason):
    build = customer if dataset == "customers" else order
    result = _run_one(project, dataset, build(**{column: value}))
    assert project.rejects(dataset) == [(2, reason)]
    assert result.rows_inserted == 0
    assert_reconciled(result)


def test_multiple_failures_in_one_row_give_one_reject_with_all_reasons(project):
    _run_one(project, "orders", order(status="lost", unit_price="-5"))
    assert project.rejects("orders") == [(2, "INVALID_ALLOWED_VALUE:status; OUT_OF_RANGE:unit_price")]


# ---------- boundary values ----------

@pytest.mark.parametrize("quantity", ["1", "1000"])
@pytest.mark.parametrize("unit_price", ["0", "0.00", "0.01", "100000", "100000.00"])
def test_boundary_values_are_accepted(project, quantity, unit_price):
    result = _run_one(project, "orders", order(quantity=quantity, unit_price=unit_price))
    assert project.rejects("orders") == []
    assert result.rows_inserted == 1


def test_timestamp_exactly_now_is_accepted_but_one_second_later_is_future(project):
    project.write("customers", [customer(customer_id="C000001", updated_at="2025-06-01T12:00:00Z"),
                                customer(customer_id="C000002", email="b@example.com",
                                         updated_at="2025-06-01T12:00:01Z")])
    result = project.run(["customers"])["customers"]
    assert counts(result) == (2, 1, 0, 1, 0, 0)
    assert project.rejects() == [(3, "FUTURE_TIMESTAMP:updated_at")]


def test_date_today_is_accepted_but_tomorrow_is_future(project):
    assert NOW.date().isoformat() == "2025-06-01"
    project.write("customers", [customer(customer_id="C000001", signup_date="2025-06-01"),
                                customer(customer_id="C000002", email="b@example.com", signup_date="2025-06-02")])
    result = project.run(["customers"])["customers"]
    assert counts(result) == (2, 1, 0, 1, 0, 0)
    assert project.rejects() == [(3, "FUTURE_TIMESTAMP:signup_date")]


# ---------- cleaning that happens before validation ----------

def test_whitespace_is_trimmed_and_email_is_lowercased(project):
    project.write("customers", [customer(full_name="  Asha Rao  ", email="  ASHA@Example.COM ")])
    project.run(["customers"])
    assert project.query("select full_name, email, email_domain from customers_current") == \
        [("Asha Rao", "asha@example.com", "example.com")]


def test_columns_can_be_in_any_order(project):
    header = list(reversed(HEADERS["customers"]))
    project.write("customers", [list(reversed(customer()))], header=header)
    result = project.run(["customers"])["customers"]
    assert result.rows_inserted == 1
    assert project.scalar("select customer_id from customers_current") == "C000001"


# ---------- reject records must be traceable ----------

def test_reject_record_is_traceable_to_its_source(project):
    project.write("customers", [customer(), customer(customer_id="C000002", email="")])
    project.run(["customers"])
    row = project.query("select dataset, source_file, row_number, reject_reasons, raw_record, first_seen_run_id, "
                        "last_seen_run_id, resolved_run_id from rejected_records")[0]
    assert row[:4] == ("customers", "customers.csv", 3, "MISSING_REQUIRED_FIELD:email")
    assert '"customer_id": "C000002"' in row[4]
    assert row[5] == row[6] == project.run_ids[0]
    assert row[7] is None


Writing tests/test_validate.py


In [45]:
%%writefile data/sample/customers.csv
customer_id,full_name,email,country_code,signup_date,updated_at
CUST-3,Omar Patel,omar.patel29@mail.example.org,GB,2024-09-29,2024-09-29T23:22:41Z
C000001,Carlos Shah,carlos.shah1@corp.example.net,AE,2022-06-12,2022-06-12T18:03:19Z
C000002,Ivan Ali,ivan.ali2@example.com,IN,2023-01-28,2023-01-28T13:28:26Z
C000036,Kofi Mensah (changed),kofi.mensah36@mail.example.org,SG,2023-05-09,2023-05-09T01:13:07Z
C000003,Carlos Sato,carlos.sato3@mail.example.org,SG,2023-03-07,2023-03-07T05:13:58Z
C000004,Vik Ali,vik.ali4@example.com,US,2022-07-26,2022-07-26T06:08:31Z
C000005,Vik Patel,vik.patel5@example.com,SG,2024-07-05,2024-07-05T05:11:49Z
C000005,Vik Lopez,vik.patel5@example.com,IN,2024-07-05,2024-09-17T23:44:00Z
C000026,John Gupta,,US,2024-07-12,2024-07-12T08:05:41Z
C000028,Li Nair,no-at-sign.example.com,SG,2022-04-01,2022-04-01T23:25:53Z
C000006,Hiro Mensah,hiro.mensah6@example.com,SG,2022-09-24,2022-09-24T02:33:36Z
C000034,Asha Shah,,XX,2023-06-04,2023-06-04T00:36:44Z
C000007,Asha Sato,asha.sato7@corp.example.net,US,2024-04-06,2024-04-06T12:50:03Z
C000030,Uma Petrov,uma.petrov30@mail.example.org,ZZ,2024-04-14,2024-04-14T15:09:14Z
C000008,Meera Silva,meera.silva8@corp.example.net,GB,2024-02-10,2024-02-10T18:10:44Z
C000009,Li Mensah,li.mensah9@corp.example.net,US,2023-03-01,2023-03-01T00:37:11Z
C000033,Tom Shah,tom.shah33@mail.example.org,GB,2023-11-02,yesterday
C000010,John Wei,john.wei10@mail.example.org,IN,2022-08-10,2022-08-10T02:51:59Z
C000002,Ivan Ali,ivan.ali2@example.com,IN,2023-01-28,2023-01-28T13:28:26Z
C000011,Tom Wei,tom.wei11@mail.example.org,GB,2023-11-06,2023-11-06T00:35:28+00:00
C000012,Kofi Berg,kofi.berg12@mail.example.org,US,2022-10-04,2022-10-04T17:18:44Z
C000013,Vik Nair,vik.nair13@mail.example.org,AE,2024-09-22,2024-09-22T16:20:24+05:30
C000014,Hiro Rao,HIRO.RAO14@CORP.EXAMPLE.NET,IN,2024-07-26,2024-07-26T11:08:29+00:00
C000015,Asha Rao,asha.rao15@mail.example.org,GB,2022-08-18,2022-08-18T03:54:01Z
C000016,Tom Ali,tom.ali16@corp.example.net,GB,2023-10-08,2023-10-08T17:41:30+05:30
C000031,Mateo Meyer,mateo.meyer31@example.com,US,2999-01-01,2023-04-23T23:38:54Z
C000032,Asha Gupta,asha.gupta32@example.com,GB,2024-13-01,2024-03-05T07:40:55Z
C000017,Ravi Brown,ravi.brown17@corp.example.net,IN,2022-05-05,2022-05-05T16:27:22Z
C000018,Julia Smith,julia.smith18@example.com,US,2023-11-19,2023-11-19T13:52:28Z
C000027,,john.iyer27@mail.example.org,IN,2024-03-06,2024-03-06T15:59:01Z
C000019,Asha Brown,asha.brown19@mail.example.org,US,2022-09-28,2022-09-28T13:10:12Z
C000035,Uma Shah,uma.shah35@example.com,AE,2024-08-07
C000020,Kofi Patel,kofi.patel20@corp.example.net,US,2024-02-05,2024-02-05T05:19:48Z
C000021,Fatima Nair,fatima.nair21@mail.example.org,GB,2024-06-24,2024-06-24T13:04:45Z
C000036,Kofi Mensah,kofi.mensah36@mail.example.org,SG,2023-05-09,2023-05-09T01:13:07Z
C000022,Tom Gupta,tom.gupta22@mail.example.org,GB,2023-07-20,2023-07-20T20:46:10+05:30
C000023,Fatima Wei,fatima.wei23@mail.example.org,IN,2022-02-23,2022-02-23T23:02:34Z
C000021,Fatima Nair,fatima.nair21@mail.example.org,GB,2024-06-24,2024-06-24T13:04:45Z
C000024,Tom Ali,tom.ali24@example.com,AE,2024-07-19,2024-07-19T00:18:49Z
C000024,Tom Das,tom.ali24@example.com,AE,2024-07-19,2024-08-25T11:49:04Z
C000025,Carlos Petrov,carlos.petrov25@example.com,US,2022-09-09,2022-09-09T07:35:37Z


Writing data/sample/customers.csv


In [46]:
%%writefile data/sample/orders.csv
order_id,customer_id,order_ts,status,quantity,unit_price,currency,updated_at
O00000001,C000025,2024-09-11T22:57:02Z,placed,1,0.00,USD,2024-09-12T04:27:02+05:30
O00000002,C000023,2024-10-04T03:22:24Z,placed,1000,100000.00,USD,2024-10-04T03:22:24Z
O00000050,,2024-11-11T09:13:53Z,placed,7,288.73,INR,2024-11-11T09:13:53Z
O00000002,C000023,2024-10-04T03:22:24Z,shipped,1000,100000.00,USD,2024-10-04T06:36:59Z
O00000002,C000023,2024-10-04T03:22:24Z,delivered,1000,100000.00,USD,2024-10-04T14:47:46Z
O00000003,C000012,2025-03-10T23:29:18+00:00,placed,1,0.01,USD,2025-03-10T23:29:18Z
O00000003,C000012,2025-03-10T23:29:18+00:00,shipped,1,0.01,USD,2025-03-12T18:35:32Z
O00000004,C000003,2024-12-21T02:40:59Z,placed,1000,0.00,USD,2024-12-21T02:40:59Z
O00000005,C000019,2024-07-29T06:28:12Z,placed,20,737.17,USD,2024-07-29T06:28:12Z
O00000005,C000019,2024-07-29T06:28:12Z,shipped,20,737.17,USD,2024-08-01T00:06:39Z
O00000041,C000010,2024-11-24T17:17:33Z,placed,0,673.14,INR,2024-11-24T17:17:33Z
O00000005,C000019,2024-07-29T06:28:12Z,delivered,20,737.17,USD,2024-08-01T10:17:38Z
O00000006,C000019,2024-01-23T12:15:27Z,delivered,4,1031.96,INR,2024-01-24T17:51:48Z
O00000056,C000003,2024-01-10T23:12:09Z,placed,4,922.01,USD,2024-01-10T23:12:09Z
O00000007,C000013,2024-05-29T16:18:30Z,placed,12,805.43,INR,2024-05-29T16:18:30Z
O00000043,C000025,2025-02-17T20:51:10Z,placed,5,-0.01,USD,2025-02-17T20:51:10Z
O00000008,C000005,2025-02-05T10:12:18Z,delivered,3,1598.55,INR,2025-02-10T04:46:37Z
O00000009,C000002,2024-05-16T23:13:07+05:30,cancelled,10,828.26,USD,2024-05-19T03:53:49+00:00
O00000010,C000025,2025-05-11T10:34:40Z,placed,6,648.70,INR,2025-05-11T10:34:40Z
O00000010,C000025,2025-05-11T10:34:40Z,shipped,6,648.70,INR,2025-05-14T02:32:34Z
O00000010,C000025,2025-05-11T10:34:40Z,delivered,6,648.70,INR,2025-05-15T06:20:26Z
O00000011,C000018,2024-06-23T14:43:16Z,placed,15,362.33,INR,2024-06-23T14:43:16Z
O00000012,C000003,2024-06-12T14:39:49+05:30,placed,5,971.37,USD,2024-06-12T09:09:49Z
O00000012,C000003,2024-06-12T14:39:49+05:30,shipped,5,971.37,USD,2024-06-13T07:48:12Z
O00000012,C000003,2024-06-12T09:09:49Z,delivered,5,971.37,USD,2024-06-13T13:14:35Z
O00000013,C000018,2024-10-26T07:45:11+05:30,placed,20,1860.39,USD,2024-10-26T07:45:11+05:30
O00000013,C000018,2024-10-26T02:15:11Z,shipped,20,1860.39,USD,2024-10-28T15:05:30Z
O00000013,C000018,2024-10-26T02:15:11Z,delivered,20,1860.39,USD,2024-10-30T19:03:35Z
O00000014,C000013,2025-04-18T15:56:10Z,placed,8,477.07,INR,2025-04-18T15:56:10Z
O00000014,C000013,2025-04-18T15:56:10Z,shipped,8,477.07,INR,2025-04-20T00:46:17Z
O00000014,C000013,2025-04-18T15:56:10Z,delivered,8,477.07,INR,2025-04-22T00:37:48Z
O00000053,C000026,2025-01-13T23:13:58Z,placed,8,607.73,USD,2025-01-13T23:13:58Z
O00000015,C000016,2024-08-14T11:37:32Z,placed,9,1183.07,USD,2024-08-14T11:37:32Z
O00000015,C000016,2024-08-14T17:07:32+05:30,shipped,9,1183.07,USD,2024-08-16T13:49:29Z
O00000015,C000016,2024-08-14T11:37:32Z,delivered,9,1183.07,USD,2024-08-19T12:10:03Z
O00000016,C000003,2024-07-04T21:06:52Z,placed,6,1440.47,INR,2024-07-04T21:06:52Z
O00000016,C000003,2024-07-04T21:06:52Z,shipped,6,1440.47,INR,2024-07-05T06:44:06Z
O00000005,C000019,2024-07-29T06:28:12Z,shipped,20,737.17,USD,2024-08-01T00:06:39Z
O00000016,C000003,2024-07-04T21:06:52Z,delivered,6,1440.47,INR,2024-07-08T02:56:43Z
ORD-11,C000016,2024-09-30T11:19:14Z,placed,8,987.89,USD,2024-09-30T11:19:14Z
O00000054,C000002,2024-06-09T19:43:24Z,lost,11,-5.00,INR,2024-06-09T19:43:24Z
O00000017,C000020,2025-05-17T20:42:41Z,placed,1,1107.39,USD,2025-05-17T20:42:41Z
O00000017,C000020,2025-05-17T20:42:41Z,shipped,1,1107.39,USD,2025-05-20T20:13:20Z
O00000018,C000003,2024-10-22T15:09:43Z,shipped,17,1789.59,USD,2024-10-22T20:59:08Z
O00000049,C000017,,placed,17,904.30,INR,2024-11-17T08:05:34Z
O00000019,C000004,2024-11-13T01:57:19+00:00,placed,14,433.93,USD,2024-11-13T01:57:19Z
O00000019,C000004,2024-11-13T01:57:19Z,shipped,14,433.93,USD,2024-11-13T11:18:06Z
O00000046,C000015,2024-02-29T21:52:34Z,placed,2.5,791.06,USD,2024-02-29T21:52:34Z
O00000019,C000004,2024-11-13T01:57:19+00:00,delivered,14,433.93,USD,2024-11-16T03:11:35Z
O00000020,C000006,2024-03-02T20:15:52+05:30,placed,12,1748.80,USD,2024-03-02T14:45:52Z
O00000020,C000006,2024-03-02T20:15:52+05:30,shipped,12,1748.80,USD,2024-03-04T21:18:05Z
O00000021,C000009,2024-01-31T20:50:17Z,delivered,16,1636.63,USD,2024-02-03T06:03:29Z
O00000009,C000002,2024-05-16T17:43:07+00:00,placed,10,828.26,USD,2024-05-16T17:43:07Z
O00000022,C000014,2025-04-08T04:06:46Z,placed,6,132.40,INR,2025-04-08T04:06:46Z
O00000023,C000025,2024-06-24T10:20:38Z,placed,9,375.06,USD,2024-06-24T10:20:38Z
O00000023,C000025,2024-06-24T10:20:38Z,shipped,9,375.06,USD,2024-06-24T20:17:02Z
O00000023,C000025,2024-06-24T10:20:38Z,delivered,9,375.06,USD,2024-06-25T13:01:35Z
O00000024,C000012,2025-03-02T04:06:22+05:30,placed,14,1121.77,USD,2025-03-01T22:36:22Z
O00000024,C000012,2025-03-01T22:36:22+00:00,shipped,14,1121.77,USD,2025-03-04T21:21:37+05:30
O00000025,C000019,2024-11-03T06:16:42Z,placed,14,1787.46,USD,2024-11-03T06:16:42Z
O00000025,C000019,2024-11-03T06:16:42Z,shipped,14,1787.46,USD,2024-11-03T12:53:58Z
O00000025,C000019,2024-11-03T06:16:42Z,delivered,14,1787.46,USD,2024-11-05T15:47:20Z
O00000006,C000019,2024-01-23T12:15:27Z,placed,4,1031.96,INR,2024-01-23T12:15:27Z
O00000026,C000014,2025-02-25T16:14:44Z,placed,14,1761.14,USD,2025-02-25T16:14:44Z
O00000026,C000014,2025-02-25T16:14:44Z,shipped,14,1761.14,USD,2025-02-26T03:38:32Z
O00000021,C000009,2024-01-31T20:50:17Z,shipped,16,1636.63,USD,2024-02-02T11:26:45Z
O00000027,C000005,2024-01-20T05:41:40Z,placed,5,1034.24,INR,2024-01-20T05:41:40Z
O00000027,C000005,2024-01-20T05:41:40Z,shipped,5,1034.24,INR,2024-01-22T08:47:14Z
O00000027,C000005,2024-01-20T11:11:40+05:30,delivered,5,1034.24,INR,2024-01-23T09:40:43Z
O00000006,C000019,2024-01-23T12:15:27Z,shipped,4,1031.96,INR,2024-01-23T15:09:13Z
O00000047,C000012,2024-10-17T13:22:59Z,lost,3,1437.93,INR,2024-10-17T13:22:59Z
O00000028,C000020,2024-01-10T08:57:01Z,placed,17,678.13,USD,2024-01-10T08:57:01Z
O00000045,C000010,2024-08-02T00:29:36Z,placed,abc,1546.20,INR,2024-08-02T00:29:36Z
O00000028,C000020,2024-01-10T08:57:01Z,cancelled,17,678.13,USD,2024-01-12T11:30:46Z
O00000029,C000021,2024-11-23T15:53:00Z,shipped,6,1397.19,INR,2024-11-26T09:08:36Z
O00000030,C000020,2024-05-12T17:05:22+00:00,placed,13,1773.35,USD,2024-05-12T17:05:22Z
O00000030,C000020,2024-05-12T17:05:22Z,shipped,13,1773.35,USD,2024-05-14T00:36:36Z
O00000030,C000020,2024-05-12T17:05:22Z,delivered,13,1773.35,USD,2024-05-16T14:16:23Z
O00000031,C000023,2024-09-02T20:39:59Z,placed,13,1665.24,USD,2024-09-02T20:39:59Z
O00000031,C000023,2024-09-02T20:39:59Z,shipped,13,1665.24,USD,2024-09-05T15:44:42Z
O00000031,C000023,2024-09-02T20:39:59Z,delivered,13,1665.24,USD,2024-09-08T13:17:44Z
O00000032,C000022,2024-03-17T22:39:25Z,placed,14,1901.81,INR,2024-03-17T22:39:25Z
O00000032,C000022,2024-03-17T22:39:25Z,shipped,14,1901.81,INR,2024-03-20T18:44:14Z
O00000033,C000022,2024-12-28T06:12:57+05:30,placed,10,702.30,INR,2024-12-28T00:42:57Z
O00000033,C000022,2024-12-28T00:42:57Z,shipped,10,702.30,INR,2024-12-29T10:15:28Z
O00000034,C000011,2024-09-02T17:31:10Z,placed,20,949.36,INR,2024-09-02T17:31:10Z
O00000034,C000011,2024-09-02T17:31:10Z,shipped,20,949.36,INR,2024-09-05T06:53:42Z
O00000034,C000011,2024-09-02T17:31:10+00:00,delivered,20,949.36,INR,2024-09-06T00:44:58Z
O00000035,C000018,2024-03-03T17:31:04Z,placed,19,678.53,USD,2024-03-03T17:31:04Z
O00000035,C000018,2024-03-03T17:31:04Z,shipped,19,678.53,USD,2024-03-05T21:52:27+00:00
O00000055,C000019,2024-06-04T03:48:29Z,placed,14,1041.98,INR,2024-06-04T03:48:29Z,unexpected_extra_field
O00000035,C000018,2024-03-03T17:31:04Z,delivered,19,678.53,USD,2024-03-08T19:12:21Z
O00000036,C000024,2025-03-04T23:28:13Z,delivered,14,1064.28,USD,2025-03-08T10:58:41+05:30
O00000056,C000003,2024-01-10T23:12:09Z,placed,4,921.01,USD,2024-01-10T23:12:09Z
O00000037,C000009,2024-12-08T14:01:43Z,placed,10,861.86,INR,2024-12-08T14:01:43Z
O00000037,C000009,2024-12-08T14:01:43Z,shipped,10,861.86,INR,2024-12-09T11:05:58Z
O00000042,C000019,2024-09-27T17:04:47Z,placed,1001,1440.19,INR,2024-09-27T17:04:47Z
O00000037,C000009,2024-12-08T14:01:43Z,delivered,10,861.86,INR,2024-12-10T06:31:50Z
O00000036,C000024,2025-03-04T23:28:13Z,delivered,14,1064.28,USD,2025-03-08T10:58:41+05:30
O00000018,C000003,2024-10-22T15:09:43Z,placed,17,1789.59,USD,2024-10-22T20:39:43+05:30
O00000038,C000001,2024-04-29T22:22:51Z,placed,17,866.71,USD,2024-04-30T03:52:51+05:30
O00000051,C000025,2999-01-01T00:00:00Z,placed,6,1032.82,INR,2025-04-29T19:32:39Z
O00000039,C000015,2024-08-22T11:43:20Z,placed,15,1558.59,INR,2024-08-22T11:43:20Z
O00000039,C000015,2024-08-22T11:43:20Z,shipped,15,1558.59,INR,2024-08-25T06:08:25Z
O00000039,C000015,2024-08-22T11:43:20Z,delivered,15,1558.59,INR,2024-08-26T13:33:31Z
O00000040,C000009,2025-06-02T18:03:20Z,placed,2,198.94,INR,2025-06-02T18:03:20Z
O00000048,C000023,2025-01-27T23:52:34Z,placed,14,1789.67,EUR,2025-01-27T23:52:34Z
O00000008,C000005,2025-02-05T10:12:18Z,placed,3,1598.55,INR,2025-02-05T10:12:18Z
O00000036,C000024,2025-03-04T23:28:13Z,shipped,14,1064.28,USD,2025-03-05T12:45:18Z
O00000044,C000005,2025-06-03T13:41:05Z,placed,12,100000.01,USD,2025-06-03T13:41:05Z
O00000021,C000009,2024-01-31T20:50:17Z,placed,16,1636.63,USD,2024-02-01T02:20:17+05:30
O00000008,C000005,2025-02-05T10:12:18Z,shipped,3,1598.55,INR,2025-02-07T08:25:14Z
O00000029,C000021,2024-11-23T21:23:00+05:30,placed,6,1397.19,INR,2024-11-23T15:53:00Z
O00000036,C000024,2025-03-05T04:58:13+05:30,placed,14,1064.28,USD,2025-03-04T23:28:13Z


Writing data/sample/orders.csv


In [47]:
%%writefile data/expected/expected_current_customers.csv
customer_id,full_name,email,country_code,updated_at
C000001,Carlos Shah,carlos.shah1@corp.example.net,AE,2022-06-12T18:03:19Z
C000002,Ivan Ali,ivan.ali2@example.com,IN,2023-01-28T13:28:26Z
C000003,Carlos Sato,carlos.sato3@mail.example.org,SG,2023-03-07T05:13:58Z
C000004,Vik Ali,vik.ali4@example.com,US,2022-07-26T06:08:31Z
C000005,Vik Lopez,vik.patel5@example.com,IN,2024-09-17T23:44:00Z
C000006,Hiro Mensah,hiro.mensah6@example.com,SG,2022-09-24T02:33:36Z
C000007,Asha Sato,asha.sato7@corp.example.net,US,2024-04-06T12:50:03Z
C000008,Meera Silva,meera.silva8@corp.example.net,GB,2024-02-10T18:10:44Z
C000009,Li Mensah,li.mensah9@corp.example.net,US,2023-03-01T00:37:11Z
C000010,John Wei,john.wei10@mail.example.org,IN,2022-08-10T02:51:59Z
C000011,Tom Wei,tom.wei11@mail.example.org,GB,2023-11-06T00:35:28Z
C000012,Kofi Berg,kofi.berg12@mail.example.org,US,2022-10-04T17:18:44Z
C000013,Vik Nair,vik.nair13@mail.example.org,AE,2024-09-22T10:50:24Z
C000014,Hiro Rao,hiro.rao14@corp.example.net,IN,2024-07-26T11:08:29Z
C000015,Asha Rao,asha.rao15@mail.example.org,GB,2022-08-18T03:54:01Z
C000016,Tom Ali,tom.ali16@corp.example.net,GB,2023-10-08T12:11:30Z
C000017,Ravi Brown,ravi.brown17@corp.example.net,IN,2022-05-05T16:27:22Z
C000018,Julia Smith,julia.smith18@example.com,US,2023-11-19T13:52:28Z
C000019,Asha Brown,asha.brown19@mail.example.org,US,2022-09-28T13:10:12Z
C000020,Kofi Patel,kofi.patel20@corp.example.net,US,2024-02-05T05:19:48Z
C000021,Fatima Nair,fatima.nair21@mail.example.org,GB,2024-06-24T13:04:45Z
C000022,Tom Gupta,tom.gupta22@mail.example.org,GB,2023-07-20T15:16:10Z
C000023,Fatima Wei,fatima.wei23@mail.example.org,IN,2022-02-23T23:02:34Z
C000024,Tom Das,tom.ali24@example.com,AE,2024-08-25T11:49:04Z
C000025,Carlos Petrov,carlos.petrov25@example.com,US,2022-09-09T07:35:37Z


Writing data/expected/expected_current_customers.csv


In [48]:
%%writefile data/expected/expected_current_orders.csv
order_id,customer_id,status,order_amount_cents,updated_at
O00000001,C000025,placed,0,2024-09-11T22:57:02Z
O00000002,C000023,delivered,10000000000,2024-10-04T14:47:46Z
O00000003,C000012,shipped,1,2025-03-12T18:35:32Z
O00000004,C000003,placed,0,2024-12-21T02:40:59Z
O00000005,C000019,delivered,1474340,2024-08-01T10:17:38Z
O00000006,C000019,delivered,412784,2024-01-24T17:51:48Z
O00000007,C000013,placed,966516,2024-05-29T16:18:30Z
O00000008,C000005,delivered,479565,2025-02-10T04:46:37Z
O00000009,C000002,cancelled,828260,2024-05-19T03:53:49Z
O00000010,C000025,delivered,389220,2025-05-15T06:20:26Z
O00000011,C000018,placed,543495,2024-06-23T14:43:16Z
O00000012,C000003,delivered,485685,2024-06-13T13:14:35Z
O00000013,C000018,delivered,3720780,2024-10-30T19:03:35Z
O00000014,C000013,delivered,381656,2025-04-22T00:37:48Z
O00000015,C000016,delivered,1064763,2024-08-19T12:10:03Z
O00000016,C000003,delivered,864282,2024-07-08T02:56:43Z
O00000017,C000020,shipped,110739,2025-05-20T20:13:20Z
O00000018,C000003,shipped,3042303,2024-10-22T20:59:08Z
O00000019,C000004,delivered,607502,2024-11-16T03:11:35Z
O00000020,C000006,shipped,2098560,2024-03-04T21:18:05Z
O00000021,C000009,delivered,2618608,2024-02-03T06:03:29Z
O00000022,C000014,placed,79440,2025-04-08T04:06:46Z
O00000023,C000025,delivered,337554,2024-06-25T13:01:35Z
O00000024,C000012,shipped,1570478,2025-03-04T15:51:37Z
O00000025,C000019,delivered,2502444,2024-11-05T15:47:20Z
O00000026,C000014,shipped,2465596,2025-02-26T03:38:32Z
O00000027,C000005,delivered,517120,2024-01-23T09:40:43Z
O00000028,C000020,cancelled,1152821,2024-01-12T11:30:46Z
O00000029,C000021,shipped,838314,2024-11-26T09:08:36Z
O00000030,C000020,delivered,2305355,2024-05-16T14:16:23Z
O00000031,C000023,delivered,2164812,2024-09-08T13:17:44Z
O00000032,C000022,shipped,2662534,2024-03-20T18:44:14Z
O00000033,C000022,shipped,702300,2024-12-29T10:15:28Z
O00000034,C000011,delivered,1898720,2024-09-06T00:44:58Z
O00000035,C000018,delivered,1289207,2024-03-08T19:12:21Z
O00000036,C000024,delivered,1489992,2025-03-08T05:28:41Z
O00000037,C000009,delivered,861860,2024-12-10T06:31:50Z
O00000038,C000001,placed,1473407,2024-04-29T22:22:51Z
O00000039,C000015,delivered,2337885,2024-08-26T13:33:31Z
O00000040,C000009,placed,39788,2025-06-02T18:03:20Z


Writing data/expected/expected_current_orders.csv


In [49]:
%%writefile data/expected/expected_rejects.csv
dataset,source_file,row_number,expected_reject_reasons,defect
customers,customers.csv,2,INVALID_FORMAT:customer_id,invalid_customer_id_format
customers,customers.csv,5,"CONFLICTING_DUPLICATE:customer_id,updated_at",conflicting_duplicate
customers,customers.csv,10,MISSING_REQUIRED_FIELD:email,missing_email
customers,customers.csv,11,INVALID_FORMAT:email,invalid_email_format
customers,customers.csv,13,MISSING_REQUIRED_FIELD:email; INVALID_ALLOWED_VALUE:country_code,multiple_defects
customers,customers.csv,15,INVALID_ALLOWED_VALUE:country_code,invalid_country_code
customers,customers.csv,18,INVALID_TYPE:updated_at,invalid_updated_at
customers,customers.csv,27,FUTURE_TIMESTAMP:signup_date,future_signup_date
customers,customers.csv,28,INVALID_TYPE:signup_date,invalid_signup_date
customers,customers.csv,31,MISSING_REQUIRED_FIELD:full_name,missing_full_name
customers,customers.csv,33,MALFORMED_ROW,malformed_row
customers,customers.csv,36,"CONFLICTING_DUPLICATE:customer_id,updated_at",conflicting_duplicate
orders,orders.csv,4,MISSING_REQUIRED_FIELD:customer_id,missing_customer_id
orders,orders.csv,12,OUT_OF_RANGE:quantity,quantity_zero
orders,orders.csv,15,"CONFLICTING_DUPLICATE:order_id,updated_at",conflicting_duplicate
orders,orders.csv,17,OUT_OF_RANGE:unit_price,negative_price
orders,orders.csv,33,INVALID_REFERENCE:customer_id,invalid_reference
orders,orders.csv,41,INVALID_FORMAT:order_id,invalid_order_id_format
orders,orders.csv,42,INVALID_ALLOWED_VALUE:status; OUT_OF_RANGE:unit_price,multiple_defects
orders,orders.csv,46,MISSING_REQUIRED_FIELD:order_ts,missing_order_ts
orders,orders.csv,49,INVALID_TYPE:quantity,quantity_fractional
orders,orders.csv,72,INVALID_ALLOWED_VALUE:status,invalid_status
orders,orders.csv,74,INVALID_TYPE:quantity,quantity_not_a_number
orders,orders.csv,92,MALFORMED_ROW,malformed_row
orders,orders.csv,95,"CONFLICTING_DUPLICATE:order_id,updated_at",conflicting_duplicate
orders,orders.csv,98,OUT_OF_RANGE:quantity,quantity_above_max
orders,orders.csv,103,FUTURE_TIMESTAMP:order_ts,future_order_ts
orders,orders.csv,108,INVALID_ALLOWED_VALUE:currency,invalid_currency
orders,orders.csv,111,OUT_OF_RANGE:unit_price,price_above_max


Writing data/expected/expected_rejects.csv


In [50]:
%%writefile data/expected/expected_summary.json
{
  "generator": {
    "seed": 42,
    "customers": 25,
    "orders": 40,
    "bad_rate": 0.05
  },
  "customers": {
    "rows_in": 41,
    "rows_inserted": 27,
    "rows_rejected": 12,
    "duplicates_dropped": 2,
    "late_records": 0,
    "history_rows": 27,
    "current_rows": 25,
    "rejects_by_reason": {
      "CONFLICTING_DUPLICATE:customer_id,updated_at": 2,
      "FUTURE_TIMESTAMP:signup_date": 1,
      "INVALID_ALLOWED_VALUE:country_code": 1,
      "INVALID_FORMAT:customer_id": 1,
      "INVALID_FORMAT:email": 1,
      "INVALID_TYPE:signup_date": 1,
      "INVALID_TYPE:updated_at": 1,
      "MALFORMED_ROW": 1,
      "MISSING_REQUIRED_FIELD:email": 1,
      "MISSING_REQUIRED_FIELD:email; INVALID_ALLOWED_VALUE:country_code": 1,
      "MISSING_REQUIRED_FIELD:full_name": 1
    }
  },
  "orders": {
    "rows_in": 114,
    "rows_inserted": 95,
    "rows_rejected": 17,
    "duplicates_dropped": 2,
    "late_records": 11,
    "history_rows": 95,
    "current_rows": 40,
    "rejects_by_reason": {
      "CONFLICTING_DUPLICATE:order_id,updated_at": 2,
      "FUTURE_TIMESTAMP:order_ts": 1,
      "INVALID_ALLOWED_VALUE:currency": 1,
      "INVALID_ALLOWED_VALUE:status": 1,
      "INVALID_ALLOWED_VALUE:status; OUT_OF_RANGE:unit_price": 1,
      "INVALID_FORMAT:order_id": 1,
      "INVALID_REFERENCE:customer_id": 1,
      "INVALID_TYPE:quantity": 2,
      "MALFORMED_ROW": 1,
      "MISSING_REQUIRED_FIELD:customer_id": 1,
      "MISSING_REQUIRED_FIELD:order_ts": 1,
      "OUT_OF_RANGE:quantity": 2,
      "OUT_OF_RANGE:unit_price": 2
    },
    "history_order_amount_cents_sum": 30118658860,
    "current_order_amount_cents_sum": 10046778686
  }
}


Writing data/expected/expected_summary.json


In [51]:
%%writefile README.md
# Mini ETL Framework

A reproducible batch ETL framework. It reads CSV files, validates every row against rules kept in YAML config, quarantines invalid rows with a reason instead of dropping them silently, keeps the full history of every record, loads results into SQLite, and records each run with counts, timings and failure reasons. The same core code runs both datasets: `customers` (parent) and `orders` (child, referencing customers). A new dataset is added with a new config file, not new code. A reconciliation report proves the loaded data agrees with the source files, the run log and a separately stored answer key.

Stack: Python, pandas, SQLite (built into Python), PyYAML, pytest. Runs in Google Colab or locally.

Status: the framework, both datasets, the seeded data generator, 188 automated tests, the reconciliation report with sample queries, and the performance measurement are built. Still to come: a separate design and trade-off note (see "Not built yet").

## Quick start

**Google Colab (documented cloud environment)**

1. Open colab.research.google.com, then File > Upload notebook, and pick `etl_framework_submission.ipynb`.
2. Runtime > Run all. Section 1 installs the libraries, section 2 writes every project file, section 3 runs the tests, section 4 loads the sample data and prints the reconciliation report, section 5 reruns the same input, section 6 runs the SQL files, section 7 loads a 100,000-order dataset, section 8 runs the benchmark (a few minutes), and section 9 packages everything into a zip you can download.

**Locally** (Python 3.10 or newer; developed on 3.14)

```bash
python -m venv .venv
# Windows PowerShell: .venv\Scripts\Activate.ps1
# macOS/Linux:       source .venv/bin/activate
python -m pip install -r requirements.txt
python -m pytest -q
python run_pipeline.py
python run_report.py --input-dir data/sample --expected-dir data/expected
```

Expected results on the included sample (columns of the printed summary):

| dataset | rows_in | rows_inserted | rows_rejected | duplicates_dropped | late_records |
|---|---|---|---|---|---|
| customers | 41 | 27 | 12 | 2 | 0 |
| orders | 114 | 95 | 17 | 2 | 11 |

`python -m pytest -q` reports 188 passed. The report command prints `Result: **PASS** (36 of 36 checks passed)`. Running the pipeline a second time inserts nothing: `rows_inserted` is 0, `rows_already_loaded` is 27 (customers) and 95 (orders), and no table changes.

## Architecture

```text
config/pipeline.yaml + config/datasets/*.yaml
                 |
                 v
        CSV file --> extract.py   read text, keep line numbers, set aside malformed rows
                 |
                 v
        validate.py   schema check (missing required column = run fails; extra column = warning)
                 |    row rules: required, type, pattern, allowed values, range, not in future
                 |                         |
          valid rows                 rejected rows
                 |                         |
                 v                         |
        transform.py   cast types, clean values, derived fields, record_hash
                 |                         |
                 v                         |
        validate.py   duplicate keys, then parent reference check
                 |                         |
                 +-------> load.py <-------+
                           one SQLite transaction per dataset
                                  |
             +--------------------+-------------------+
             v                    v                   v
     <name>_history        <name>_current      rejected_records
                                  |
                       metrics.py -> pipeline_runs      logging_setup.py -> JSON log lines
                                  |
                       reconcile.py -> run_report.py -> reports/reconciliation_report.md
```

`pipeline.py` coordinates the steps for each dataset in the order listed in `config/pipeline.yaml`. `config_loader.py` reads and checks the config files. `parsing.py` holds the shared type-parsing helpers. `errors.py` defines the named failure types. `reconcile.py` generates its checks from the same dataset configs. No module contains customer or order logic: every rule, key, table name and derived field comes from the dataset config. `generator/generate_data.py` creates the synthetic input and the answer key; `run_benchmark.py` measures performance.

## Configuration and commands

| Command | Purpose |
|---|---|
| `python run_pipeline.py` | Load every dataset in `config/pipeline.yaml` |
| `python run_report.py` | Reconciliation report for a database (exit 0 = all checks passed) |
| `python -m generator.generate_data` | Generate the small sample and its expected answers |
| `python run_benchmark.py` | Measure performance and the optimization |
| `python -m pytest -q` | Run the automated tests |

`run_pipeline.py` options:

| Option | Default | Purpose |
|---|---|---|
| `--config FILE` | `config/pipeline.yaml` | Pipeline config: database, input folder, log file, dataset order |
| `--input-dir DIR` | `data/sample` | Folder with the source CSV files |
| `--db FILE` | `output/warehouse.db` | SQLite target database |
| `--log-file FILE` | `logs/pipeline.log` | JSON-lines log file (appended) |
| `--dataset NAME` | all | Run only this dataset (repeatable) |

Exit code 0 means every dataset succeeded, 1 means a dataset failed, 2 means a configuration or argument error. Each invocation gets one `run_id` (a UUID) shared by its datasets and stamped on every log line, metrics row, loaded row and reject row.

`run_report.py` takes `--db`, `--input-dir` (re-counts the source files), `--expected-dir` (compares with the answer key) and `--report-file`; it exits 0 when every check passes, 1 when one fails, 2 when no report can be made. `run_benchmark.py` takes `--customers`, `--orders`, `--seed`, `--repeats`, `--no-scaling`, `--no-profile` and `--report-file`.

**Adding a dataset with config only.** Create `config/datasets/<name>.yaml`, add its path to `datasets` in `config/pipeline.yaml`, and place the CSV in the input folder. A dataset config defines `name`, `source_file`, `natural_key`, `version_column`, the target tables, `columns` (type, required, pattern, allowed_values, min, max, not_in_future, transform), `references` and `derived` fields. Column types are `string`, `integer`, `decimal`, `date`, `timestamp`. Derived operations are `split_part`, `multiply_to_cents` and `date_part`; a new kind of calculation would need a small code addition. Unknown options in a config file fail the run before any data is read, so a typo cannot silently disable a rule. A test adds a third dataset (`products`) this way, with no change to `etl/`.

## Data dictionary

### Input files

| File | Columns | Meaning |
|---|---|---|
| `data/sample/customers.csv` | `customer_id`, `full_name`, `email`, `country_code`, `signup_date`, `updated_at` | One row per customer version. `customer_id` is `C` plus 6 digits; `country_code` is one of IN, US, GB, AE, SG; `email` needs one `@` and a dot after it and is stored in lowercase; `signup_date` is `YYYY-MM-DD`; `signup_date` and `updated_at` may not be in the future. |
| `data/sample/orders.csv` | `order_id`, `customer_id`, `order_ts`, `status`, `quantity`, `unit_price`, `currency`, `updated_at` | One row per order version. `order_id` is `O` plus 8 digits; `customer_id` must exist in `customers_current`; `status` is placed, shipped, delivered or cancelled; `quantity` is 1 to 1000; `unit_price` is 0 to 100000 (0 is allowed); `currency` is INR or USD. |

Extraction reads every value as text, trims whitespace, and treats empty text as missing. Rows with the wrong number of fields are rejected as `MALFORMED_ROW`. Fully blank lines are ignored but counted in the log. A file with only a header is a valid empty batch; a completely empty file fails the run. Columns are matched by name, so their order does not matter.

### Output tables (SQLite)

| Table | Key and contents |
|---|---|
| `customers_history`, `orders_history` | Append-only, one row per version, primary key (natural key, `updated_at`). Stores business columns, derived columns, `record_hash`, `run_id`, `source_file`, `source_row_number`, `loaded_at`. |
| `customers_current`, `orders_current` | One row per natural key, the latest version only. Same columns as history. Replaced only when the incoming `updated_at` is newer. |
| `rejected_records` | One row per distinct bad record: `reject_id`, `dataset`, `source_file`, `row_number` (header is line 1), `reject_reasons`, `raw_record` (the original row as JSON), `first_seen_run_id`, `first_seen_at`, `last_seen_run_id`, `last_seen_at`, `resolved_run_id`, `resolved_at`. |
| `pipeline_runs` | One row per dataset per run: `run_id`, `dataset`, `source_file`, `status` (RUNNING, SUCCESS, FAILED, SKIPPED), `started_at`, `ended_at`, `duration_seconds`, `rows_in`, `rows_inserted`, `rows_already_loaded`, `rows_rejected`, `duplicates_dropped`, `late_records`, four per-stage timings, `failure_reason`, `warnings`. |

Derived fields: `email_domain` (text after the `@`), `order_amount_cents` (quantity times unit price times 100, exact decimal arithmetic rounded half up to a whole number of cents, so source and target totals reconcile exactly), `order_date` (UTC date of `order_ts`), and `record_hash` (SHA-256 of the cleaned business columns).

Naming: snake_case tables and columns; timestamps are ISO-8601 UTC with second precision (`2024-02-01T10:00:00Z`; an offset such as `+05:30` is converted to UTC and a timestamp without an offset is read as UTC); `run_id` and `source_file` appear on loaded and rejected rows; IDs that identify data come from the source and are never random.

## Validation rules and reject codes

Each rejected input row produces one reject record that lists every failed rule, for example `INVALID_ALLOWED_VALUE:status; OUT_OF_RANGE:unit_price`.

| Code | When |
|---|---|
| `MALFORMED_ROW` | Wrong number of fields on the line |
| `MISSING_REQUIRED_FIELD` | A required value is empty |
| `INVALID_TYPE` | Not a valid integer, decimal, date or timestamp (for example `2024-02-30`) |
| `INVALID_FORMAT` | Fails the column pattern (id format, email) |
| `INVALID_ALLOWED_VALUE` | Not in the allowed list; matching is case-sensitive |
| `OUT_OF_RANGE` | Below the minimum or above the maximum |
| `FUTURE_TIMESTAMP` | Date or timestamp later than the run time |
| `CONFLICTING_DUPLICATE` | Same key and version but different content, inside the file or against an already loaded record |
| `INVALID_REFERENCE` | `customer_id` has no matching row in `customers_current` |

A missing required column, a duplicated column name or a file with no header is not a row reject: it fails the dataset with a `failure_reason` such as `SCHEMA_MISSING_COLUMNS: ['email']`, and datasets after it in the same run are marked `SKIPPED`. Extra columns are ignored and recorded as a warning.

Duplicate handling inside one file: the first copy of an exact duplicate is kept and the rest are counted in `duplicates_dropped`. Copies with the same key and version but different content cannot be trusted, so all copies are rejected as `CONFLICTING_DUPLICATE`.

## Idempotency, retry, restart and late data

Loading one dataset is a single transaction: history, current and rejects are written together or not at all. A `RUNNING` row is committed to `pipeline_runs` before any work starts, so a crashed run leaves evidence; failures are recorded as `FAILED` with their reason. Before committing, the load checks `rows_in = rows_inserted + rows_already_loaded + rows_rejected + duplicates_dropped` and rolls everything back if it does not hold.

Rerunning the same file is safe. A version already stored with the same `record_hash` is counted as `rows_already_loaded` and nothing is written. Bad records already seen keep their first reject record and only `last_seen_run_id` and `last_seen_at` change; every run still counts them in its own `rows_rejected`. Retry and restart are the same command again: a failed transaction was rolled back, and anything already committed is skipped through the primary keys.

Late and out-of-order versions: an older version arriving after a newer one is stored in history, counted in `late_records`, and does not replace the current row, so for example an order stays `delivered` when an old `placed` version arrives later. A backfill is a run on an older file; old versions go to history and the current table is protected the same way.

Orders whose customer does not exist yet are rejected as `INVALID_REFERENCE`. Rerunning the same orders file after the customer has loaded inserts the order and marks the earlier reject as resolved (`resolved_run_id`, matched by file name and row number).

## Reproducible sample and larger run

The generator uses a fixed seed (default 42): the same seed and options always give identical files. It writes the **input** files (`customers.csv`, `orders.csv`) to one folder and the **answer key** to another, so labels never sit next to the data they describe:

| File in `data/expected/` | Meaning |
|---|---|
| `expected_rejects.csv` | Every planted bad row: file, line number, exact expected reject reason, and a readable defect name |
| `expected_summary.json` | Expected counts per dataset (rows in, inserted, rejected, duplicates, late, history and current rows) and order totals in cents |
| `expected_current_customers.csv`, `expected_current_orders.csv` | The latest version of every valid record |

It plants valid records with several versions, boundary orders (quantity 1 and 1000, price 0.00 and 100000.00), timestamps in `Z`, `+00:00` and `+05:30` forms, padded names and upper-case e-mails, older versions placed after newer ones in the file, exact duplicates, and 11 customer and 16 order defect types (missing values, wrong types, out-of-range values, conflicting duplicates, unknown customers, malformed lines and a row with two defects). Every defect type appears at any size. Generate a larger dataset (about 24,600 customer rows and 240,000 order rows) with:

```bash
python -m generator.generate_data --customers 20000 --orders 100000 --out-dir data/large --expected-dir data/large/expected
python run_pipeline.py --input-dir data/large --db output/large.db
python run_report.py --db output/large.db --input-dir data/large --expected-dir data/large/expected
```

## Correctness evidence and tests

`python -m pytest -q` runs 188 tests, each using its own temporary folder and database. They cover every required edge case: happy path, empty and header-only input, malformed rows, every required column blank, 28 invalid-value cases with exact reasons, boundary values and future-date edges, exact and conflicting duplicates, schema changes (missing, extra, repeated columns), late and out-of-order data (including an old status not overwriting a newer one), invalid references and their replay, reruns, failure mid-load with rollback and retry, a crash leaving a `RUNNING` row, log and metrics content, configuration mistakes, a new dataset added with config only, the command-line exit codes, deterministic generation, the reconciliation report (including corrupting the database on purpose to prove each check fails), and the performance tooling. Separate tests compare the pipeline's output with the generator's answer key for three seeds, the committed sample, and a 10,000-order run.

`python run_report.py --input-dir data/sample --expected-dir data/expected` prints a reconciliation report with 36 checks, each counting violations (0 = PASS):

- `rows_in = inserted + already loaded + rejected + duplicates dropped` for every successful run;
- the source file re-counted independently of the pipeline equals `rows_in`;
- primary keys are unique, every current row equals the newest history row, every history key has a current row, history rows equal the sum of `rows_inserted`;
- every order's customer exists, and the order total in cents agrees between current and the newest history versions;
- every reject has a reason, source file, line number and original row, and uses a documented reason code;
- the run counts, table row counts, rejected rows and reasons, current rows and order totals equal the answer key.

`sql/reconciliation.sql` holds 14 of the same checks as plain SQL and `sql/sample_queries.sql` holds 14 analysis queries (current status counts, revenue by country, order timelines from history, reject summaries, lineage, run history and throughput). `reports/reconciliation_report.md` and `reports/reconciliation_report_large.md` are saved results: both pass all 36 checks.

## Performance and trade-offs

`python run_benchmark.py` generates the data, runs the pipeline repeatedly on an empty database, and writes `reports/performance_report.md`. Measured on Windows, Python 3.14, pandas 3.1.1, 12 CPUs, with 24,600 customer rows and 240,373 order rows:

- The whole run takes about 10.9 s, roughly 24,000 rows per second. Within the orders dataset, load is the largest stage (44%), then transform (26%) and validate (25%); extract is 5%. Throughput stays between about 21,000 and 24,000 rows per second from 5,000 to 100,000 orders, so cost grows in proportion to the data.
- A profile of the first version showed about 2.5 million slow per-value accesses into pandas string columns, coming from row loops that iterate a column directly. The optimization converts each column to a plain list once (`_plain` in `etl/transform.py`). It made the transform stage 13% faster (faster in 5 of 5 paired repeats) and the whole run about 5% faster (4 of 5 repeats), and cut the slow accesses from 2,553,363 to 523,247. A content fingerprint of every table is identical before and after, so the stored data did not change.
- The gain is moderate. SQLite inserts and pandas' own string handling dominate what remains. Installing `pyarrow` would likely help more, but it is an extra dependency and was not measured.
- Other ideas measured separately during development changed the total by only 2 to 4%, within run-to-run noise, and were not adopted. Memory use was not measured.

Design trade-offs: SQLite and pandas keep the setup to one `pip install` and fit a single-machine batch job, at the cost of single-writer loading and string handling that is slower than a columnar engine. History is kept as an append-only table plus a current table instead of SCD Type 2: late data becomes one more history row and cannot corrupt date ranges, and `valid_to` can be derived with a window function when needed (sample query 4).

## Known limitations

- Timestamps are kept to second precision; sub-second differences are dropped.
- `unit_price` is stored as a floating-point `REAL`; the exact money value is `order_amount_cents`.
- Marking a reject as resolved matches by file name and row number, so a file edited in a way that shifts rows can resolve a reject for a different record.
- Rejects are checked in stages: a row rejected by an earlier stage is not checked by later stages (a row with a rule failure is not also reference-checked).
- A parent dataset must come before its children in `config/pipeline.yaml`.
- Batch processing with a single writer; concurrent writers are not handled.
- Changing a dataset config after its tables exist does not migrate the existing tables.
- The reconciliation report compares the latest successful run of each dataset and assumes nothing else writes to the database.
- The answer key and the pipeline share the same reject-reason names, so a naming mistake made in both would not be caught by the comparison.
- Developed and tested on Windows with Python 3.14 and pandas 3.1.1. The Colab notebook has been executed cell by cell on that machine; other platforms and versions are untested until it is run on Colab.

## Not built yet

A separate design and trade-off note (the trade-offs above are a summary).

## Project contents

```text
config/            pipeline.yaml and one YAML file per dataset (customers, orders)
etl/               extract, validate, transform, load, metrics, logging_setup, pipeline,
                   config_loader, parsing, errors, reconcile
generator/         seeded synthetic data generator with a separate answer key
sql/               reconciliation.sql and sample_queries.sql
tests/             188 automated tests
data/sample/       generated customers.csv and orders.csv (seed 42)
data/expected/     the answer key for the sample
reports/           saved reconciliation and performance reports
run_pipeline.py    command-line entry point
run_report.py      reconciliation report
run_benchmark.py   performance measurement
requirements.txt   pinned dependencies
logs/ output/      created at run time (JSON log file, SQLite database)
```


Writing README.md


In [52]:
import os
missing = [f for f in ['requirements.txt', 'pytest.ini', 'config/pipeline.yaml', 'config/datasets/customers.yaml', 'config/datasets/orders.yaml', 'etl/__init__.py', 'etl/config_loader.py', 'etl/errors.py', 'etl/extract.py', 'etl/load.py', 'etl/logging_setup.py', 'etl/metrics.py', 'etl/parsing.py', 'etl/pipeline.py', 'etl/reconcile.py', 'etl/transform.py', 'etl/validate.py', 'generator/__init__.py', 'generator/generate_data.py', 'run_pipeline.py', 'run_report.py', 'run_benchmark.py', 'sql/reconciliation.sql', 'sql/sample_queries.sql', 'tests/__init__.py', 'tests/conftest.py', 'tests/helpers.py', 'tests/test_cli.py', 'tests/test_config.py', 'tests/test_duplicates.py', 'tests/test_extract.py', 'tests/test_generator.py', 'tests/test_idempotency.py', 'tests/test_late_data.py', 'tests/test_observability.py', 'tests/test_performance.py', 'tests/test_reconcile.py', 'tests/test_recovery.py', 'tests/test_references.py', 'tests/test_schema.py', 'tests/test_transform.py', 'tests/test_validate.py', 'data/sample/customers.csv', 'data/sample/orders.csv', 'data/expected/expected_current_customers.csv', 'data/expected/expected_current_orders.csv', 'data/expected/expected_rejects.csv', 'data/expected/expected_summary.json', 'README.md'] if not os.path.exists(f)]
assert not missing, f'files missing - re-run the %%writefile cells: {missing}'
print('all', 49, 'project files present')

all 49 project files present


## 3. Run the tests

Expected: `188 passed`. Each test uses its own temporary folder and database.

In [53]:
!python -m pytest -q -p no:cacheprovider 2>&1 | tail -n 30

........................................................................ [ 38%]
........................................................................ [ 76%]
............................................                             [100%]
=============================== warnings summary ===============================
../usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63
  /usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
    from pandas.core import (

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
188 passed, 1 warning in 84.83s (0:01:24)


## 4. Sample data: load, then reconcile with the source files and the expected answers

Expected: customers `41 in / 27 inserted / 12 rejected / 2 duplicates / 0 late`, orders `114 / 95 / 17 / 2 / 11`. The first line removes any old database so this section always starts clean; the JSON log lines are hidden here (they are in `logs/pipeline.log`) but any error text and the exit code are shown.

In [54]:
!rm -f output/warehouse.db logs/pipeline.log
!python run_pipeline.py > /tmp/etl_out.txt 2> /tmp/etl_err.txt; echo "exit code: $?"; cat /tmp/etl_out.txt; grep -v '^{"ts"' /tmp/etl_err.txt

exit code: 0

run_id: f6b341f3-2a23-4165-8682-66935810bde6
dataset                    status              rows_in        rows_inserted  rows_already_loaded        rows_rejected   duplicates_dropped         late_records     duration_seconds
customers                 SUCCESS                   41                   27                    0                   12                    2                    0                0.106
orders                    SUCCESS                  114                   95                    0                   17                    2                   11                0.117
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


The reconciliation report: `Result: **PASS** (36 of 36 checks passed)`. Section 5 of the report compares the database with the answer key in `data/expected/`, which the generator stored separately from the input.

In [55]:
!python run_report.py --input-dir data/sample --expected-dir data/expected

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
# Reconciliation report

Generated: 2026-11-14T12:08:41Z  
Database: `output/warehouse.db`  
Result: **PASS** (36 of 36 checks passed)

## 1. Run reconciliation (latest successful run of each dataset)

`rows_in = rows_inserted + rows_already_loaded + rows_rejected + duplicates_dropped`

| dataset | run | rows_in | inserted | already_loaded | rejected | duplicates_dropped | late (kept in history) | sum of parts = rows_in |
|---|---|---|---|---|---|---|---|---|
| customers | f6b341f3 | 41 | 27 | 0 | 12 | 2 | 0 | 41 = 41: PASS |
| orders | f6b341f3 | 114 | 95 | 0 | 17 | 2 | 11 | 114 = 114: PASS |

## 2. Where the rows went: rejects by reason (latest run)

| dataset | reject reason | rows |
|---|---|---|
| customers | CONFLICTING_DUPLICATE:customer_id,updated_at | 2 |
| customers | F

## 5. Rerun the same input: nothing may change

Expected: `rows_inserted 0`; customers `rows_already_loaded 27`, orders `rows_already_loaded 95`; the report still passes.

In [56]:
!python run_pipeline.py > /tmp/etl_out.txt 2> /tmp/etl_err.txt; echo "exit code: $?"; cat /tmp/etl_out.txt; grep -v '^{"ts"' /tmp/etl_err.txt

exit code: 0

run_id: 5924aad2-3e91-427e-affd-ea48d8214c1d
dataset                    status              rows_in        rows_inserted  rows_already_loaded        rows_rejected   duplicates_dropped         late_records     duration_seconds
customers                 SUCCESS                   41                    0                   27                   12                    2                    0                0.086
orders                    SUCCESS                  114                    0                   95                   17                    2                    0                0.108
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [57]:
!python run_report.py --input-dir data/sample --expected-dir data/expected | head -n 5

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
# Reconciliation report

Generated: 2026-11-14T12:08:42Z  
Database: `output/warehouse.db`  
Result: **FAIL** (34 of 36 checks passed)


## 6. SQL: reconciliation checks and sample queries

In [58]:
import sqlite3


def statements(path):
    """Split a .sql file into complete statements (comment-only lines between them are skipped)."""
    buffer = ""
    for line in open(path, encoding="utf-8"):
        if line.strip().startswith("--") and not buffer:
            continue
        buffer += line
        if sqlite3.complete_statement(buffer):
            yield buffer.strip()
            buffer = ""


con = sqlite3.connect("file:output/warehouse.db?mode=ro", uri=True)       # read-only
print("sql/reconciliation.sql: every check must show 0 violations")
for statement in statements("sql/reconciliation.sql"):
    name, violations = con.execute(statement).fetchone()
    print(f"  {'PASS' if violations == 0 else 'FAIL'}  {name}  ({violations})")

queries = list(statements("sql/sample_queries.sql"))
print()
print(len(queries), "sample queries in sql/sample_queries.sql; running Q1, Q2 and Q4:")
for number in (1, 2, 4):
    cursor = con.execute(queries[number - 1])
    print()
    print(f"Q{number}: " + " | ".join(d[0] for d in cursor.description))
    for row in cursor.fetchmany(8):
        print("   " + " | ".join(str(x) for x in row))
con.close()

sql/reconciliation.sql: every check must show 0 violations
  PASS  run_counts_add_up  (0)
  PASS  customers_history_key_unique  (0)
  PASS  customers_current_key_unique  (0)
  PASS  orders_history_key_unique  (0)
  PASS  orders_current_key_unique  (0)
  PASS  customers_current_is_newest_history_row  (0)
  PASS  orders_current_is_newest_history_row  (0)
  PASS  customers_history_has_current  (0)
  PASS  orders_history_has_current  (0)
  PASS  orders_reference_existing_customer  (0)
  PASS  customers_history_equals_inserted  (0)
  PASS  orders_history_equals_inserted  (0)
  PASS  orders_amount_cents_total  (0)
  PASS  rejects_traceable  (0)

14 sample queries in sql/sample_queries.sql; running Q1, Q2 and Q4:

Q1: status | orders | total_amount
   delivered | 22 | 100282041.34
   shipped | 9 | 134908.25
   placed | 7 | 31026.46
   cancelled | 2 | 19810.81

Q2: country_code | delivered_orders | revenue
   IN | 4 | 100031614.97
   US | 11 | 165196.54
   GB | 3 | 53013.68
   AE | 2 | 18716.4

## 7. Larger dataset: 20,000 customers and 100,000 orders (about 265,000 rows)

The generator writes the input and, separately, the expected answers. On Colab this takes roughly a minute.

In [59]:
!python -m generator.generate_data --customers 20000 --orders 100000 --out-dir data/large --expected-dir data/large/expected

customers  rows=  24600  valid=  23049  rejected= 1090  exact_duplicates= 461  late=  631
orders     rows= 240373  valid= 230452  rejected= 5312  exact_duplicates=4609  late=26102
inputs   -> /content/data/large
expected -> /content/data/large/expected


In [60]:
!rm -f output/large.db
!python run_pipeline.py --input-dir data/large --db output/large.db --log-file logs/large.log > /tmp/etl_out.txt 2> /tmp/etl_err.txt; echo "exit code: $?"; cat /tmp/etl_out.txt; grep -v '^{"ts"' /tmp/etl_err.txt

exit code: 0

run_id: 49c94ffe-baf9-4d66-82b8-98cdcbace64d
dataset                    status              rows_in        rows_inserted  rows_already_loaded        rows_rejected   duplicates_dropped         late_records     duration_seconds
customers                 SUCCESS                24600                23049                    0                 1090                  461                  631                 1.04
orders                    SUCCESS               240373               230452                    0                 5312                 4609                26102               16.042
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [61]:
!python run_report.py --db output/large.db --input-dir data/large --expected-dir data/large/expected --report-file reports/reconciliation_report_large.md | head -n 8

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
# Reconciliation report

Generated: 2026-11-14T12:09:12Z  
Database: `output/large.db`  
Result: **PASS** (36 of 36 checks passed)

## 1. Run reconciliation (latest successful run of each dataset)



## 8. Performance: where the time goes and one measured optimization

Runs the pipeline on a fresh 100,000-order dataset, 3 repeats of the code before and after the optimization (alternating), a scaling run and a profile; takes a few minutes. The report is printed and saved to `reports/performance_report.md`. Timings depend on the machine, so Colab's numbers will differ from the ones in the README.

In [62]:
!python run_benchmark.py --repeats 3

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
generating 20,000 customers + 100,000 orders (seed 42)
measuring before/after:
  run 1/3 before  22.68 s
  run 1/3 after   17.05 s
  run 2/3 after   17.30 s
  run 2/3 before  20.92 s
  run 3/3 before  22.07 s
  run 3/3 after   17.36 s
  scaling: 5000 orders -> 1.01 s
  scaling: 25000 orders -> 3.89 s
  scaling: 50000 orders -> 10.65 s
# Performance report

Generated: 2026-11-14T12:13:00Z  
Machine: Linux-6.6.159+-x86_64-with-glibc2.39, 2 CPUs; Python 3.13.18, pandas 3.1.1, SQLite 3.45.1  
Data: seed 42, 24,600 customer rows + 240,373 order rows (20,000 customers, 100,000 orders, with bad rows, duplicates and late versions)  
Method: 3 repeats per variant, alternating order, empty database each run, medians shown (see the notes at the end for what these numbers can and cannot tell

## 9. Package the results

Creates `etl_project.zip` with all the code, tests, SQL, sample data and reports.

In [63]:
import zipfile
from pathlib import Path

include = ["README.md", "pytest.ini", "requirements.txt", "run_pipeline.py", "run_report.py", "run_benchmark.py",
           "config", "etl", "generator", "sql", "tests", "data/sample", "data/expected", "reports"]
with zipfile.ZipFile("etl_project.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for item in include:
        root = Path(item)
        for path in sorted(root.rglob("*") if root.is_dir() else [root]):
            if path.is_file() and "__pycache__" not in path.parts:
                archive.write(path)
with zipfile.ZipFile("etl_project.zip") as archive:
    print(len(archive.namelist()), "files in etl_project.zip, for example:")
    for name in archive.namelist()[:8]:
        print("  ", name)

52 files in etl_project.zip, for example:
   README.md
   pytest.ini
   requirements.txt
   run_pipeline.py
   run_report.py
   run_benchmark.py
   config/datasets/customers.yaml
   config/datasets/orders.yaml


In [64]:
try:
    from google.colab import files
    files.download("etl_project.zip")
except Exception as exc:  # not running in the Colab front end
    print("download skipped:", exc)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>